<a href="https://colab.research.google.com/github/destinyanselem/ziroq101/blob/main/PROJECT_ZIROQ_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJECT ZIROQ — Full-Source AI Short-Form Production Studio

**Upgraded from the supplied notebook, preserving its `Source`, `Word`, `Sentence`, `Transcript`, `Segment`, and `ClipPlan` foundations and the proven FFmpeg/ASS approach.**

The pipeline is designed to produce five distinct, source-faithful vertical edits from one authorized long-form source. It performs full-source chunked transcription, builds a content map, proposes related multi-interval stories, renders word-timed captions and a speech-first audio mix, acquires licensed music and contextual B-roll through multiple providers, runs technical/editorial gates, and only then stages a complete Drive package.

**Important operating truth:** free Colab is not guaranteed to have a GPU or unlimited time. External asset APIs require free keys; these are not paid credits. When no external asset is available, ZIROQ still creates original procedural music and keeps the source footage as the main visual. It never bypasses a licence check just to force a stock asset into the video.

### Asset setup (recommended)

- **Coverr (recommended single provider):** create a free account, open Profile → API, create an application, and copy the API key. The current free tier offers 50 requests/hour and requires no credit card. Put the key in a Colab Secret named `COVERR_API_KEY`, turn on that secret's Notebook access, or provide it as an environment variable with that name. ZIROQ searches Coverr for both music and B-roll, excludes premium/sponsored items and AI-generated stock video, records provenance, and sends the required video-download tracking call. On the free API tier, generated publishing captions include Coverr/creator credit where the creator is listed.
- **Pixabay (optional B-roll backup):** create a free API key and set `PIXABAY_API_KEY` in Colab Secrets and enable Notebook access. ZIROQ persists the API search response for at least 24 hours, as Pixabay requires. Pixabay does not require author attribution, but source/author are retained in the manifest.
- **Internet Archive (music backup):** searches are broader than before; candidate tracks are accepted only after item-level CC0/CC BY metadata verification and successful audio decoding. Public Domain Mark is not treated as a licence.

### Run order

Run cells from top to bottom. Paste one authorized YouTube URL or a local Colab video path, then run all cells and review all five outputs and the Drive package.

### No-key/manual option

Place manually downloaded and individually verified music in `PROJECT_ZIROQ/music_pool/` and B-roll in `PROJECT_ZIROQ/broll_pool/` (inside Drive when mounted). Each folder needs a JSON manifest named `music_pool_manifest.json` or `broll_pool_manifest.json`. Entries require `filename`, `license`, `license_url`, `source_url`, `title`, and `manual_verified: true`; add `artist`/`author`, `attribution`, and `attribution_required` where applicable. Filenames must be relative to the pool folder. Mixkit entries must use the exact Free License page for the media type. ZIROQ rejects absolute paths, traversal paths, unknown licence labels, invalid media, and missing source pages.

**Feature boundaries:** Coverr's licence prohibits using its content or metadata for AI training, fine-tuning, model evaluation, or datasets; this notebook only uses stock media as edited footage/music in finished videos. Coverr items still require visual review for trademarks/property rights. Pixabay use is subject to its full Content License and prohibited uses. Pexels is not part of the automatic provider chain because its current API page reports that new API-key issuance is paused.

### v4 batch 1 changes (non-Gemini fixes)

- **Framing:** OpenCV 5.x wheels ship no Haar face cascade, which silently forced a centre crop on every shot. A detector is now resolved explicitly (bundled → recovered from an OpenCV 4.x wheel, download only); if none exists the shot keeps the full frame (`framing_uncertain_mode`) and the log says so.
- **Resume:** the five story plans are checkpointed per source and reused when the transcript and config are unchanged (`resume_plans`).
- **Editorial QA:** quotation integrity and sentence-boundary checks now verify against the transcript; context warnings are specific and listed per clip as `review_flags`.
- **Render retry:** a retry changes the CRF locally instead of mutating `CFG`.
- **Classifier:** `STORY_CUES` now matches lowercase tokens.


### v4 batch 2 changes
- **CELL 3b/3c (new):** Gemini core — consent gate, key manager (per-project quota tracking, failover, no key ever printed/saved), structured-JSON client with validation/repair/cache, stage tracker, capability parity matrix, and mock-based tests. With no keys or no consent, ZIROQ runs exactly as in batch 1 (local-only).
- **Cell 0:** installs `google-genai>=2.3.0`.
- **CFG:** new `zero_cost_mode`, `external_ai_enabled` and `gemini_*` settings. Add keys as Colab Secrets `GEMINI_API_KEY_1`…`GEMINI_API_KEY_5` (notebook access on).
- Gemini is not yet wired into clip selection; that is batch 3.

### v4 batch 3 changes
- **CELL 6b/6c (new): Gemini Director.** Gemini watches the whole video from its public YouTube link and proposes the five clips (timestamps, hook, music brief, B-roll words). The notebook snaps every timestamp to real Whisper sentence boundaries, verifies the quoted opening/closing words, caps clips at 90 s, rejects overlaps, and fills any gap with the local selector.
- **CELL 7:** director plans take priority; local selection is the fallback and gap-filler. The plan checkpoint now records whether the director was active.
- **CELL 8:** Gemini's music brief (mood + search words) now drives Coverr / Internet Archive music search, and its B-roll words drive the B-roll query. Choosing among provider results is still rule-based.
- New CFG: `use_gemini_director`, `gemini_agentic_video`, `director_min_clip_sec`.

### v4 batch 4 changes
- **Visual direction:** the Gemini director now also reports each clip's layout (single speaker / multi-speaker / slides) and speaker position samples. With no working local face detector, a single-speaker clip is cropped to Gemini's positions; slides and multi-speaker shots keep the full frame. Local face tracking still wins when it works.
- New CFG: `use_gemini_framing_hint`, `trust_gemini_layout`.

### v4 batch 5 changes
- **Music + B-roll ranking:** Gemini orders the licensed search results by fit to the clip's brief (it only re-orders; licence and decode checks still run on every candidate). Falls back to the old order.
- **Local files:** if the source is a local file, a small 360p proxy is uploaded to Gemini's File API (pinned to the uploading key, deleted after use) so the director works without a YouTube link.
- **Sound effects:** Gemini proposes cues; the notebook validates them and synthesises whooshes/impacts/pops/risers locally (no licences), mixed under speech. Without Gemini cues, a whoosh is placed at joins between source intervals.
- New CFG: `gemini_rerank`, `gemini_upload_local`, `enable_sfx`, `sfx_max_cues`, `sfx_local_fallback`.

### v4 batch 6 changes (finishing studio)
- **CELL 9b (new):** (1) completeness gate: a clip never ends on an unfinished sentence or opens on a dangling connector; (2) one Gemini request reviews all five clips as a strict editor and its advice is executed only as bounded, validated sentence fixes; (3) filler-word and long-pause removal (cuts only in silence), with 12 ms audio fades and an alternating mild punch-in to hide jump cuts.
- **Split-screen:** two speakers side by side (Gemini reports their positions) are stacked top/bottom instead of squeezed into one crop.
- **Director** aims for `director_clip_sec` (default 30-75 s; hard cap 90 s).
- Report now includes a `finishing` block per clip.


In [ ]:
# CELL 0 — Runtime setup with yt-dlp PO Token architecture
import os, sys
print("Updating system packages and installing ffmpeg / deno...")
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core curl

# Install Deno for JS Challenge solving (required by modern yt-dlp / BgUtils)
if not os.path.exists("/usr/local/bin/deno"):
    !curl -fsSL https://deno.land/install.sh | sh
    !ln -sf /root/.deno/bin/deno /usr/local/bin/deno

# Update yt-dlp and install the BgUtils plugin
%pip -q install --upgrade yt-dlp
%pip -q install bgutil-ytdlp-pot-provider
%pip -q install faster-whisper ctranslate2 numpy scipy opencv-python-headless pillow requests scikit-learn "google-genai>=2.3.0"

# Configure and run the local bgutil server in the background
import subprocess, time
print("Initializing BgUtils PO Token Provider Server...")
# Start background local server
subprocess.Popen(["deno", "run", "--allow-net", "--allow-env", "npm:bgutil-ytdlp-pot-provider"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Setup Complete. Provider server active on default port 4416.")

Updating system packages and installing ffmpeg / deno...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
sh: 105: cannot open /dev/tty: No such device or address
Initializing BgUtils PO Token Provider Server...
Setup Complete. Provider server active on default port 4416.


In [ ]:
import os
import sys
import socket
import subprocess
import json
import time

def run_cmd(cmd, timeout=30):
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return res.returncode, res.stdout.strip(), res.stderr.strip()
    except Exception as e:
        return -1, "", str(e)

print("=== DIAGNOSTIC CHECK 1: Deno Environment ===")
code, out, err = run_cmd(["deno", "--version"])
print(f"Exit Code: {code}\nStdout:\n{out}\nStderr:\n{err}\n")

print("=== DIAGNOSTIC CHECK 2: yt-dlp Environment ===")
try:
    import yt_dlp
    print(f"yt-dlp package version: {yt_dlp.version.__version__}")
except ImportError as e:
    print(f"Failed to import yt-dlp: {e}")

# Check EJS dependency availability via cli execution
code, out, err = run_cmd([sys.executable, "-m", "pip", "show", "yt-dlp-ejs"], timeout=15)
print(f"yt-dlp-ejs status code {code}: {out or err}")

print("=== DIAGNOSTIC CHECK 3: BgUtils Plugin Discovery ===")
# Detect if the plugin registry can find the bgutil provider
try:
    from yt_dlp.plugins import directories
    print("yt-dlp plugin directories:", directories())
except Exception as e:
    print(f"Error reading yt-dlp plugin directories: {e}")

print("=== DIAGNOSTIC CHECK 4: BgUtils Local Server & Port Health ===")
# Test connection to local port 4416
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(2.0)
try:
    s.connect(("127.0.0.1", 4416))
    print("SUCCESS: Port 4416 is listening and accepting connections!")
    s.close()
except Exception as e:
    print(f"FAILURE: Cannot connect to Port 4416: {e}")

# Let us list active Node/Deno/bgutil processes
code, out, err = run_cmd(["pgrep", "-a", "deno"])
print(f"Active Deno processes (pgrep):\n{out or 'None found'}\n")

print("=== DIAGNOSTIC CHECK 5: Cookie File Integrity ===")
cookie_path = "/content/www.youtube.com_cookies.txt"
if os.path.exists(cookie_path):
    size = os.path.getsize(cookie_path)
    print(f"Cookie file found at {cookie_path} (Size: {size} bytes)")
    # Display first line safely to verify standard Netscape format without exposing tokens
    with open(cookie_path, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                print(f"Sample row format check (redacted): {line.split()[:4]} ...")
                break
else:
    print(f"WARNING: Cookie file NOT found at {cookie_path}")

print("=== DIAGNOSTIC CHECK 6: Controlled Metadata Call ===")
test_url = "https://www.youtube.com/watch?v=2c81cbv0SKk"
meta_opts = {
    "quiet": True,
    "no_warnings": True,
    "noplaylist": True,
    "skip_download": True,
    "format": "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
    "ignore_no_formats_error": True,
    "extractor_args": {
        "youtube": {
            "player_client": ["mweb", "tv", "web_safari"]
        }
    }
}
if os.path.exists(cookie_path):
    meta_opts["cookiefile"] = cookie_path

try:
    with yt_dlp.YoutubeDL(meta_opts) as ydl:
        info = ydl.extract_info(test_url, download=False)
    print(f"Video Title: {info.get('title')}")
    print(f"Playability Status: {info.get('playability_status', 'N/A')}")
    formats = info.get('formats', [])
    print(f"Available formats found: {len(formats)}")
except Exception as e:
    print(f"Metadata extraction failed: {e}")

=== DIAGNOSTIC CHECK 1: Deno Environment ===
Exit Code: 0
Stdout:
deno 2.9.8 (stable, release, x86_64-unknown-linux-gnu)
v8 15.0.245.2-rusty
typescript 6.0.3
Stderr:


=== DIAGNOSTIC CHECK 2: yt-dlp Environment ===
yt-dlp package version: 2026.10.26
yt-dlp-ejs status code 1: WARNING: Package(s) not found: yt-dlp-ejs
=== DIAGNOSTIC CHECK 3: BgUtils Plugin Discovery ===
yt-dlp plugin directories: ['/usr/local/lib/python3.13/dist-packages/yt_dlp_plugins']
=== DIAGNOSTIC CHECK 4: BgUtils Local Server & Port Health ===
FAILURE: Cannot connect to Port 4416: [Errno 111] Connection refused
Active Deno processes (pgrep):
None found

=== DIAGNOSTIC CHECK 5: Cookie File Integrity ===
Cookie file found at /content/www.youtube.com_cookies.txt (Size: 2437 bytes)
Sample row format check (redacted): ['.youtube.com', 'TRUE', '/', 'FALSE'] ...
=== DIAGNOSTIC CHECK 6: Controlled Metadata Call ===
Video Title: 1 Millionaire Boss vs 20 Gen Z Workers (ft. Kevin O’Leary) | Surrounded
Playability Status: N/A


In [ ]:
import os
import sys
import socket
import subprocess
import time
import json

def run_sys_cmd(cmd, timeout=120):
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return res.returncode, res.stdout.strip(), res.stderr.strip()
    except Exception as e:
        return -1, "", str(e)

print("=== STEP 1: UPGRADING DEPENDENCIES ===")
# Upgrade yt-dlp to ensure latest decryptors are loaded
code, out, err = run_sys_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "yt-dlp[default]", "bgutil-ytdlp-pot-provider"])
print(f"Pip setup return code: {code}\nStdout snippet: {out[:300]}\nStderr snippet: {err[:300]}\n")

print("=== STEP 2: STARTING BGUTILS SERVER ===")
# Clean up any previously orphaned Deno/bgutil instances
run_sys_cmd(["pkill", "-f", "deno"])
run_sys_cmd(["pkill", "-f", "bgutil"])
time.sleep(1)

# Launch the official upstream server cleanly utilizing the npm: identifier compatible with Deno
server_proc = subprocess.Popen(
    ["deno", "run", "--allow-net", "--allow-env", "npm:bgutil-ytdlp-pot-provider"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Loop check with backoff to verify port 4416 is listening
server_active = False
for attempt in range(8):
    time.sleep(2.0)
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2.0)
    try:
        s.connect(("127.0.0.1", 4416))
        print(f"SUCCESS: Connected to Port 4416 on attempt {attempt + 1}!")
        s.close()
        server_active = True
        break
    except Exception:
        pass

if not server_active:
    print("WARNING: Could not connect to Port 4416. Server stdout/stderr:")
    try:
        outs, errs = server_proc.communicate(timeout=2)
        print(f"STDOUT:\n{outs}\nSTDERR:\n{errs}")
    except Exception as e:
        print(f"Failed to read logs: {e}")

print("\n=== STEP 3: PLUGIN DISCOVERY CHECK ===")
import yt_dlp
try:
    from yt_dlp.plugins import directories
    print("yt-dlp recognized plugin directories:", directories())
except Exception as e:
    print(f"Plugin directories error: {e}")

print("\n=== STEP 4: AUTHENTICATED DOWNLOAD WITH PO-TOKEN & EJS ===")
cookie_path = "/content/www.youtube.com_cookies.txt"
test_url = "https://www.youtube.com/watch?v=2c81cbv0SKk"
output_tmpl = "/content/ziroq_work/cache/2c81cbv0SKk_test_download.%(ext)s"

opts = {
    "quiet": False,
    "no_warnings": False,
    "noplaylist": True,
    "outtmpl": output_tmpl,
    "format": "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
    "merge_output_format": "mp4",
    "ignore_no_formats_error": False,
    "verbose": True,
    "remote_components": {"ejs:github"},
    "extractor_args": {
        "youtube": {
            "player_client": ["mweb", "tv", "web_safari"]
        }
    }
}
if os.path.exists(cookie_path):
    opts["cookiefile"] = cookie_path
    print(f"Using cookies from: {cookie_path}")

try:
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([test_url])
    print("Download completed successfully.")
except Exception as e:
    print(f"Download execution failed: {e}")

print("\n=== STEP 5: FFPROBE MEDIA VERIFICATION ===")
import glob
downloaded_files = glob.glob("/content/ziroq_work/cache/2c81cbv0SKk_test_download.*")
if downloaded_files:
    target_file = downloaded_files[0]
    print(f"Found downloaded file: {target_file}")
    ff_code, ff_out, ff_err = run_sys_cmd([
        "ffprobe", "-v", "error", "-show_format", "-show_streams", "-print_format", "json", target_file
    ])
    if ff_code == 0:
        print("FFprobe successfully parsed and validated the file streams.")
        try:
            meta_js = json.loads(ff_out)
            for stream in meta_js.get("streams", []):
                print(f"Stream Index {stream.get('index')}: {stream.get('codec_type')} ({stream.get('codec_name')})")
        except Exception as jse:
            print(f"Could not parse JSON format: {jse}")
    else:
        print(f"FFprobe stream check FAILED with exit code {ff_code}:\n{ff_err}")
else:
    print("No downloaded file found to verify.")

=== STEP 1: UPGRADING DEPENDENCIES ===
Pip setup return code: 0
Stdout snippet: Requirement already satisfied: bgutil-ytdlp-pot-provider in /usr/local/lib/python3.13/dist-packages (2.0.2)
Stderr snippet: 

=== STEP 2: STARTING BGUTILS SERVER ===


[debug] Encodings: locale utf-8, fs utf-8, pref utf-8, out UTF-8 (No ANSI), error UTF-8 (No ANSI), screen UTF-8 (No ANSI)
[debug] yt-dlp version stable@2026.10.26 from yt-dlp/yt-dlp [c91f8de0b] (pip) API
[debug] params: {'quiet': False, 'no_warnings': False, 'noplaylist': True, 'outtmpl': '/content/ziroq_work/cache/2c81cbv0SKk_test_download.%(ext)s', 'format': 'bestvideo[height<=1080]+bestaudio/best[height<=1080]/best', 'merge_output_format': 'mp4', 'ignore_no_formats_error': False, 'verbose': True, 'remote_components': {'ejs:github'}, 'extractor_args': {'youtube': {'player_client': ['mweb', 'tv', 'web_safari']}}, 'cookiefile': '/content/www.youtube.com_cookies.txt', 'js_runtimes': {'deno': {}}, 'compat_opts': set(), 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-us,en;q=0.5', 'Sec-Fetch-Mo

STDOUT:

STDERR:
error: JSR package not found: @brainicism/bgutil-ytdlp-pot-provider


=== STEP 3: PLUGIN DISCOVERY CHECK ===
yt-dlp recognized plugin directories: ['/usr/local/lib/python3.13/dist-packages/yt_dlp_plugins']

=== STEP 4: AUTHENTICATED DOWNLOAD WITH PO-TOKEN & EJS ===
Using cookies from: /content/www.youtube.com_cookies.txt
[youtube] Extracting URL: https://www.youtube.com/watch?v=2c81cbv0SKk
[youtube] 2c81cbv0SKk: Downloading webpage
[youtube] 2c81cbv0SKk: Downloading mweb client config


[debug] [youtube] Forcing "main" player JS variant for player b559f50c
        original url = /s/player/b559f50c/player_es6.vflset/en_US/base.js
[debug] [youtube] 2c81cbv0SKk: Detected experiment to bind GVS PO Token to video ID for mweb client


[youtube] 2c81cbv0SKk: Downloading mweb player API JSON


[debug] [youtube] 2c81cbv0SKk: Detected a 6s ad for mweb


... (9 outputs left out)

[debug] [youtube] 2c81cbv0SKk: Some mweb client subtitles require a PO Token which was not provided. They will be discarded since they are not downloadable as-is. You can manually pass a Subtitles PO Token for this client with --extractor-args "youtube:po_token=mweb.subs+XXX" . For more information, refer to  https://github.com/yt-dlp/yt-dlp/wiki/PO-Token-Guide
[debug] [youtube] 2c81cbv0SKk: Some web_safari client subtitles require a PO Token which was not provided. They will be discarded since they are not downloadable as-is. You can manually pass a Subtitles PO Token for this client with --extractor-args "youtube:po_token=web_safari.subs+XXX" . For more information, refer to  https://github.com/yt-dlp/yt-dlp/wiki/PO-Token-Guide
[debug] [youtube] 2c81cbv0SKk: There are missing subtitles languages because a PO token was not provided. Automatic captions for 14 languages are missing.
[debug] Sort order given by extractor: quality, res, fps, hdr:12, source, vcodec, channels, acodec, lang,

[info] 2c81cbv0SKk: Downloading 1 format(s): 96-14


[debug] Invoking hlsnative downloader on "https://manifest.googlevideo.com/api/manifest/hls_playlist/expire/1794991021/ei/TQ_9av-QH82Cpt8P7feUsQ8/ip/34.80.135.0/id/d9cf3571bbf448a9/itag/96/source/youtube/requiressl/yes/ratebypass/yes/pfa/1/sgoap/clen%3D109988234%3Bdur%3D6796.108%3Bgir%3Dyes%3Bitag%3D140%3Blmt%3D1794381790509911%3Bxtags%3Dacont%3Doriginal:lang%3Den-US/sgovp/clen%3D1305084494%3Bdur%3D6796.038%3Bgir%3Dyes%3Bitag%3D137%3Blmt%3D1794893573503284/rqh/1/hls_chunk_host/rr2---sn-ajun55-5x.googlevideo.com/xpc/EgVo2aDSNQ%3D%3D/cps/856/met/1794969421,/mh/tt/mm/18,29/mn/sn-ajun55-5x,sn-ojun55-5i/ms/aub,rdu/mv/m/mvi/2/pl/23/rms/aub,aub/force_finished/1/pcm2/no/siu/1/bui/AZFAFb8pB9rmNRodpm7ncN0jFh2VJq-uou3DGJZFRL4KWxPI-dT7pAnn-ngobYm9pW-tOBvT8A/spc/nfZ8ZtK8jv6FA751XxniXKXgRJssOQb2KHT4esUzlqHb5Gi1NAFJpiHfWnL5ECLPUDkcmnl7eALd5dE/vprv/1/ns/4TTKer_tdHLDAy2q9Eybw6Ya/playlist_type/CLEAN/dover/11/txp/5532234/epbp/1/mt/1794968942/fvip/4/keepalive/yes/fexp/51565116,52250119,52323638/n/5X9S-RKK

[download] /content/ziroq_work/cache/2c81cbv0SKk_test_download.mp4 has already been downloaded
[download] 100% of    1.32GiB
Download completed successfully.

=== STEP 5: FFPROBE MEDIA VERIFICATION ===
Found downloaded file: /content/ziroq_work/cache/2c81cbv0SKk_test_download.mp4
FFprobe successfully parsed and validated the file streams.
Stream Index 0: audio (aac)
Stream Index 1: video (h264)


In [ ]:
# CELL 0b — START HERE (phone-friendly form: no console input box)
#@title ▶ START HERE — paste your link and (optionally) type I AGREE { display-mode: "form" }
VIDEO_URL = "https://www.youtube.com/watch?v=2c81cbv0SKk"  #@param {type:"string"}
TYPE_I_AGREE_TO_USE_GEMINI = "I AGREE"  #@param {type:"string"}
# Gemini consent text (read before typing I AGREE): your TRANSCRIPT is sent to Google's Gemini API. On Google's UNPAID tier
# the content may be used to improve Google products and read by human reviewers, so never use private/confidential videos.
# Leave the second box empty to run 100% local (no Gemini). Use keys from AI Studio projects with NO billing to stay at ₦0.
import os
if VIDEO_URL.strip():
    os.environ["ZIROQ_URL"] = VIDEO_URL.strip()
os.environ["ZIROQ_GEMINI_CONSENT"] = TYPE_I_AGREE_TO_USE_GEMINI.strip()
print("Link saved." if VIDEO_URL.strip() else "No link in the form: Cell 4 will ask for it.",
      "| Gemini:", "consent received" if TYPE_I_AGREE_TO_USE_GEMINI.strip() == "I AGREE" else "OFF (local-only)")

Link saved. | Gemini: consent received


In [ ]:
# CELL 1 — Configuration and runtime diagnostics
import os, re, sys, json, glob, math, time, uuid, csv, wave, hashlib, shutil, subprocess, textwrap, random, bisect, importlib.util
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Any, Tuple

import numpy as np

CFG = {
    "work_dir": "/content/ziroq_work",
    "drive_mount": "/content/drive",
    "drive_root_name": "PROJECT_ZIROQ",
    "out_w": 1080, "out_h": 1920, "out_fps": 30,
    "max_clip_duration": 90.0,
    "preferred_min_duration": 55.0,
    "transcribe_chunk_sec": 600.0,
    "transcribe_overlap_sec": 2.0,
    "whisper_gpu_model": "small",
    "whisper_cpu_model": "base",
    "whisper_beam_size": 3,
    "whisper_language": None,          # None = automatic detection
    "use_semantic_embeddings": True,
    "semantic_model_id": "sentence-transformers/all-MiniLM-L6-v2",
    "coverr_api_key": os.environ.get("COVERR_API_KEY", ""),
    "enable_coverr_assets": True,
    "pixabay_api_key": os.environ.get("PIXABAY_API_KEY", ""),
    "enable_internet_archive_music": True,
    "enable_pixabay_broll": True,
    "broll_cache_hours": 24,
    "broll_max_downloads_per_run": 5,
    "render_preset": "veryfast",
    "render_crf": 19,
    "qa_retry_count": 1,
    "min_story_score": 32.0,
    "required_clip_count": 5,
    "allow_fewer_clips": True,
    "run_self_tests": os.environ.get("ZIROQ_RUN_TESTS", "") == "1",   # developer tests; OFF for normal use (they can fail for unrelated reasons)
    "credit_source_in_description": True,   # adds "Source: <title> — <channel>" to the YouTube description
    "verbose_qa": False,                    # True prints every QA check; False prints only failures (kinder to phones)
    "download_zip_when_no_drive": True,              # if the video only supports 3-4 distinct stories, make those instead of stopping
    "min_clip_count": 3,
    "framing_uncertain_mode": "fit_blur",   # when subject detection is unavailable/unsure: "fit_blur" (keep full frame) or "center_crop" (old behaviour)
    "resume_plans": True,                  # reuse the five story plans for the same source/transcript/config
    # ---- Gemini (batch 2). Keys come from Colab Secrets GEMINI_API_KEY_1..N — never paste keys here.
    "zero_cost_mode": True,                # never selects a paid-only path; hard request caps below
    "external_ai_enabled": None,           # None = ask once (typed consent); True/False = decide in advance
    "gemini_keys_are_separate_projects": True,   # your keys come from different projects; secret GEMINI_KEY_PROJECTS can override
    "gemini_model_reasoning": ["gemini-3.8-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite"],  # tried in order; unavailable ones are skipped at runtime
    "gemini_model_fast": ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"],
    "gemini_max_requests_per_run": 60,     # hard cap across all keys (cost + quota guard)
    "gemini_max_attempts_per_call": 6,
    "gemini_max_wait_sec": 120,            # longest we wait for a quota window before using the local fallback
    "gemini_min_interval_sec": 4.0,        # pacing per project
    "gemini_request_timeout_sec": 110,
    "gemini_call_budget_sec": 120,          # HARD wall-clock cap for any one Gemini task (all retries/waits included); then the local fallback runs
    "gemini_breaker_sec": 600,              # after 2 timeouts in a row, skip Gemini for this long instead of retrying again and again
    "gemini_prompt_char_cap": 400000,
    "gemini_live_probe": False,             # a few tiny requests to report what your keys can really do
    "gemini_probe_agentic": True,
    "use_gemini_director": True,           # Gemini watches the YouTube link and proposes the clips + music direction
    "gemini_agentic_video": False,         # True = "agentic" video processing (deeper, slower; test it first)
    "director_min_clip_sec": 20.0,
    "use_gemini_framing_hint": True,      # when no local face detector works, use Gemini's speaker-position samples
    "trust_gemini_layout": True,
    "gemini_rerank": True,                # Gemini orders licensed music / B-roll search results by fit (licence checks still run after)
    "gemini_upload_local": True,          # local video files: upload a small proxy to Gemini (File API, auto-deleted after use)
    "enable_sfx": True, "sfx_max_cues": 5, "sfx_local_fallback": True,
    # ---- v4 batch 6: finishing studio
    "enable_completeness_gate": True,     # never end on an unfinished sentence / open on a dangling connector
    "enable_gemini_critic": True,          # one extra Gemini request reviews all five clips as a strict editor
    "enable_tighten": True,                # remove filler words and long pauses (cuts only in silence/filler)
    "tighten_pause_sec": 0.60, "tighten_max_fraction": 0.20, "tighten_fillers": True,
    "punch_in_zoom": True,                 # alternate a mild zoom on tightened cuts to hide the jump
    "enable_split_screen": True,           # two speakers side by side -> stacked layout (needs Gemini's speaker positions)
    "director_clip_sec": [30, 75],         # length Gemini aims for (hard cap stays 90 s)          # Gemini saying "slides/multi-speaker" forces full-frame (fit_blur) layout
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
os.makedirs(CFG["work_dir"], exist_ok=True)


def log(stage, message):
    print(f"[{time.strftime('%H:%M:%S')}] {stage:<18} {message}", flush=True)


def first_line(cmd):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
        return (p.stdout or p.stderr).strip().splitlines()[0]
    except Exception as exc:
        return f"unavailable ({type(exc).__name__})"

print("Run ID:", RUN_ID)
print("Python:", sys.version.split()[0])
print("FFmpeg:", first_line(["ffmpeg", "-version"]))
print("FFprobe:", first_line(["ffprobe", "-version"]))
try:
    import ctranslate2
    GPU_COUNT = ctranslate2.get_cuda_device_count()
except Exception:
    GPU_COUNT = 0
print("CTranslate2 CUDA devices:", GPU_COUNT)
try:
    import cv2
    print("OpenCV:", cv2.__version__)
except Exception as exc:
    print("OpenCV unavailable:", exc)

# Keep all outputs and resumable caches grouped by this run.
RUN_DIR = os.path.join(CFG["work_dir"], RUN_ID)
CACHE_ROOT = os.path.join(CFG["work_dir"], "cache")
ASSET_DIR = os.path.join(RUN_DIR, "assets")
RENDER_DIR = os.path.join(RUN_DIR, "render")
REPORT_DIR = os.path.join(RUN_DIR, "reports")
for _p in (RUN_DIR, CACHE_ROOT, ASSET_DIR, RENDER_DIR, REPORT_DIR):
    os.makedirs(_p, exist_ok=True)

print("Run workspace:", RUN_DIR)

Run ID: 20261118T021422Z_ac87be
Python: 3.13.18
FFmpeg: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
FFprobe: ffprobe version 6.1.1-3ubuntu5 Copyright (c) 2007-2023 the FFmpeg developers
CTranslate2 CUDA devices: 0
OpenCV: 5.0.0
Run workspace: /content/ziroq_work/20261118T021422Z_ac87be


In [ ]:
# CELL 2 — Data contracts, file safety, FFprobe, state, and timeline mapping
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Any

@dataclass
class Source:
    video_id: str
    title: str
    duration: float
    video_path: str
    audio_path: str = ""
    width: int = 0
    height: int = 0

@dataclass
class Word:
    word: str
    start: float
    end: float
    prob: float = 1.0

@dataclass
class Sentence:
    text: str
    start: float
    end: float

@dataclass
class Transcript:
    language: str
    words: List[Word]
    sentences: List[Sentence]

@dataclass
class Segment:
    source_start: float
    source_end: float
    purpose: str
    transcript: str
    crop_fx: float = 0.5
    role: str = "support"
    confidence: float = 1.0

    @property
    def duration(self):
        return max(0.0, self.source_end - self.source_start)

@dataclass
class ClipPlan:
    title: str
    narrative_type: str
    segments: List[Segment]
    target_duration: float = 90.0   # ceiling, not a padding target
    notes: Dict[str, Any] = field(default_factory=dict)

    def content_duration(self):
        return sum(s.duration for s in self.segments)

    @property
    def duration(self):
        return self.content_duration()

    def validate(self, source_duration=None):
        if not self.segments:
            raise ValueError("ClipPlan has no source segments")
        if self.content_duration() <= 0:
            raise ValueError("ClipPlan duration is not positive")
        if self.content_duration() > min(float(self.target_duration), CFG["max_clip_duration"]) + 0.02:
            raise ValueError("ClipPlan exceeds the 90-second maximum")
        for seg in self.segments:
            if seg.source_start < 0 or seg.source_end <= seg.source_start:
                raise ValueError(f"Invalid source interval: {seg.source_start}-{seg.source_end}")
            if source_duration is not None and seg.source_end > source_duration + 0.10:
                raise ValueError(f"Segment exceeds source duration: {seg.source_end}>{source_duration}")
        return True

    def to_dict(self):
        return asdict(self)


def safe_filename(value, maxlen=72):
    value = re.sub(r"[^\w\s-]", "", str(value or ""), flags=re.UNICODE).strip()
    value = re.sub(r"\s+", "_", value)
    return (value or "clip")[:maxlen]


def atomic_write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def run(cmd, cwd=None, timeout=None, check=True, quiet=False):
    cmd = [str(x) for x in cmd]
    if not quiet:
        log("exec", " ".join(cmd)[:200])
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd[0]}\n{result.stderr[-3500:]}")
    return result


def ffprobe_json(path):
    result = run(["ffprobe", "-v", "error", "-print_format", "json", "-show_format", "-show_streams", path], quiet=True)
    return json.loads(result.stdout)


def sha256_file(path, block=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            data = f.read(block)
            if not data:
                break
            digest.update(data)
    return digest.hexdigest()


STATE = {"run_id": RUN_ID, "stage": "initialized", "events": []}
def save_state(stage, **details):
    STATE["stage"] = stage
    STATE["events"].append({"utc": datetime.now(timezone.utc).isoformat(), "stage": stage, **details})
    atomic_write_json(os.path.join(RUN_DIR, "state.json"), STATE)


def build_sentences(words):
    sentences, current = [], []
    for i, word in enumerate(words):
        current.append(word)
        next_gap = words[i + 1].start - word.end if i + 1 < len(words) else 99.0
        if re.search(r"[.!?…]['\")\]]*$", word.word) or next_gap > 1.0:
            sentences.append(Sentence(" ".join(x.word for x in current), current[0].start, current[-1].end))
            current = []
    if current:
        sentences.append(Sentence(" ".join(x.word for x in current), current[0].start, current[-1].end))
    return sentences


def timeline_words(plan, words):
    """Map source timestamps to the assembled output timeline; no padding is introduced."""
    mapped, offset = [], 0.0
    starts = [w.start for w in words]
    for seg_index, seg in enumerate(plan.segments):
        lo = bisect.bisect_left(starts, seg.source_start - 0.01)
        hi = bisect.bisect_right(starts, seg.source_end + 0.01)
        for word in words[lo:hi]:
            mid = (word.start + word.end) / 2
            if not (seg.source_start <= mid < seg.source_end):
                continue
            t0 = offset + max(word.start, seg.source_start) - seg.source_start
            t1 = offset + min(word.end, seg.source_end) - seg.source_start
            if t1 <= t0:
                continue
            clean = re.sub(r"[{}\\]", "", word.word).strip()
            if clean:
                mapped.append({"segment_index": seg_index, "word": clean, "start": t0, "end": t1, "confidence": word.prob})
        offset += seg.duration
    return mapped


# Small, deterministic integrity helpers.
def validate_segment_list(segments, source_duration, max_duration=90.0):
    if not segments:
        raise ValueError("No segments supplied")
    for seg in segments:
        if seg.source_start < 0 or seg.source_end <= seg.source_start or seg.source_end > source_duration + 0.1:
            raise ValueError("Invalid segment boundary")
    total = sum(s.duration for s in segments)
    if total > max_duration + 0.02:
        raise ValueError("Segment list exceeds max duration")
    return total

log("contracts", "stable source/transcript/plan contracts and timeline mapper ready")

[02:14:29] contracts          stable source/transcript/plan contracts and timeline mapper ready


In [ ]:
# CELL 3 — Google Drive mount (upload happens only after all five clips pass)
DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    if not os.path.ismount(CFG["drive_mount"]):
        drive.mount(CFG["drive_mount"])
    DRIVE_AVAILABLE = os.path.isdir(os.path.join(CFG["drive_mount"], "MyDrive"))
except Exception as exc:
    print("Google Drive is not mounted in this runtime:", type(exc).__name__, str(exc)[:300])

DRIVE_ROOT = os.path.join(CFG["drive_mount"], "MyDrive", CFG["drive_root_name"])
if DRIVE_AVAILABLE:
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print("Drive delivery root:", DRIVE_ROOT)
else:
    print("Drive upload will be skipped. Verified local outputs remain in:", RUN_DIR)

Mounted at /content/drive
Drive delivery root: /content/drive/MyDrive/PROJECT_ZIROQ


In [ ]:
# CELL 3b — Gemini core: consent, key manager, quota-aware scheduler, structured client, stage tracker
# Purpose : make Gemini usable as ZIROQ's editorial director WITHOUT ever being a single point of failure.
# Contracts: gemini_structured(...) -> {"ok": bool, "data": ..., "fallback_required": bool, ...}; it never raises for
#            availability problems, so every caller keeps its local fallback. API keys are read from Colab Secrets,
#            are never printed/logged/persisted, and every message passes through gem_scrub().
# API facts used (Google docs, checked 2026-11-13): google-genai >= 2.3.0, client.interactions.create(model, input,
#            response_format={"type":"text","mime_type":"application/json","schema":...}, store=False);
#            rate limits are applied PER PROJECT (not per key); unpaid-tier data may be human-reviewed.
import base64, importlib.metadata
from datetime import timedelta
from zoneinfo import ZoneInfo

GEM_SLEEP = time.sleep          # indirection so tests can run instantly
GEM_NOW = time.time
GEMINI_CACHE_DIR = os.path.join(CACHE_ROOT, "gemini_cache")
GEMINI_STATE_PATH = os.path.join(DRIVE_ROOT if DRIVE_AVAILABLE else CACHE_ROOT, "_GEMINI_STATE", "gemini_state.json")

GEMINI_DOC_FACTS = {
    "as_of": "2026-11-13",
    "agentic_video_models_documented": ["gemini-4-argon", "gemini-3.8-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite"],
    "file_api_retention_hours": 48,
    "unpaid_terms": "unpaid-tier input/output may be used to improve Google products and read by human reviewers",
    "rate_limit_scope": "per project, not per API key; RPD resets at midnight Pacific",
}

CONSENT_TEXT = """
================ EXTERNAL AI PROCESSING — READ BEFORE CONTINUING ================
ZIROQ can send your TRANSCRIPT (and, in later steps, video frames/clips) to Google's Gemini API.
* On Google's UNPAID tier, Google's terms say submitted content may be used to improve its products and may be read
  by human reviewers (disconnected from your account). Do not send sensitive, confidential or personal material.
* ZERO_COST_MODE cannot detect whether a project has billing enabled. To stay at $0, use keys created in Google AI Studio
  projects that have NO billing account linked. Hard per-run request caps below limit worst-case usage either way.
* Answer anything other than the exact phrase to run LOCAL-ONLY (all fallbacks still produce your five clips).
==================================================================================="""


class GeminiUnavailable(Exception):
    """No key/model can serve the request right now. Callers must use their local fallback."""
    def __init__(self, message, waiting_for_quota=False, attempts=0):
        super().__init__(message)
        self.waiting_for_quota, self.attempts = waiting_for_quota, attempts


class GeminiRequestError(Exception):
    """The request itself is invalid (bad schema/arguments); retrying or rotating keys cannot help."""


# ----------------------------------------------------------------------------- secrets & scrubbing
_GEM_KEY_RE = re.compile(r"AIza[0-9A-Za-z_\-]{20,}")
_GEM_KNOWN_SECRETS = []


def gem_scrub(text):
    text = str(text)
    for secret in _GEM_KNOWN_SECRETS:
        if secret:
            text = text.replace(secret, "[redacted-key]")
    return _GEM_KEY_RE.sub("[redacted-key]", text)


def gem_log(message):
    log("gemini", gem_scrub(message))


def _gemini_secret(name):
    value = os.environ.get(name, "")
    if value:
        return str(value).strip()
    try:
        from google.colab import userdata
        return str(userdata.get(name) or "").strip()
    except Exception:
        return ""


def detect_gemini_keys():
    found, seen = [], set()
    def add(value):
        value = str(value or "").strip().strip("\"'")
        if value and value not in seen:
            seen.add(value); found.append(value)
    for name in [f"GEMINI_API_KEY_{i}" for i in range(1, 13)] + ["GEMINI_API_KEY", "GOOGLE_API_KEY"]:
        add(_gemini_secret(name))
    for part in re.split(r"[,\s;]+", _gemini_secret("GEMINI_API_KEYS") or ""):
        add(part)
    return [(f"key{i}", k) for i, k in enumerate(found, 1)]


def gemini_project_map(labels):
    """Key label -> project label. Default follows CFG['gemini_keys_are_separate_projects']; override with a secret
    GEMINI_KEY_PROJECTS such as 'key1=A,key2=A,key3=B' when some keys share a project (they then share one quota)."""
    explicit = {}
    for part in re.split(r"[,\n;]+", _gemini_secret("GEMINI_KEY_PROJECTS") or ""):
        if "=" in part:
            a, b = part.split("=", 1)
            explicit[a.strip()] = b.strip()
    separate = CFG.get("gemini_keys_are_separate_projects", True)
    return {lab: explicit.get(lab, f"project-{lab}" if separate else "project-shared") for lab in labels}


# ----------------------------------------------------------------------------- error classification
def classify_gemini_error(exc):
    text = gem_scrub(f"{type(exc).__name__}: {exc}")
    low = text.lower()
    code = None
    for attr in ("code", "status_code", "http_status"):
        v = getattr(exc, attr, None)
        if isinstance(v, int):
            code = v; break
        if isinstance(v, str) and v.isdigit():
            code = int(v); break
    if code is None:
        m = re.search(r"\b(400|401|403|404|408|429|500|502|503|504)\b", low)
        code = int(m.group(1)) if m else None
    retry_after = None
    m = re.search(r"retry(?:\s*in|delay['\"]?\s*[:=]\s*['\"]?)\s*(\d+(?:\.\d+)?)\s*s", low)
    if m:
        retry_after = float(m.group(1))
    auth_markers = ("api key not valid", "api_key_invalid", "api key expired", "unauthenticated", "permission_denied",
                    "reported as leaked", "key has been revoked", "invalid api key", "consumer_suspended",
                    "api_key_service_blocked", "api key not found")
    if code == 401 or code == 403 or any(x in low for x in auth_markers):
        kind = "auth"
    elif code == 429 or "resource_exhausted" in low or "quota" in low:
        if "limit: 0" in low:
            kind = "quota_zero"
        elif any(x in low for x in ("perday", "per day", "daily", "requestsperday")):
            kind = "quota_daily"
        else:
            kind = "rate_limit"
    elif code == 404 or "not_found" in low or "no longer available" in low or "is not supported" in low:
        kind = "model_unavailable"
    elif code == 400 or "invalid_argument" in low:
        kind = "bad_request"
    elif isinstance(exc, TimeoutError) or code in (408, 504) or any(x in low for x in ("timeout", "timed out", "deadline")):
        kind = "timeout"
    elif code in (500, 502, 503) or any(x in low for x in ("unavailable", "internal", "overloaded", "connection", "reset by peer")):
        kind = "transient"
    else:
        kind = "unknown"
    return {"kind": kind, "code": code, "retry_after": retry_after, "message": text[:400]}


def _next_pacific_midnight(now):
    tz = ZoneInfo("America/Los_Angeles")
    dt = datetime.fromtimestamp(now, tz)
    return (dt + timedelta(days=1)).replace(hour=0, minute=0, second=5, microsecond=0).timestamp()


def _default_client_factory(key):
    from google import genai
    timeout_ms = int(float(CFG.get("gemini_request_timeout_sec", 180)) * 1000)
    try:
        return genai.Client(api_key=key, http_options={"timeout": timeout_ms})
    except TypeError:
        return genai.Client(api_key=key)


class GeminiResult:
    def __init__(self, text, model, project, key_label, attempts):
        self.text, self.model, self.project, self.key_label, self.attempts = text, model, project, key_label, attempts


# ----------------------------------------------------------------------------- key + quota manager
class GeminiPool:
    """Credential failover + PER-PROJECT quota tracking. Keys of the same project share one cooldown, so rotating
    among them never bypasses a quota; keys of different projects are tracked independently."""

    def __init__(self, keys, client_factory=None, state_path=None, project_map=None):
        project_map = project_map or {}
        self.client_factory = client_factory or _default_client_factory
        self.state_path = state_path
        self.entries = []
        for label, key in keys:
            _GEM_KNOWN_SECRETS.append(key)
            self.entries.append({"label": label, "project": project_map.get(label, f"project-{label}"), "key": key,
                                 "fp": hashlib.sha256(key.encode()).hexdigest()[:8], "client": None,
                                 "auth_failed": False, "auth_ts": 0.0, "fail_streak": 0, "cool_until": 0.0,
                                 "requests": 0, "errors": 0, "last_call": 0.0, "validated": 0.0})
        self.cooldowns, self.blocked = {}, {}
        self.requests_made = 0
        self.timeouts_in_row, self.breaker_until = 0, 0.0
        self.enabled = bool(self.entries)
        self.disabled_reason = "" if self.entries else "no Gemini API keys configured"
        self.capabilities, self.call_log = {}, []
        self._load_state()

    # -- persistence of NON-SENSITIVE operational state (fingerprints only, never keys)
    def _load_state(self):
        self.state = {"keys": {}, "cooldowns": {}, "blocked": {}}
        try:
            if self.state_path and os.path.isfile(self.state_path):
                with open(self.state_path, encoding="utf-8") as f:
                    self.state = json.load(f)
        except Exception:
            self.state = {"keys": {}, "cooldowns": {}, "blocked": {}}
        now = GEM_NOW()
        for e in self.entries:
            ks = self.state.get("keys", {}).get(e["fp"], {})
            if ks.get("auth_failed") and now - ks.get("auth_ts", 0) < 12 * 3600:
                e["auth_failed"], e["auth_ts"] = True, ks.get("auth_ts", 0)
            e["validated"] = ks.get("validated", 0.0)
            for model, until in self.state.get("cooldowns", {}).get(e["fp"], {}).items():
                if until > now:
                    self.cooldowns[(e["project"], model)] = max(self.cooldowns.get((e["project"], model), 0), until)
            for model, reason in self.state.get("blocked", {}).get(e["fp"], {}).items():
                self.blocked[(e["project"], model)] = reason

    def save_state(self):
        if not self.state_path:
            return
        keys, cds, blk = {}, {}, {}
        for e in self.entries:
            keys[e["fp"]] = {"auth_failed": e["auth_failed"], "auth_ts": e["auth_ts"], "validated": e["validated"],
                             "requests": e["requests"], "errors": e["errors"], "label": e["label"], "project": e["project"]}
            cds[e["fp"]] = {m: u for (p, m), u in self.cooldowns.items() if p == e["project"] and u > GEM_NOW()}
            blk[e["fp"]] = {m: r for (p, m), r in self.blocked.items() if p == e["project"]}
        try:
            atomic_write_json(self.state_path, {"saved_utc": datetime.now(timezone.utc).isoformat(),
                                                "keys": keys, "cooldowns": cds, "blocked": blk})
        except Exception:
            pass

    # -- selection
    def _client(self, entry):
        if entry["client"] is None:
            entry["client"] = self.client_factory(entry["key"])
        return entry["client"]

    def _select(self, model, pin=None):
        """Return (entry, 0) for a usable key, or (None, seconds_until_one_frees) or (None, None) if none ever will.
        `pin` restricts selection to one key label (uploaded files can only be read by the key/project that uploaded them)."""
        now = GEM_NOW()
        projects = {}
        for e in self.entries:
            if pin is not None and e["label"] != pin:
                continue
            projects.setdefault(e["project"], []).append(e)
        best, min_wait = None, None
        for proj, es in projects.items():
            if (proj, model) in self.blocked:
                continue
            healthy = [e for e in es if not e["auth_failed"]]
            if not healthy:
                continue
            cd = self.cooldowns.get((proj, model), 0.0)
            if cd > now:
                min_wait = (cd - now) if min_wait is None else min(min_wait, cd - now)
                continue
            avail = [e for e in healthy if e["cool_until"] <= now]
            if not avail:
                wait = min(e["cool_until"] for e in healthy) - now
                min_wait = wait if min_wait is None else min(min_wait, wait)
                continue
            proj_last = max(e["last_call"] for e in es)
            cand = min(avail, key=lambda e: e["last_call"])
            if best is None or proj_last < best[0]:
                best = (proj_last, cand)
        if best:
            return best[1], 0.0
        return None, min_wait

    def usable_projects(self, model):
        return sorted({e["project"] for e in self.entries if not e["auth_failed"] and (e["project"], model) not in self.blocked})

    def _pace(self, entry):
        interval = float(CFG.get("gemini_min_interval_sec", 4.0))
        proj_last = max(e["last_call"] for e in self.entries if e["project"] == entry["project"])
        gap = interval - (GEM_NOW() - proj_last)
        if gap > 0:
            GEM_SLEEP(gap)

    def _invoke(self, entry, model, input, response_format, system_instruction, generation_config):
        client = self._client(entry)
        kwargs = {"model": model, "input": input, "store": False}   # store=False: do not keep prompts server-side
        if response_format:
            kwargs["response_format"] = response_format
        if system_instruction:
            kwargs["system_instruction"] = system_instruction
        if generation_config:
            kwargs["generation_config"] = generation_config
        return client.interactions.create(**kwargs)

    def _handle_error(self, entry, model, info):
        kind, now, proj = info["kind"], GEM_NOW(), entry["project"]
        if kind == "auth":
            entry["auth_failed"], entry["auth_ts"] = True, now
            gem_log(f"{entry['label']} rejected (authentication/permission); excluded for this session")
        elif kind == "rate_limit":
            wait = max(5.0, info["retry_after"] or 30.0) + random.uniform(0, 3)
            self.cooldowns[(proj, model)] = now + wait
            gem_log(f"{proj}: rate limit on {model}; cooling down {wait:.0f}s (project-level, other keys of this project share it)")
        elif kind == "quota_daily":
            self.cooldowns[(proj, model)] = _next_pacific_midnight(now)
            gem_log(f"{proj}: daily quota exhausted for {model}; paused until midnight Pacific")
        elif kind in ("quota_zero", "model_unavailable"):
            self.blocked[(proj, model)] = kind
            gem_log(f"{proj}: {model} not available to this project ({kind})")
        else:  # transient / timeout / unknown / empty -> exponential backoff with jitter, then fail over to another key
            entry["fail_streak"] += 1
            if kind != "timeout":      # a timeout already cost the full wait; sleeping on top of it only makes the user wait more
                GEM_SLEEP(min(8.0, 2 ** min(entry["fail_streak"], 3) + random.uniform(0, 1.0)))
            if entry["fail_streak"] >= 2:
                entry["cool_until"] = GEM_NOW() + 45.0
            gem_log(f"{entry['label']}: {kind} on {model}; backing off (streak {entry['fail_streak']}) — {gem_scrub(str(info.get('message', '')))[:110]}")
        self.save_state()

    def _invoke_limited(self, remaining, *args):
        """Run one request with a hard wall-clock deadline, whatever the SDK does; an abandoned call is simply ignored."""
        import concurrent.futures as cf
        ex = cf.ThreadPoolExecutor(max_workers=1)
        fut = ex.submit(self._invoke, *args)
        try:
            return fut.result(timeout=max(0.2, min(remaining, 600.0)))
        except cf.TimeoutError:
            raise TimeoutError(f"no answer within {max(0.2, remaining):.0f}s; call abandoned") from None
        finally:
            ex.shutdown(wait=False)

    def generate(self, models, input, response_format=None, system_instruction=None, generation_config=None, task_id="", pin=None):
        t_start = GEM_NOW()
        budget_sec = float(CFG.get("gemini_call_budget_sec", 120))
        if GEM_NOW() < self.breaker_until:
            raise GeminiUnavailable("Gemini timed out repeatedly a moment ago; skipping it for now (local fallback runs)", False, 0)
        max_attempts = int(CFG.get("gemini_max_attempts_per_call", 6))
        max_wait = float(CFG.get("gemini_max_wait_sec", 120))
        budget = int(CFG.get("gemini_max_requests_per_run", 60))
        attempts, waited, last_kind, saw_quota = 0, 0.0, None, False
        for model in models:
            while True:
                if self.requests_made >= budget:
                    raise GeminiUnavailable(f"per-run request budget ({budget}) reached", attempts=attempts)
                entry, wait = self._select(model, pin)
                if entry is None:
                    if wait is None:
                        break
                    if waited + wait > max_wait:
                        saw_quota = True
                        break
                    GEM_SLEEP(wait + random.uniform(0, 1)); waited += wait
                    continue
                if attempts >= max_attempts:
                    raise GeminiUnavailable(f"gave up after {attempts} attempts (last: {last_kind})", saw_quota, attempts)
                remaining = budget_sec - (GEM_NOW() - t_start)
                if remaining <= 0.5:
                    raise GeminiUnavailable(f"time budget of {budget_sec:.0f}s reached (last: {last_kind})", saw_quota, attempts)
                attempts += 1
                self._pace(entry)
                self.requests_made += 1
                entry["requests"] += 1
                entry["last_call"] = GEM_NOW()
                try:
                    interaction = self._invoke_limited(remaining, entry, model, input, response_format, system_instruction, generation_config)
                    text = getattr(interaction, "output_text", None)
                    if not text or not str(text).strip():
                        raise ValueError("empty output (possibly blocked by safety filters)")
                    entry["fail_streak"] = 0
                    self.timeouts_in_row = 0
                    self.call_log.append({"task": task_id, "model": model, "project": entry["project"], "key": entry["label"],
                                          "utc": datetime.now(timezone.utc).isoformat()})
                    self.save_state()
                    return GeminiResult(str(text), model, entry["project"], entry["label"], attempts)
                except Exception as exc:
                    info = classify_gemini_error(exc)
                    last_kind = info["kind"]
                    entry["errors"] += 1
                    saw_quota = saw_quota or info["kind"] in ("rate_limit", "quota_daily", "quota_zero")
                    if info["kind"] == "bad_request":
                        raise GeminiRequestError(info["message"]) from None
                    self._handle_error(entry, model, info)
                    if info["kind"] == "timeout":
                        self.timeouts_in_row += 1
                        if self.timeouts_in_row >= 2:
                            self.breaker_until = GEM_NOW() + float(CFG.get("gemini_breaker_sec", 600))
                            raise GeminiUnavailable("Gemini timed out twice in a row; pausing it for a few minutes (local fallback runs)", saw_quota, attempts)
        raise GeminiUnavailable(f"no Gemini key/model could serve the request (last: {last_kind})", saw_quota, attempts)

    def upload_file(self, path, poll_timeout=120):
        """Upload media with the first healthy key; returns {label, name, uri, mime_type}. Use generate(..., pin=label) afterwards."""
        last = "no usable key"
        for e in self.entries:
            if e["auth_failed"]:
                continue
            try:
                client = self._client(e)
                f = client.files.upload(file=path)
                def state(x):
                    s = getattr(x, "state", None)
                    return "ACTIVE" if s is None else str(getattr(s, "name", s) or "").upper().replace("FILESTATE.", "")
                waited = 0
                while state(f) in ("PROCESSING", "STATE_UNSPECIFIED", "") and waited < poll_timeout:
                    GEM_SLEEP(5); waited += 5
                    f = client.files.get(name=f.name)
                if state(f) != "ACTIVE":
                    raise RuntimeError(f"uploaded file state is {state(f)}")
                return {"label": e["label"], "name": f.name, "uri": f.uri, "mime_type": getattr(f, "mime_type", None) or "video/mp4"}
            except Exception as exc:
                info = classify_gemini_error(exc)
                last = info["kind"]
                if info["kind"] == "auth":
                    e["auth_failed"], e["auth_ts"] = True, GEM_NOW()
                gem_log(f"{e['label']}: media upload failed ({info['kind']})")
        raise GeminiUnavailable(f"could not upload media to any Gemini key (last: {last})")

    def delete_file(self, info):
        try:
            entry = next(e for e in self.entries if e["label"] == info["label"])
            self._client(entry).files.delete(name=info["name"])
        except Exception:
            pass

    def validate_keys(self):
        """Cheap key check via models.list (not a generation request); cached 12h; inconclusive results never disable a key."""
        now = GEM_NOW()
        for e in self.entries:
            if e["auth_failed"] or (now - e["validated"] < 12 * 3600):
                continue
            try:
                listing = self._client(e).models.list()
                next(iter(listing), None)
                e["validated"] = now
            except Exception as exc:
                info = classify_gemini_error(exc)
                if info["kind"] == "auth":
                    e["auth_failed"], e["auth_ts"] = True, now
                gem_log(f"{e['label']}: validation {'FAILED (auth)' if info['kind'] == 'auth' else 'inconclusive (' + info['kind'] + ')'}")
        self.save_state()

    def health_report(self):
        rows = []
        for e in self.entries:
            rows.append({"key": e["label"], "fingerprint": e["fp"][:4], "project": e["project"],
                         "health": "auth_failed" if e["auth_failed"] else "ok", "requests": e["requests"], "errors": e["errors"]})
        return rows


# ----------------------------------------------------------------------------- structured output client
def validate_json_schema(value, schema, path="$"):
    """Minimal JSON-Schema validator (type/required/properties/items/enum/min/max/minItems/maxItems/additionalProperties)."""
    errors = []
    types = schema.get("type")
    if types is not None:
        tl = types if isinstance(types, list) else [types]
        def ok_type(t):
            return ((t == "string" and isinstance(value, str)) or (t == "boolean" and isinstance(value, bool)) or
                    (t == "integer" and isinstance(value, int) and not isinstance(value, bool)) or
                    (t == "number" and isinstance(value, (int, float)) and not isinstance(value, bool)) or
                    (t == "object" and isinstance(value, dict)) or (t == "array" and isinstance(value, list)) or
                    (t == "null" and value is None))
        if not any(ok_type(t) for t in tl):
            return [f"{path}: expected {'|'.join(tl)}, got {type(value).__name__}"]
    if "enum" in schema and value not in schema["enum"]:
        errors.append(f"{path}: {value!r} not in {schema['enum']}")
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if "minimum" in schema and value < schema["minimum"]:
            errors.append(f"{path}: {value} < minimum {schema['minimum']}")
        if "maximum" in schema and value > schema["maximum"]:
            errors.append(f"{path}: {value} > maximum {schema['maximum']}")
    if isinstance(value, dict):
        for req in schema.get("required", []):
            if req not in value:
                errors.append(f"{path}: missing required '{req}'")
        props = schema.get("properties", {})
        for k, v in value.items():
            if k in props:
                errors += validate_json_schema(v, props[k], f"{path}.{k}")
            elif schema.get("additionalProperties") is False:
                errors.append(f"{path}: unexpected property '{k}'")
    if isinstance(value, list):
        if "minItems" in schema and len(value) < schema["minItems"]:
            errors.append(f"{path}: fewer than {schema['minItems']} items")
        if "maxItems" in schema and len(value) > schema["maxItems"]:
            errors.append(f"{path}: more than {schema['maxItems']} items")
        if "items" in schema:
            for i, item in enumerate(value[:200]):
                errors += validate_json_schema(item, schema["items"], f"{path}[{i}]")
    return errors[:10]


def _extract_json(text):
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    return json.loads(t)


def _gemini_cache_key(task_id, prompt, schema, system, media):
    blob = json.dumps({"t": task_id, "p": prompt, "s": schema, "y": system, "m": media}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()


def gemini_models_for(tier):
    names = CFG.get("gemini_model_reasoning") if tier == "reasoning" else CFG.get("gemini_model_fast")
    return list(names or [])


def gemini_structured(task_id, prompt, schema, tier="reasoning", system=None, media=None, validator=None,
                      max_repairs=1, temperature=0.2, use_cache=True, pin=None, cache_media=None):
    """Run one structured Gemini task. NEVER raises for availability problems:
    returns {"ok": False, "fallback_required": True, "error_kind": ...} so the caller uses its local fallback."""
    out = {"ok": False, "task_id": task_id, "data": None, "fallback_required": True, "source": None,
           "model": None, "project": None, "error_kind": None, "error": None, "attempts": 0}
    pool = globals().get("GEMINI")
    if pool is None or not getattr(pool, "enabled", False):
        out.update(error_kind="disabled", error=getattr(pool, "disabled_reason", "Gemini disabled") if pool else "Gemini not initialised")
        return out
    if len(prompt) > int(CFG.get("gemini_prompt_char_cap", 400000)):
        out.update(error_kind="request", error=f"prompt too large ({len(prompt)} chars)")
        return out
    cache_path = os.path.join(GEMINI_CACHE_DIR, _gemini_cache_key(task_id, prompt, schema, system, cache_media if cache_media is not None else media) + ".json")
    if use_cache and os.path.isfile(cache_path):
        try:
            with open(cache_path, encoding="utf-8") as f:
                cached = json.load(f)
            out.update(ok=True, data=cached["data"], fallback_required=False, source="cache",
                       model=cached.get("model"), project=cached.get("project"))
            return out
        except Exception:
            pass
    response_format = {"type": "text", "mime_type": "application/json", "schema": schema}
    request_input = [media, {"type": "text", "text": prompt}] if media else prompt
    gen_cfg = {"temperature": temperature}
    current_prompt, last_errors = prompt, []
    for repair in range(max_repairs + 1):
        if repair:
            current_prompt = (prompt + "\n\nYour previous answer was rejected:\n- " + "\n- ".join(last_errors[:8]) +
                              "\nReturn corrected JSON only, obeying the schema and the sentence-ID rules.")
            request_input = [media, {"type": "text", "text": current_prompt}] if media else current_prompt
        try:
            res = (pool.generate(gemini_models_for(tier), request_input, response_format, system, gen_cfg, task_id, pin=pin) if pin is not None
                   else pool.generate(gemini_models_for(tier), request_input, response_format, system, gen_cfg, task_id))
        except GeminiRequestError as exc:
            out.update(error_kind="request", error=gem_scrub(exc)); return out
        except GeminiUnavailable as exc:
            out.update(error_kind="waiting_for_quota" if exc.waiting_for_quota else "unavailable",
                       error=gem_scrub(exc), attempts=exc.attempts); return out
        out.update(model=res.model, project=res.project, attempts=out["attempts"] + res.attempts)
        try:
            data = _extract_json(res.text)
        except Exception as exc:
            last_errors = [f"output was not valid JSON ({type(exc).__name__})"]; continue
        last_errors = validate_json_schema(data, schema)
        if not last_errors and validator is not None:
            try:
                last_errors = list(validator(data) or [])
            except Exception as exc:
                last_errors = [f"validator error: {type(exc).__name__}"]
        if not last_errors:
            out.update(ok=True, data=data, fallback_required=False, source="gemini")
            try:
                atomic_write_json(cache_path, {"data": data, "model": res.model, "project": res.project,
                                               "task_id": task_id, "cached_utc": datetime.now(timezone.utc).isoformat()})
            except Exception:
                pass
            return out
    out.update(error_kind="invalid_output", error="; ".join(last_errors[:5]))
    return out


# ----------------------------------------------------------------------------- stage tracker
STAGE_STATUSES = ("pending", "running", "completed", "failed", "waiting_for_quota", "completed_using_fallback", "requires_review")
STAGES = {}


def stage_set(name, status, decided_by=None, detail=""):
    assert status in STAGE_STATUSES, status
    STAGES[name] = {"status": status, "decided_by": decided_by, "detail": detail, "utc": datetime.now(timezone.utc).isoformat()}
    STATE["stages"] = STAGES
    save_state(f"stage:{name}", status=status, decided_by=decided_by)


def print_stage_report():
    print("\nSTAGE REPORT")
    for name, row in STAGES.items():
        print(f"  {name:<34} {row['status']:<26} {row['decided_by'] or '-':<22} {row['detail'][:70]}")


# ----------------------------------------------------------------------------- live capability probe (explicit, small)
def gemini_live_probe(pool, include_video=True, include_agentic=True):
    """A handful of tiny requests that REPORT what the account can actually do; nothing is assumed from model names."""
    schema = {"type": "object", "properties": {"ok": {"type": "boolean"}}, "required": ["ok"]}
    rf = {"type": "text", "mime_type": "application/json", "schema": schema}
    report = {"checked_utc": datetime.now(timezone.utc).isoformat(), "structured": {}, "video": {}, "agentic": {}}
    top = gemini_models_for("reasoning")[:1]
    models = list(dict.fromkeys(top + gemini_models_for("fast")[:1]))
    def attempt(models_, inp):
        try:
            _old_attempts = CFG.get("gemini_max_attempts_per_call", 6)
            CFG["gemini_max_attempts_per_call"] = 2        # a probe must never spend minutes retrying
            try:
                res = pool.generate(models_, inp, rf, None, {"temperature": 0.0}, "probe")
            finally:
                CFG["gemini_max_attempts_per_call"] = _old_attempts
            return {"supported": bool(_extract_json(res.text).get("ok") in (True, False)), "model": res.model, "project": res.project}
        except GeminiUnavailable as exc:
            return {"supported": None, "error": gem_scrub(exc)}
        except GeminiRequestError as exc:
            return {"supported": False, "error": gem_scrub(exc)}
        except Exception as exc:
            return {"supported": None, "error": gem_scrub(f"{type(exc).__name__}: {exc}")}
    for m in models:
        report["structured"][m] = attempt([m], 'Reply with JSON {"ok": true}.')
    if include_video and top and any(r.get("supported") for r in report["structured"].values()):
        path = os.path.join(RUN_DIR, "gemini_probe.mp4")
        if not os.path.isfile(path):
            run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "testsrc=size=160x120:rate=5:duration=2",
                 "-c:v", "libx264", "-pix_fmt", "yuv420p", path], quiet=True)
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        part = {"type": "video", "data": b64, "mime_type": "video/mp4"}
        report["video"][top[0]] = attempt(top, [part, {"type": "text", "text": 'Is there video content? Reply JSON {"ok": true}.'}])
        if include_agentic:
            apart = dict(part, processing="agentic")
            report["agentic"][top[0]] = attempt(top, [apart, {"type": "text", "text": 'Is there video content? Reply JSON {"ok": true}.'}])
    pool.capabilities = report
    return report


# ----------------------------------------------------------------------------- capability parity (honest status)
PARITY_MATRIX = [
    ("Topic discovery",            "Gemini topic map",                     "CELL 6 lexicon/embedding content map",              "implemented"),
    ("Story reconstruction",       "Gemini sentence-ID story proposals",   "CELL 7 multi-interval story builder",               "implemented"),
    ("Clip selection / hooks",     "Gemini ranking + hook reasoning",      "CELL 7 editorial scoring",                          "implemented"),
    ("Story validation",           "Gemini critique of each candidate",    "Editorial QA (transcript-verified) + review flags", "implemented"),
    ("Whole-video visual understanding", "Gemini video / agentic video",   "scene detection + OCR + keyframes",                 "partial (director sees video; local scene/OCR not built)"),
    ("Portrait composition",       "Gemini visual direction",              "face detector + fit-blur layout rules",             "partial (batch 1)"),
    ("B-roll research / verification", "Gemini visual search brief",       "provider adapters + metadata ranking",              "implemented (Gemini re-ranks results)"),
    ("Music selection",            "Gemini mood/energy brief",             "mood classifier + licensed pool + procedural bed",  "implemented (Gemini brief + re-rank)"),
    ("Sound-effects cue plan",     "Gemini cue sheet",                     "rule-based cue planner (joins) + procedural SFX",   "implemented"),
    ("Preview review / correction","Gemini reviews contact sheets",        "automated technical QA",                            "partial (existing)"),
]


def print_parity_matrix():
    print("\nCAPABILITY PARITY (Gemini task -> independent local fallback)")
    for cap, g, local, status in PARITY_MATRIX:
        print(f"  {cap:<34} | {local:<52} | {status}")


# ----------------------------------------------------------------------------- consent + setup
def ensure_external_ai_consent():
    choice = CFG.get("external_ai_enabled")
    if choice is True:
        return True
    if choice is False:
        return False
    pre = (os.environ.get("ZIROQ_GEMINI_CONSENT", "") or _gemini_secret("ZIROQ_GEMINI_CONSENT") or "").strip()
    if pre:                       # typed into the Start-here form (or a Colab Secret): no console input box needed (phone-friendly)
        return pre == "I AGREE"
    print(CONSENT_TEXT)
    try:
        answer = input('Type "I AGREE" to enable Gemini (anything else = local-only): ')
    except (EOFError, OSError):
        answer = ""
    return answer.strip() == "I AGREE"


def setup_gemini(factory=None, state_path=None):
    keys = detect_gemini_keys()
    if not keys:
        pool = GeminiPool([], factory, state_path)
        gem_log("NO KEYS FOUND, so there is no consent prompt. In Colab: left sidebar > key icon (Secrets) > add GEMINI_API_KEY_1 ... GEMINI_API_KEY_5, "
                "switch 'Notebook access' ON for each, then re-run this cell. Running with local fallbacks only.")
        return pool
    try:
        version = importlib.metadata.version("google-genai")
        major_minor = tuple(int(x) for x in re.findall(r"\d+", version)[:2])
        sdk_ok = major_minor >= (2, 3)
    except Exception:
        version, sdk_ok = None, False
    pool = GeminiPool(keys, factory, state_path if state_path else GEMINI_STATE_PATH,
                      gemini_project_map([k for k, _ in keys]))
    if factory is None and not sdk_ok:
        pool.enabled, pool.disabled_reason = False, f"google-genai>=2.3.0 required (found {version}); run Cell 0 again"
        gem_log(pool.disabled_reason)
        return pool
    if not ensure_external_ai_consent():
        pool.enabled, pool.disabled_reason = False, "external AI processing not authorised by the user"
        gem_log("external AI processing NOT enabled; ZIROQ runs entirely on its local components")
        return pool
    pool.validate_keys()
    if not any(not e["auth_failed"] for e in pool.entries):
        pool.enabled, pool.disabled_reason = False, "no valid Gemini key"
    return pool


if "GEMINI" not in globals() or not getattr(globals().get("GEMINI"), "enabled", False) or globals().get("GEMINI_FORCE_SETUP", False):
    GEMINI = setup_gemini()
print("\nGEMINI STATUS:", "ENABLED" if GEMINI.enabled else f"local-only ({GEMINI.disabled_reason})")
for row in GEMINI.health_report():
    print(f"  {row['key']} (fp {row['fingerprint']}…) project={row['project']} health={row['health']}")
if GEMINI.enabled:
    print("  zero_cost_mode:", CFG.get("zero_cost_mode", True), "| per-run request cap:", CFG.get("gemini_max_requests_per_run"),
          "| reasoning models:", gemini_models_for("reasoning"))
    if CFG.get("gemini_live_probe", True):
        PROBE = gemini_live_probe(GEMINI, include_video=True, include_agentic=CFG.get("gemini_probe_agentic", True))
        for what in ("structured", "video", "agentic"):
            for m, r in PROBE[what].items():
                print(f"  probe {what:<10} {m:<22} supported={r.get('supported')} {('— ' + r['error'][:90]) if r.get('error') else ''}")
print_parity_matrix()
save_state("gemini_ready", enabled=GEMINI.enabled)


GEMINI STATUS: ENABLED
  key1 (fp 371d…) project=project-key1 health=ok
  zero_cost_mode: True | per-run request cap: 60 | reasoning models: ['gemini-3.8-flash', 'gemini-3.6-flash', 'gemini-3.5-flash-lite']

CAPABILITY PARITY (Gemini task -> independent local fallback)
  Topic discovery                    | CELL 6 lexicon/embedding content map                 | implemented
  Story reconstruction               | CELL 7 multi-interval story builder                  | implemented
  Clip selection / hooks             | CELL 7 editorial scoring                             | implemented
  Story validation                   | Editorial QA (transcript-verified) + review flags    | implemented
  Whole-video visual understanding   | scene detection + OCR + keyframes                    | partial (director sees video; local scene/OCR not built)
  Portrait composition               | face detector + fit-blur layout rules                | partial (batch 1)
  B-roll research / verification     | pro

In [ ]:
# CELL 3c — Gemini core tests (fake client; makes NO network calls and uses NO real keys)
def run_gemini_core_tests():
    import types, tempfile
    global GEM_SLEEP, GEM_NOW, GEMINI
    real_sleep, real_now, real_gemini = GEM_SLEEP, GEM_NOW, globals().get("GEMINI")
    old_cfg = {k: CFG.get(k) for k in ("gemini_min_interval_sec", "gemini_max_requests_per_run", "gemini_max_wait_sec", "gemini_call_budget_sec")}
    clock = {"t": 1_800_000_000.0}
    slept = []
    GEM_NOW = lambda: clock["t"]
    def fake_sleep(s):
        slept.append(s); clock["t"] += s
    GEM_SLEEP = fake_sleep
    CFG.update(gemini_min_interval_sec=0.0, gemini_max_requests_per_run=50, gemini_max_wait_sec=120)
    tmp = tempfile.mkdtemp()
    global GEMINI_CACHE_DIR
    old_cache = GEMINI_CACHE_DIR
    GEMINI_CACHE_DIR = os.path.join(tmp, "cache")

    class Err(Exception):
        def __init__(self, msg, code=None):
            super().__init__(msg); self.code = code

    class Fake:
        """script: dict key -> list of outcomes (str = text, Exception = raise); last item repeats."""
        def __init__(self, key, script, log_):
            self.key, self.script, self.log_ = key, script, log_
            self.interactions = types.SimpleNamespace(create=self.create)
            self.models = types.SimpleNamespace(list=lambda: iter([1]))
        def create(self, **kw):
            self.log_.append((self.key, kw["model"], kw.get("store")))
            seq = self.script.get(self.key, ['{"ok": true}'])
            item = seq.pop(0) if len(seq) > 1 else seq[0]
            if isinstance(item, Exception):
                raise item
            return types.SimpleNamespace(output_text=item)

    def make(keys, script, projects=None):
        calls = []
        pool = GeminiPool([(f"key{i}", k) for i, k in enumerate(keys, 1)], lambda k: Fake(k, script, calls), None,
                          projects or {f"key{i}": f"P{i}" for i in range(1, len(keys) + 1)})
        return pool, calls

    SCHEMA = {"type": "object", "properties": {"n": {"type": "integer"}}, "required": ["n"]}
    FAKEKEY1, FAKEKEY2, FAKEKEY3 = "AIza" + "A" * 35, "AIza" + "B" * 35, "AIza" + "C" * 35
    try:
        # 1. error classification
        assert classify_gemini_error(Err("API key not valid. Please pass a valid API key", 400))["kind"] == "auth"
        assert classify_gemini_error(Err("RESOURCE_EXHAUSTED retry in 17s", 429))["retry_after"] == 17.0
        assert classify_gemini_error(Err("429 quota exceeded for metric per day", 429))["kind"] == "quota_daily"
        assert classify_gemini_error(Err("quota exceeded, limit: 0", 429))["kind"] == "quota_zero"
        assert classify_gemini_error(Err("model not found", 404))["kind"] == "model_unavailable"
        assert classify_gemini_error(Err("boom", 503))["kind"] == "transient"
        assert classify_gemini_error(TimeoutError("x"))["kind"] == "timeout"
        assert classify_gemini_error(Err("bad schema", 400))["kind"] == "bad_request"

        # 2. scrubbing: a key never survives gem_scrub, even inside exception text
        GeminiPool([("key1", FAKEKEY1)], lambda k: None, None, {})
        assert FAKEKEY1 not in gem_scrub(f"failed with {FAKEKEY1} and AIza{'Z'*30}")

        # 3. no keys / disabled -> graceful fallback result, never an exception
        GEMINI = GeminiPool([], None, None)
        r = gemini_structured("t_nokeys", "x", SCHEMA)
        assert not r["ok"] and r["fallback_required"] and r["error_kind"] == "disabled"

        # 4. success + store=False + cache hit performs no second request
        pool, calls = make([FAKEKEY1], {FAKEKEY1: ['{"n": 3}']})
        GEMINI = pool
        r = gemini_structured("t_ok", "hello", SCHEMA)
        assert r["ok"] and r["data"] == {"n": 3} and calls[0][2] is False
        r2 = gemini_structured("t_ok", "hello", SCHEMA)
        assert r2["ok"] and r2["source"] == "cache" and len(calls) == 1

        # 5. invalid key fails over to the next key; the bad key is excluded afterwards
        pool, calls = make([FAKEKEY1, FAKEKEY2], {FAKEKEY1: [Err("API key not valid", 400)], FAKEKEY2: ['{"n": 1}']})
        GEMINI = pool
        r = gemini_structured("t_auth", "p1", SCHEMA)
        assert r["ok"] and pool.entries[0]["auth_failed"] and not pool.entries[1]["auth_failed"]
        gemini_structured("t_auth2", "p2", SCHEMA)
        assert [c[0] for c in calls].count(FAKEKEY1) == 1

        # 6. rate limit: cooldown is per PROJECT, so a second key of the SAME project is not used to bypass it,
        #    while a key from a DIFFERENT project serves immediately.
        same = {"key1": "PA", "key2": "PA", "key3": "PB"}
        pool, calls = make([FAKEKEY1, FAKEKEY2, FAKEKEY3],
                           {FAKEKEY1: [Err("429 RESOURCE_EXHAUSTED retry in 40s", 429)], FAKEKEY2: ['{"n": 2}'], FAKEKEY3: ['{"n": 9}']}, same)
        GEMINI = pool
        r = gemini_structured("t_rl", "p3", SCHEMA)
        assert r["ok"] and r["project"] == "PB" and r["data"] == {"n": 9}
        assert all(c[0] != FAKEKEY2 for c in calls)

        # 7. all projects rate limited -> waits (simulated clock) within max_wait, then succeeds
        pool, calls = make([FAKEKEY1], {FAKEKEY1: [Err("RESOURCE_EXHAUSTED retry in 20s", 429), '{"n": 5}']})
        GEMINI = pool
        r = gemini_structured("t_wait", "p4", SCHEMA)
        assert r["ok"] and sum(slept) >= 20

        # 8. wait too long -> fallback_required with waiting_for_quota
        CFG["gemini_max_wait_sec"] = 5
        pool, calls = make([FAKEKEY1], {FAKEKEY1: [Err("RESOURCE_EXHAUSTED retry in 90s", 429)]})
        GEMINI = pool
        r = gemini_structured("t_wait2", "p5", SCHEMA)
        assert not r["ok"] and r["fallback_required"] and r["error_kind"] == "waiting_for_quota"
        CFG["gemini_max_wait_sec"] = 120

        # 9. daily quota blocks until Pacific midnight (never retried before then); quota_zero model falls through
        pool, calls = make([FAKEKEY1], {FAKEKEY1: [Err("quota exceeded per day", 429)]})
        GEMINI = pool
        r = gemini_structured("t_day", "p6", SCHEMA)
        assert not r["ok"] and pool.cooldowns
        pool, calls = make([FAKEKEY1], {FAKEKEY1: [Err("quota exceeded, limit: 0", 429), '{"n": 4}']})
        GEMINI = pool
        r = gemini_structured("t_zero", "p7", SCHEMA)
        assert r["ok"] and len({c[1] for c in calls}) == 2, calls   # first model blocked, second model served

        # 10. malformed JSON -> one repair attempt succeeds; persistent garbage -> invalid_output fallback
        pool, calls = make([FAKEKEY1], {FAKEKEY1: ["not json at all", '{"n": 7}']})
        GEMINI = pool
        r = gemini_structured("t_rep", "p8", SCHEMA)
        assert r["ok"] and r["data"] == {"n": 7} and len(calls) == 2
        pool, calls = make([FAKEKEY1], {FAKEKEY1: ['{"n": "wrong"}']})
        GEMINI = pool
        r = gemini_structured("t_bad", "p9", SCHEMA)
        assert not r["ok"] and r["error_kind"] == "invalid_output" and r["fallback_required"]

        # 11. semantic validator (e.g. hallucinated sentence ids) triggers repair, then rejection
        pool, calls = make([FAKEKEY1], {FAKEKEY1: ['{"n": 99}', '{"n": 1}']})
        GEMINI = pool
        r = gemini_structured("t_val", "p10", SCHEMA, validator=lambda d: [] if d["n"] < 10 else ["n refers to a non-existent id"])
        assert r["ok"] and r["data"]["n"] == 1

        # 12. transient + timeout: exponential backoff then success; no key is permanently disabled
        pool, calls = make([FAKEKEY1], {FAKEKEY1: [Err("503 unavailable", 503), TimeoutError("timed out"), '{"n": 6}']})
        GEMINI = pool
        r = gemini_structured("t_tr", "p11", SCHEMA)
        assert r["ok"] and not pool.entries[0]["auth_failed"]

        # 13. bad request is not retried or rotated
        pool, calls = make([FAKEKEY1, FAKEKEY2], {FAKEKEY1: [Err("invalid_argument: schema", 400)]})
        GEMINI = pool
        r = gemini_structured("t_req", "p12", SCHEMA)
        assert not r["ok"] and r["error_kind"] == "request" and len(calls) == 1

        # 14. per-run request budget is a hard stop (cost guard)
        CFG["gemini_max_requests_per_run"] = 2
        pool, calls = make([FAKEKEY1], {FAKEKEY1: [Err("503", 503)]})
        GEMINI = pool
        r = gemini_structured("t_bud", "p13", SCHEMA)
        assert not r["ok"] and len(calls) <= 2
        CFG["gemini_max_requests_per_run"] = 50

        # 15. persisted state never contains a key, only fingerprints; and reloads auth failures
        sp = os.path.join(tmp, "state", "g.json")
        pool = GeminiPool([("key1", FAKEKEY1)], lambda k: Fake(k, {FAKEKEY1: [Err("API key not valid", 400)]}, []), sp, {"key1": "P1"})
        GEMINI = pool
        gemini_structured("t_state", "p14", SCHEMA)
        raw = open(sp).read()
        assert FAKEKEY1 not in raw and "AIza" not in raw
        pool2 = GeminiPool([("key1", FAKEKEY1)], lambda k: None, sp, {"key1": "P1"})
        assert pool2.entries[0]["auth_failed"]

        # 15b. file upload is pinned to the uploading key; polling until ACTIVE; delete
        pool, calls = make([FAKEKEY1, FAKEKEY2], {FAKEKEY1: ['{"n": 1}'], FAKEKEY2: ['{"n": 2}']})
        c1 = pool._client(pool.entries[0])
        st = {"polls": 0}
        mk = lambda s: types.SimpleNamespace(name="files/abc", uri="https://x/files/abc", mime_type="video/mp4", state=types.SimpleNamespace(name=s))
        def _get(name):
            st["polls"] += 1; return mk("ACTIVE")
        c1.files = types.SimpleNamespace(upload=lambda file: mk("PROCESSING"), get=_get, delete=lambda name: st.update(deleted=name))
        info = pool.upload_file("x.mp4")
        assert info["label"] == "key1" and st["polls"] >= 1
        GEMINI = pool
        r = gemini_structured("t_pin", "pp", SCHEMA, pin=info["label"], media={"type": "video", "uri": info["uri"]}, cache_media={"local": "v"})
        assert r["ok"] and calls[-1][0] == FAKEKEY1
        pool.delete_file(info); assert st["deleted"] == "files/abc"

        # 15c. a hanging Gemini call is abandoned at the deadline; two timeouts open a breaker so later calls fail instantly
        import time as _t
        class Hang:
            def __init__(self, key): self.interactions = types.SimpleNamespace(create=self.create); self.models = types.SimpleNamespace(list=lambda: iter([1]))
            def create(self, **kw): _t.sleep(1.5); return types.SimpleNamespace(output_text='{"n": 1}')
        CFG["gemini_call_budget_sec"] = 0.6
        pool = GeminiPool([("key1", FAKEKEY1)], lambda k: Hang(k), None, {"key1": "P1"})
        GEMINI = pool
        t0 = _t.time(); r = gemini_structured("t_hang", "p_hang", SCHEMA); took = _t.time() - t0
        assert not r["ok"] and r["fallback_required"] and took < 2.9, (r, took)
        t0 = _t.time(); r2 = gemini_structured("t_hang2", "p_hang2", SCHEMA)
        assert not r2["ok"] and _t.time() - t0 < 0.3     # breaker: no second wait
        CFG["gemini_call_budget_sec"] = 120

        # 16. stage tracker + schema validator
        stage_set("unit_test_stage", "completed_using_fallback", decided_by="local")
        assert STAGES["unit_test_stage"]["status"] == "completed_using_fallback"
        STAGES.pop("unit_test_stage", None)
        assert validate_json_schema({"a": [1, "x"]}, {"type": "object", "properties": {"a": {"type": "array", "items": {"type": "integer"}}}})
        assert not validate_json_schema({"a": [1, 2]}, {"type": "object", "properties": {"a": {"type": "array", "items": {"type": "integer"}}}})
        return True
    finally:
        GEM_SLEEP, GEM_NOW, GEMINI, GEMINI_CACHE_DIR = real_sleep, real_now, real_gemini, old_cache
        CFG.update({k: v for k, v in old_cfg.items() if v is not None})


if CFG.get("run_self_tests"):
    GEMINI_CORE_TESTS_PASSED = run_gemini_core_tests()
    print("GEMINI_CORE_TESTS_PASSED =", GEMINI_CORE_TESTS_PASSED)
else:
    GEMINI_CORE_TESTS_PASSED = None
    print("Self-tests skipped (this is normal). Nothing to do in this cell.")

Self-tests skipped (this is normal). Nothing to do in this cell.


In [ ]:
import sys
import os
import re
import glob
import subprocess
from urllib.parse import urlparse, parse_qs

try:
    import yt_dlp
except ImportError:
    print("yt-dlp is not present in the environment. Installing quietly...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp"], check=True)
    import yt_dlp

def extract_video_id(url):
    parsed = urlparse(url.strip())
    host = parsed.netloc.lower()
    for prefix in ("www.", "m."):
        if host.startswith(prefix):
            host = host[len(prefix):]
    video_id = ""
    if host == "youtu.be":
        video_id = parsed.path.strip("/").split("/")[0]
    elif host in ("youtube.com", "music.youtube.com"):
        if parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [""])[0]
        elif parsed.path.startswith(("/shorts/", "/embed/", "/live/")):
            parts = parsed.path.split("/")
            video_id = parts[2] if len(parts) > 2 else ""
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id or ""):
        raise ValueError("Enter one valid YouTube video URL (playlists are not supported).")
    return video_id

def probe_source_file(path):
    info = ffprobe_json(path)
    video = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), None)
    audio = next((s for s in info.get("streams", []) if s.get("codec_type") == "audio"), None)
    if not video:
        raise RuntimeError("Source has no video stream")
    if not audio:
        raise RuntimeError("Source has no audio stream; transcription requires audio")
    return float(info["format"]["duration"]), int(video["width"]), int(video["height"])

def find_downloaded_source(folder):
    candidates = [p for p in glob.glob(os.path.join(folder, "source.*"))
                  if not p.endswith((".part", ".ytdl", ".json", ".wav"))]
    return max(candidates, key=os.path.getsize) if candidates else None

print("Paste an authorized YouTube video URL OR a local video path uploaded to this Colab runtime.")
USER_INPUT = (os.environ.get("ZIROQ_URL", "").strip() or input("YouTube URL or local video path: ")).strip().strip("'")
if not USER_INPUT:
    raise ValueError("A YouTube URL or local video path is required")

COOKIES_FILE = "/content/www.youtube.com_cookies.txt"

SOURCE_META = {}
if os.path.isfile(USER_INPUT):
    video_path = os.path.abspath(USER_INPUT)
    source_title = Path(video_path).stem
    video_id = safe_filename(Path(video_path).stem, 28) + "_" + sha256_file(video_path)[:10]
    SOURCE_DIR = os.path.join(CACHE_ROOT, video_id)
    os.makedirs(SOURCE_DIR, exist_ok=True)
    log("source", "using supplied local media file")
else:
    video_id = extract_video_id(USER_INPUT)
    SOURCE_DIR = os.path.join(CACHE_ROOT, video_id)
    os.makedirs(SOURCE_DIR, exist_ok=True)
    video_path = find_downloaded_source(SOURCE_DIR)
    if video_path:
        log("download", f"reusing cached file {os.path.basename(video_path)}")
    else:
        log("metadata", "retrieving video metadata; no download started yet")
        meta_opts = {
            "quiet": True,
            "no_warnings": True,
            "noplaylist": True,
            "skip_download": True,
            "format": "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
            "ignore_no_formats_error": True,
            "extractor_args": {
                "youtube": {
                    "player_client": ["mweb", "tv", "web_safari"]
                }
            }
        }
        if os.path.isfile(COOKIES_FILE):
            meta_opts["cookiefile"] = COOKIES_FILE
            log("metadata", f"Using cookies file: {COOKIES_FILE}")
        try:
            with yt_dlp.YoutubeDL(meta_opts) as ydl:
                meta = ydl.extract_info(USER_INPUT, download=False)
        except Exception as exc:
            err_msg = str(exc)
            if "confirm you’re not a bot" in err_msg or "Sign in" in err_msg:
                raise RuntimeError(
                    f"YouTube blocked the request with bot detection: {err_msg}\n\n"
                    f"SOLUTION:\n"
                    f"1. Install a Netscape-format cookie exporter browser extension (e.g., 'Get cookies.txt LOCALLY').\n"
                    f"2. Go to YouTube while signed into your account, and export the cookies text file.\n"
                    f"3. Upload this fresh file to Google Colab as: /content/www.youtube.com_cookies.txt\n"
                    f"4. Re-run this cell to resume!"
                ) from exc
            raise RuntimeError(f"Could not access video metadata: {exc}. For an authorized copy, upload a local file.") from exc
        if meta.get("is_live"):
            raise RuntimeError("Live streams are not supported")
        if meta.get("availability") in ("needs_auth", "subscriber_only", "premium_only"):
            raise RuntimeError("This video requires authentication or membership. Access controls are not bypassed.")
        if meta.get("duration") and meta["duration"] > 4 * 3600:
            log("warning", "Very long source: expect Colab runtime limits")
        source_title = meta.get("title") or video_id
        SOURCE_META = {"channel": meta.get("channel") or meta.get("uploader") or "", "tags": list(meta.get("tags") or [])[:20],
                       "categories": list(meta.get("categories") or [])[:3], "description": str(meta.get("description") or "")[:600]}
        try:
            atomic_write_json(os.path.join(SOURCE_DIR, "source_meta.json"), SOURCE_META)
        except Exception:
            pass

        # Standardize format selection to be fully adaptive and fallback gracefully
        opts = {
            "quiet": False, "no_warnings": True, "noplaylist": True,
            "outtmpl": os.path.join(SOURCE_DIR, "source.%(ext)s"),
            "format": "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
            "merge_output_format": "mp4", "retries": 5,
            "ignore_no_formats_error": True,
            "verbose": True,
            "extractor_args": {
                "youtube": {
                    "player_client": ["mweb", "tv", "web_safari"]
                }
            }
        }
        if os.path.isfile(COOKIES_FILE):
            opts["cookiefile"] = COOKIES_FILE
            log("download", f"Consistently using cookies file: {COOKIES_FILE}")

        with yt_dlp.YoutubeDL(opts) as ydl:
            ydl.download([USER_INPUT])
        video_path = find_downloaded_source(SOURCE_DIR)
        if not video_path:
            raise RuntimeError("Download returned but no complete source media file was found")

source_duration, source_w, source_h = probe_source_file(video_path)
SOURCE = Source(video_id=video_id, title=source_title, duration=source_duration,
                video_path=video_path, width=source_w, height=source_h)
SOURCE_DIR = os.path.join(CACHE_ROOT, video_id)
if not SOURCE_META:
    try:
        SOURCE_META = json.load(open(os.path.join(SOURCE_DIR, "source_meta.json"), encoding="utf-8"))
    except Exception:
        SOURCE_META = {}
TRANSCRIPT_PATH = os.path.join(SOURCE_DIR, "transcript_full.json")
print(f"Source: {SOURCE.title}\nDuration: {SOURCE.duration/60:.1f} minutes | Resolution: {SOURCE.width}x{SOURCE.height}")
save_state("source_ready", source=asdict(SOURCE))

Paste an authorized YouTube video URL OR a local video path uploaded to this Colab runtime.
[02:20:52] metadata           retrieving video metadata; no download started yet
[02:20:52] metadata           Using cookies file: /content/www.youtube.com_cookies.txt
[02:21:00] download           Consistently using cookies file: /content/www.youtube.com_cookies.txt


[debug] Encodings: locale utf-8, fs utf-8, pref utf-8, out UTF-8 (No ANSI), error UTF-8 (No ANSI), screen UTF-8 (No ANSI)
[debug] yt-dlp version stable@2026.10.26 from yt-dlp/yt-dlp [c91f8de0b] (pip) API
[debug] params: {'quiet': False, 'no_warnings': True, 'noplaylist': True, 'outtmpl': '/content/ziroq_work/cache/2c81cbv0SKk/source.%(ext)s', 'format': 'bestvideo[height<=1080]+bestaudio/best[height<=1080]/best', 'merge_output_format': 'mp4', 'retries': 5, 'ignore_no_formats_error': True, 'verbose': True, 'extractor_args': {'youtube': {'player_client': ['mweb', 'tv', 'web_safari']}}, 'cookiefile': '/content/www.youtube.com_cookies.txt', 'js_runtimes': {'deno': {}}, 'remote_components': set(), 'compat_opts': set(), 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-us,en;q=0.5', 'Sec-Fetch-Mode':

[youtube] Extracting URL: https://www.youtube.com/watch?v=2c81cbv0SKk
[youtube] 2c81cbv0SKk: Downloading webpage
[youtube] 2c81cbv0SKk: Downloading mweb client config


[debug] [youtube] Forcing "main" player JS variant for player b559f50c
        original url = /s/player/b559f50c/player_es6.vflset/en_US/base.js
[debug] [youtube] 2c81cbv0SKk: Detected experiment to bind GVS PO Token to video ID for mweb client


[youtube] 2c81cbv0SKk: Downloading mweb player API JSON


[debug] [youtube] 2c81cbv0SKk: Detected a 6s ad skippable after 5s for mweb


... (5 outputs left out)

[debug] [youtube] 2c81cbv0SKk: Detected a 6s ad skippable after 5s for web_safari


[youtube] [jsc:deno] Solving JS challenges using deno


[debug] [youtube] [jsc:deno] Checking if npm packages are cached
[debug] [youtube] [jsc:deno] Running deno: /usr/local/bin/deno run --ext=js --no-code-cache --no-prompt --no-remote --no-lock --node-modules-dir=none --no-config --cached-only -
[debug] WARNING: [youtube] [jsc] Remote components challenge solver script (deno) and NPM package (deno) were skipped. These may be required to solve JS challenges. You can enable these downloads with  --remote-components ejs:github  (recommended) or  --remote-components ejs:npm , respectively. For more information and alternatives, refer to  https://github.com/yt-dlp/yt-dlp/wiki/EJS
[debug] WARNING: [youtube] 2c81cbv0SKk: n challenge solving failed: Some formats may be missing. Ensure you have a supported JavaScript runtime and challenge solver script distribution installed. Review any warnings presented before this message. For more details, refer to  https://github.com/yt-dlp/yt-dlp/wiki/EJS
[debug] WARNING: [youtube] 2c81cbv0SKk: mweb client h

DownloadError: ERROR: [youtube] 2c81cbv0SKk: No video formats found!

In [ ]:
# CELL 5 — Full-source, checkpointed Faster-Whisper transcription
# One implementation only: audio chunks are decoded to NumPy arrays, avoiding the earlier WAV/PyAV failure path.
import traceback


def load_pcm16_wav(path):
    with wave.open(path, "rb") as wf:
        if wf.getframerate() != 16000 or wf.getnchannels() != 1 or wf.getsampwidth() != 2:
            raise RuntimeError("Internal audio chunk must be mono 16 kHz PCM16")
        raw = wf.readframes(wf.getnframes())
    return np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0


def extract_audio_chunk(source, start_sec, end_sec, out_path):
    if os.path.isfile(out_path) and os.path.getsize(out_path) > 44:
        return out_path
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    duration = max(0.1, end_sec - start_sec)
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-ss", f"{start_sec:.3f}",
           "-t", f"{duration:.3f}", "-i", source.video_path, "-vn", "-ac", "1", "-ar", "16000",
           "-c:a", "pcm_s16le", out_path]
    run(cmd, timeout=900, quiet=True)
    if not os.path.isfile(out_path) or os.path.getsize(out_path) <= 44:
        raise RuntimeError(f"Audio chunk extraction failed at {start_sec:.1f}s")
    return out_path


def load_whisper_model(device, model_name):
    from faster_whisper import WhisperModel
    compute = "float16" if device == "cuda" else "int8"
    log("transcribe", f"loading Faster-Whisper {model_name} on {device}/{compute}")
    return WhisperModel(model_name, device=device, compute_type=compute)


def transcribe_chunk(model, audio_array, offset_sec, keep_start, keep_end):
    segments, info = model.transcribe(
        audio_array,
        language=CFG["whisper_language"],
        word_timestamps=True,
        vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 450, "speech_pad_ms": 150},
        beam_size=CFG["whisper_beam_size"],
        condition_on_previous_text=False,
    )
    result = []
    for segment in segments:
        for item in (segment.words or []):
            token = (item.word or "").strip()
            if not token:
                continue
            global_start = offset_sec + float(item.start)
            global_end = offset_sec + float(item.end)
            midpoint = (global_start + global_end) / 2
            # The overlapping edge audio provides context, but only the core interval is retained.
            if keep_start - 0.001 <= midpoint < keep_end - 0.001:
                result.append(Word(token, round(global_start, 3), round(global_end, 3), float(item.probability)))
    return result, getattr(info, "language", "unknown")


def transcribe_full_source(source, transcript_path):
    if os.path.isfile(transcript_path):
        try:
            cached = json.load(open(transcript_path, encoding="utf-8"))
            cached_words = [Word(**w) for w in cached["words"]]
            if cached.get("source_duration") and abs(cached["source_duration"] - source.duration) > 1.0:
                raise ValueError("Cached transcript belongs to a different source duration")
            log("transcribe", f"loaded completed full transcript ({len(cached_words)} words)")
            return Transcript(cached.get("language", "unknown"), cached_words,
                              [Sentence(**s) for s in cached.get("sentences", [])])
        except Exception as exc:
            log("transcribe", f"full transcript cache ignored: {exc}")

    chunk_dir = os.path.join(SOURCE_DIR, "transcript_chunks")
    os.makedirs(chunk_dir, exist_ok=True)
    chunk_sec = float(CFG["transcribe_chunk_sec"])
    overlap = float(CFG["transcribe_overlap_sec"])
    n_chunks = int(math.ceil(source.duration / chunk_sec))
    device = "cuda" if GPU_COUNT > 0 else "cpu"
    model_name = CFG["whisper_gpu_model"] if device == "cuda" else CFG["whisper_cpu_model"]
    try:
        model = load_whisper_model(device, model_name)
    except Exception as exc:
        if device != "cuda":
            raise
        log("transcribe", f"GPU model initialization failed; falling back to CPU ({type(exc).__name__})")
        device, model_name = "cpu", CFG["whisper_cpu_model"]
        model = load_whisper_model(device, model_name)

    all_words, languages = [], []
    for idx in range(n_chunks):
        core_start = idx * chunk_sec
        core_end = min(source.duration, (idx + 1) * chunk_sec)
        checkpoint = os.path.join(chunk_dir, f"chunk_{idx:05d}.json")
        if os.path.isfile(checkpoint):
            try:
                row = json.load(open(checkpoint, encoding="utf-8"))
                if abs(row.get("core_start", -999) - core_start) < 0.01 and abs(row.get("core_end", -999) - core_end) < 0.01:
                    words_here = [Word(**w) for w in row["words"]]
                    all_words.extend(words_here)
                    languages.append(row.get("language", "unknown"))
                    log("transcribe", f"chunk {idx+1}/{n_chunks}: resumed ({len(words_here)} words)")
                    continue
            except Exception:
                pass

        fetch_start = max(0.0, core_start - overlap)
        fetch_end = min(source.duration, core_end + overlap)
        wav_path = os.path.join(chunk_dir, f"audio_{idx:05d}.wav")
        extract_audio_chunk(source, fetch_start, fetch_end, wav_path)
        audio_array = load_pcm16_wav(wav_path)
        try:
            words_here, language = transcribe_chunk(model, audio_array, fetch_start, core_start, core_end)
        except Exception as exc:
            if device == "cuda":
                log("transcribe", f"chunk {idx+1}: CUDA failed, retrying this chunk on CPU: {type(exc).__name__}")
                model = load_whisper_model("cpu", CFG["whisper_cpu_model"])
                device, model_name = "cpu", CFG["whisper_cpu_model"]
                words_here, language = transcribe_chunk(model, audio_array, fetch_start, core_start, core_end)
            else:
                raise
        atomic_write_json(checkpoint, {
            "chunk_index": idx, "core_start": core_start, "core_end": core_end,
            "language": language, "device": device, "model": model_name,
            "words": [asdict(w) for w in words_here],
        })
        all_words.extend(words_here)
        languages.append(language)
        log("transcribe", f"chunk {idx+1}/{n_chunks} complete: {len(words_here)} words | {core_end/60:.1f}/{source.duration/60:.1f} min")
        save_state("transcribing", completed_chunks=idx+1, total_chunks=n_chunks, words=len(all_words), device=device, model=model_name)
        del audio_array

    all_words.sort(key=lambda w: (w.start, w.end))
    # Boundary duplicates are suppressed conservatively without discarding repeated words spoken at different times.
    deduped = []
    for word in all_words:
        if deduped and word.word.lower() == deduped[-1].word.lower() and abs(word.start - deduped[-1].start) < 0.12:
            if word.prob > deduped[-1].prob:
                deduped[-1] = word
        else:
            deduped.append(word)
    if not deduped:
        raise RuntimeError("No words were transcribed from the complete source")
    sentences = build_sentences(deduped)
    language = max(set(languages), key=languages.count) if languages else "unknown"
    transcript = Transcript(language, deduped, sentences)
    atomic_write_json(transcript_path, {
        "source_id": source.video_id, "source_duration": source.duration,
        "language": language, "words": [asdict(w) for w in deduped],
        "sentences": [asdict(s) for s in sentences],
    })
    return transcript


TRANSCRIPT = transcribe_full_source(SOURCE, TRANSCRIPT_PATH)
if not TRANSCRIPT.sentences:
    raise RuntimeError("Transcript contains no reconstructed sentences")
coverage = (max(w.end for w in TRANSCRIPT.words) / SOURCE.duration) if SOURCE.duration else 0.0
print(f"FULL TRANSCRIPT: {len(TRANSCRIPT.words):,} words | {len(TRANSCRIPT.sentences):,} sentences | language={TRANSCRIPT.language}")
print(f"Last transcribed word: {max(w.end for w in TRANSCRIPT.words)/60:.1f} min of {SOURCE.duration/60:.1f} min ({coverage:.1%} timeline coverage)")
save_state("transcribed", words=len(TRANSCRIPT.words), sentences=len(TRANSCRIPT.sentences), language=TRANSCRIPT.language)

[13:56:48] transcribe         loading Faster-Whisper base on cpu/int8
[13:59:44] transcribe         chunk 1/12 complete: 2043 words | 10.0/113.3 min
[14:01:48] transcribe         chunk 2/12 complete: 1982 words | 20.0/113.3 min
[14:03:58] transcribe         chunk 3/12 complete: 2170 words | 30.0/113.3 min
[14:06:00] transcribe         chunk 4/12 complete: 2054 words | 40.0/113.3 min
[14:08:13] transcribe         chunk 5/12 complete: 2342 words | 50.0/113.3 min
[14:10:24] transcribe         chunk 6/12 complete: 2125 words | 60.0/113.3 min
[14:12:26] transcribe         chunk 7/12 complete: 1925 words | 70.0/113.3 min
[14:14:41] transcribe         chunk 8/12 complete: 1931 words | 80.0/113.3 min
[14:16:47] transcribe         chunk 9/12 complete: 2071 words | 90.0/113.3 min
[14:18:52] transcribe         chunk 10/12 complete: 2012 words | 100.0/113.3 min
[14:21:00] transcribe         chunk 11/12 complete: 1944 words | 110.0/113.3 min
[14:21:40] transcribe         chunk 12/12 complete: 630 w

In [ ]:
# CELL 6 — Whole-source content map + optional local semantic embeddings
from collections import Counter, defaultdict

EMOTION_LEXICON = {
    "hope": {"hope", "believe", "possible", "future", "dream", "opportunity", "change"},
    "tension": {"fear", "risk", "danger", "wrong", "crisis", "lose", "failure", "mistake"},
    "surprise": {"unexpected", "shocked", "surprise", "actually", "truth", "secret", "nobody"},
    "amusement": {"funny", "laugh", "joke", "ridiculous", "absurd"},
    "reflection": {"learned", "realize", "understand", "meaning", "remember", "experience"},
}
TOPIC_LEXICON = {
    "money": {"money", "wealth", "rich", "poor", "invest", "debt", "cash", "finance", "financial"},
    "business": {"business", "customer", "company", "market", "sales", "product", "profit", "startup"},
    "entrepreneurship": {"entrepreneur", "founder", "build", "company", "startup", "risk", "venture"},
    "self-improvement": {"habit", "discipline", "growth", "mindset", "goal", "success", "failure", "confidence"},
    "psychology": {"psychology", "behavior", "brain", "mind", "emotion", "decision", "belief"},
    "leadership": {"leader", "team", "manage", "culture", "trust", "leadership", "people"},
    "technology": {"technology", "software", "computer", "ai", "data", "algorithm", "internet"},
    "relationships": {"relationship", "friend", "family", "partner", "marriage", "trust"},
}
HOOK_CUES = {"why", "how", "never", "truth", "secret", "mistake", "wrong", "actually", "nobody", "everyone", "because"}
STORY_CUES = {"when", "then", "i", "my", "we", "years", "happened", "remember", "story", "experience"}
PAYOFF_CUES = {"therefore", "lesson", "learned", "result", "that's why", "which means", "so", "in the end"}
ACTION_CUES = {"start", "stop", "build", "practice", "change", "ask", "write", "measure", "focus", "save", "invest", "decide"}


def tokens(text):
    return re.findall(r"[a-z0-9']+", (text or "").lower())


def classify_text(text):
    low = (text or "").lower()
    tok = set(tokens(low))
    topics = [topic for topic, vocab in TOPIC_LEXICON.items() if len(tok & vocab) >= 1]
    emotions = [emotion for emotion, vocab in EMOTION_LEXICON.items() if len(tok & vocab) >= 1]
    if any(c in low for c in ("because", "the reason", "this is why", "which means")):
        role = "explanation"
    elif any(c in low for c in ("i learned", "the lesson", "what matters", "in the end", "that's why", "the result")):
        role = "payoff"
    elif len(tok & STORY_CUES) >= 2 or any(c in low for c in ("when i was", "one day", "years ago", "i remember")):
        role = "story"
    elif len(tok & ACTION_CUES) >= 2 or any(c in low for c in ("you should", "you need to", "the first step")):
        role = "lesson"
    else:
        role = "claim"
    return topics, emotions, role


def build_content_map(transcript):
    records = []
    starts = [w.start for w in transcript.words]
    for index, sentence in enumerate(transcript.sentences):
        lo = bisect.bisect_left(starts, sentence.start - 0.01)
        hi = bisect.bisect_right(starts, sentence.end + 0.01)
        related_words = transcript.words[lo:hi]
        avg_conf = float(np.mean([w.prob for w in related_words])) if related_words else 0.0
        topics, emotions, role = classify_text(sentence.text)
        records.append({
            "sentence_index": index, "start": sentence.start, "end": sentence.end,
            "text": sentence.text, "topics": topics, "emotions": emotions,
            "role": role, "confidence": avg_conf,
            "word_count": len(related_words),
        })
    return records


def build_sentence_vectors(records):
    texts = [r["text"] for r in records]
    if CFG["use_semantic_embeddings"]:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer(CFG["semantic_model_id"])
            vectors = model.encode(texts, batch_size=64, show_progress_bar=True,
                                   normalize_embeddings=True, convert_to_numpy=True)
            log("semantic", f"using local sentence embeddings: {CFG['semantic_model_id']}")
            return np.asarray(vectors, dtype=np.float32), "sentence-transformers"
        except Exception as exc:
            log("semantic", f"local embedding model unavailable ({type(exc).__name__}); using TF-IDF fallback")
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(max_features=18000, ngram_range=(1, 2), stop_words="english", min_df=1)
    matrix = vectorizer.fit_transform(texts)
    log("semantic", "using local TF-IDF similarity fallback (lexical, not a neural semantic model)")
    return matrix, "tfidf_fallback"


CONTENT_MAP = build_content_map(TRANSCRIPT)
SENTENCE_VECTORS, SIMILARITY_BACKEND = build_sentence_vectors(CONTENT_MAP)
content_map_path = os.path.join(REPORT_DIR, "content_knowledge_map.json")
atomic_write_json(content_map_path, {
    "source_id": SOURCE.video_id, "source_duration": SOURCE.duration,
    "language": TRANSCRIPT.language, "similarity_backend": SIMILARITY_BACKEND,
    "sentence_count": len(CONTENT_MAP), "sentences": CONTENT_MAP,
})

# Summarize dominant topics/emotions from the entire transcript, not a sampled opening section.
topic_counts = Counter(t for row in CONTENT_MAP for t in row["topics"])
emotion_counts = Counter(e for row in CONTENT_MAP for e in row["emotions"])
print("Whole-source topic map:", topic_counts.most_common(10) or "no lexicon matches")
print("Whole-source emotion cues:", emotion_counts.most_common(8) or "no lexicon matches")
print("Similarity backend:", SIMILARITY_BACKEND)
save_state("content_map_ready", sentences=len(CONTENT_MAP), similarity_backend=SIMILARITY_BACKEND)

In [ ]:
# CELL 6b — Gemini DIRECTOR (brain) -> validated ClipPlans (body executes)
# Gemini watches the whole video from its public YouTube link and returns timestamped clip proposals + music direction.
# The notebook NEVER trusts timestamps blindly: each one is snapped to local Whisper sentence boundaries, checked against the
# transcript, de-duplicated and length-capped. Anything Gemini cannot supply is filled by the local selector (CELL 7).
# Free-tier note (Google docs, 2026-11-13): YouTube-URL input is a free preview, max 8 hours of video/day/project, public videos only.
DIRECTOR_SCHEMA = {
    "type": "object",
    "properties": {
        "video_summary": {"type": "string"},
        "clips": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "narrative_type": {"type": "string", "enum": ["lesson", "story", "myth_vs_truth", "framework", "warning", "case_study", "surprising_insight"]},
                "start": {"type": "string"},          # "MM:SS" or "H:MM:SS"
                "end": {"type": "string"},
                "hook": {"type": "string"},
                "why_it_works": {"type": "string"},
                "opening_words": {"type": "string"},  # first ~8 words spoken, used to verify the timestamp
                "closing_words": {"type": "string"},
                "score": {"type": "integer", "minimum": 1, "maximum": 100},
                "topics": {"type": "array", "items": {"type": "string"}},
                "music": {"type": "object", "properties": {
                    "mood": {"type": "string", "enum": ["reflective", "uplifting", "playful", "curious", "documentary"]},
                    "energy": {"type": "string", "enum": ["low", "medium", "high"]},
                    "search_terms": {"type": "string"},   # 2-5 words for a royalty-free music library
                    "avoid": {"type": "string"}},
                    "required": ["mood", "energy", "search_terms"]},
                "broll_query": {"type": "string"},
                "sfx_cues": {"type": "array", "items": {"type": "object", "properties": {
                    "time": {"type": "string"}, "kind": {"type": "string", "enum": ["whoosh", "impact", "pop", "riser"]}},
                    "required": ["time", "kind"]}},
                "framing": {"type": "object", "properties": {
                    "layout": {"type": "string", "enum": ["single_speaker", "multi_speaker", "slides_or_screen", "other"]},
                    "speakers_x": {"type": "array", "items": {"type": "number", "minimum": 0, "maximum": 1}},
                    "speaker_x": {"type": "array", "items": {"type": "object", "properties": {
                        "time": {"type": "string"}, "x": {"type": "number", "minimum": 0, "maximum": 1}},
                        "required": ["time", "x"]}}},
                    "required": ["layout"]},
                "visual_notes": {"type": "string"}},
            "required": ["title", "start", "end", "hook", "opening_words", "closing_words", "score", "music"]}},
    },
    "required": ["clips"],
}


def parse_timecode(value):
    """'MM:SS', 'H:MM:SS' or seconds -> float seconds; None if unparseable."""
    if isinstance(value, (int, float)):
        return float(value)
    parts = str(value).strip().split(":")
    try:
        nums = [float(p) for p in parts]
    except ValueError:
        return None
    if not 1 <= len(nums) <= 3:
        return None
    sec = 0.0
    for n in nums:
        sec = sec * 60 + n
    return sec


def _norm_tokens(text):
    return re.findall(r"[a-z0-9']+", str(text).lower())


def _token_overlap(a, b):
    ta, tb = _norm_tokens(a), set(_norm_tokens(b))
    return sum(1 for t in ta if t in tb) / len(ta) if ta else 0.0


def snap_interval_to_sentences(start, end, transcript, opening_words="", closing_words="", search_pad=12.0):
    """Align a Gemini timestamp pair to real sentence boundaries. The opening/closing words Gemini reports are used to
    re-locate the boundary if its timestamp drifted (Gemini timecodes are ~1s-accurate, sometimes worse in long videos).
    Returns (start, end, quality) or None if the transcript cannot support the interval."""
    sents = transcript.sentences
    if not sents:
        return None
    def best_sentence(t, words, mode):
        near = [i for i, s in enumerate(sents) if s.end >= t - search_pad and s.start <= t + search_pad]
        if not near:
            return None
        def key(i):
            s = sents[i]
            text_score = _token_overlap(words, s.text) if (words and mode == "start") else \
                         (_token_overlap(words, s.text) if words else 0.0)
            dist = abs((s.start if mode == "start" else s.end) - t)
            return (text_score >= 0.6, text_score, -dist)
        return max(near, key=key)
    si, ei = best_sentence(start, opening_words, "start"), best_sentence(end, closing_words, "end")
    if si is None or ei is None or ei < si:
        return None
    s_sent, e_sent = sents[si], sents[ei]
    open_ok = (not opening_words) or _token_overlap(opening_words, " ".join(x.text for x in sents[si:si + 2])) >= 0.5
    close_ok = (not closing_words) or _token_overlap(closing_words, " ".join(x.text for x in sents[max(si, ei - 1):ei + 1])) >= 0.5
    quality = "verified" if (open_ok and close_ok) else "timestamp_only"
    return max(0.0, s_sent.start - 0.05), e_sent.end + 0.10, quality


def trim_to_limit(start, end, transcript, limit):
    """If a proposal exceeds the 90s cap, end it at the last sentence boundary that fits (never mid-sentence)."""
    if end - start <= limit:
        return start, end
    ends = [s.end + 0.10 for s in transcript.sentences if start < s.end + 0.10 <= start + limit]
    return (start, ends[-1]) if ends and ends[-1] - start >= 20.0 else (start, None)


def parse_sfx_cues(raw):
    out = []
    for c in raw or []:
        if isinstance(c, dict) and c.get("kind") in ("whoosh", "impact", "pop", "riser"):
            t = parse_timecode(c.get("time"))
            if t is not None:
                out.append((float(t), c["kind"]))
    return sorted(out)[:6]


def parse_framing_hint(raw):
    """Gemini framing -> {"layout": str, "points": [(source_seconds, x 0..1)]} with invalid samples dropped."""
    if not isinstance(raw, dict):
        return None
    layout = raw.get("layout") if raw.get("layout") in ("single_speaker", "multi_speaker", "slides_or_screen", "other") else "other"
    pts = []
    for p in raw.get("speaker_x") or []:
        t, x = parse_timecode(p.get("time")), p.get("x")
        if t is not None and isinstance(x, (int, float)) and 0.0 <= x <= 1.0:
            pts.append((float(t), float(x)))
    xs = sorted(float(x) for x in (raw.get("speakers_x") or []) if isinstance(x, (int, float)) and not isinstance(x, bool) and 0.0 <= x <= 1.0)
    return {"layout": layout, "points": sorted(pts), "speakers_x": xs if len(xs) == 2 else []}


def director_plans_from_response(data, transcript, source_duration, required, plan_cls=None):
    """Pure function (testable): Gemini JSON -> list of validated ClipPlans + rejection log."""
    plans, rejected, used = [], [], []
    max_len = float(CFG["max_clip_duration"])
    clips = sorted(data.get("clips", []), key=lambda c: -int(c.get("score", 0)))
    for c in clips:
        t0, t1 = parse_timecode(c.get("start")), parse_timecode(c.get("end"))
        title = str(c.get("title", "?"))[:60]
        if t0 is None or t1 is None or t1 <= t0:
            rejected.append((title, "unparseable timestamps")); continue
        if t1 > source_duration + 2.0 or t0 < 0:
            rejected.append((title, "timestamps outside the video")); continue
        snapped = snap_interval_to_sentences(t0, min(t1, source_duration), transcript, c.get("opening_words", ""), c.get("closing_words", ""))
        if snapped is None:
            rejected.append((title, "no transcript support at those times")); continue
        s0, s1, quality = snapped
        s0, s1 = trim_to_limit(s0, s1, transcript, max_len)
        if s1 is None:
            rejected.append((title, "cannot trim to 90s at a sentence boundary")); continue
        if s1 - s0 < float(CFG.get("director_min_clip_sec", 20.0)):
            rejected.append((title, "too short after snapping")); continue
        if any(min(s1, b) - max(s0, a) > 0.25 * (s1 - s0) for a, b in used):
            rejected.append((title, "overlaps a higher-scored clip")); continue
        text = " ".join(s.text for s in transcript.sentences if s.end > s0 and s.start < s1).strip()
        seg = Segment(source_start=round(s0, 2), source_end=round(s1, 2), purpose="gemini_director", transcript=text,
                      role="hook", confidence=1.0 if quality == "verified" else 0.7)
        music = c.get("music") or {}
        plan = ClipPlan(title=title, narrative_type=c.get("narrative_type", "lesson"), segments=[seg],
                        target_duration=max_len,
                        notes={"decided_by": "gemini_director", "boundary_quality": quality, "editorial_score": float(c.get("score", 50)),
                               "topics": [str(t).lower() for t in (c.get("topics") or [])][:4] or ["business"],
                               "hook": c.get("hook", ""), "why_it_works": c.get("why_it_works", ""),
                               "music_brief": {k: music.get(k) for k in ("mood", "energy", "search_terms", "avoid")},
                               "broll_query": c.get("broll_query", ""), "visual_notes": c.get("visual_notes", ""),
                               "gemini_framing": parse_framing_hint(c.get("framing")),
                               "sfx_cues": parse_sfx_cues(c.get("sfx_cues"))})
        try:
            plan.validate(source_duration)
        except ValueError as exc:
            rejected.append((title, str(exc))); continue
        plans.append(plan); used.append((s0, s1))
        if len(plans) >= required:
            break
    return plans, rejected


DIRECTOR_PROMPT = """You are the editorial director for a short-form video studio. Watch the ENTIRE video, then choose the {n} best
self-contained moments for vertical Shorts/Reels/TikTok (each ideally {lo}-{hi} seconds; hard maximum 90).
Prefer moments with a strong first sentence, a complete idea that makes sense without the rest of the video, a payoff or lesson, and
emotional or intellectual surprise. Spread the picks across different parts of the video and different ideas; never overlap.
Each clip must be one CONTINUOUS stretch. Start on the first word of a sentence and end after the last word of a sentence.
Give start/end as MM:SS (or H:MM:SS) exactly as shown in the video player. Quote the first and last ~8 spoken words exactly
(opening_words/closing_words) so a program can verify the timestamps against the transcript.
For music: pick the mood that best supports the clip WITHOUT competing with speech, and 2-5 search words usable on a royalty-free
music library (instrument/genre/feel, no artist or song names). For broll_query give 3-6 concrete visual words, no names or brands.
For sfx_cues: at most 4 sound effects, only where one naturally supports a transition or a key emphasis (whoosh, impact, pop, riser).
Give time as MM:SS in the source video; return an empty list if none is needed; never place one over an important word.
For framing: say whether the main shot is a single_speaker, multi_speaker, slides_or_screen or other. For single_speaker give 2-8
speaker_x samples across the clip (for multi_speaker with exactly two people side by side give speakers_x = their two face centres, left then right): time (MM:SS) and x = horizontal centre of the speaker's face as a fraction of frame width (0=left, 1=right).
Return {n_plus} candidates ranked by score so unusable ones can be dropped. Video title: {title}."""


def gemini_director(source_url, source, transcript):
    """Returns (plans, info). plans may be shorter than required; the caller fills the remainder locally."""
    info = {"used": False, "reason": "", "rejected": []}
    if not (GEMINI.enabled and CFG.get("use_gemini_director", True)):
        info["reason"] = "gemini disabled"; return [], info
    required = int(CFG["required_clip_count"])
    prompt = DIRECTOR_PROMPT.format(n=required, n_plus=required + 4, title=source.title, lo=CFG.get('director_clip_sec', [30, 75])[0], hi=CFG.get('director_clip_sec', [30, 75])[1])
    pin, cache_media, uploaded = None, None, None
    if re.search(r"(youtube\.com|youtu\.be)", str(source_url or "")):
        media = {"type": "video", "uri": source_url}
    elif CFG.get("gemini_upload_local", True) and source.video_path and os.path.isfile(source.video_path):
        stage_set("gemini_video_direction", "running", "gemini", "uploading a small proxy of the local file")
        try:
            proxy = build_gemini_proxy(source)
            uploaded = GEMINI.upload_file(proxy)
        except Exception as exc:
            info["reason"] = f"media upload failed: {gem_scrub(exc)[:100]}"
            stage_set("gemini_video_direction", "completed_using_fallback", "local", info["reason"]); return [], info
        media = {"type": "video", "uri": uploaded["uri"], "mime_type": uploaded["mime_type"]}
        pin, cache_media = uploaded["label"], {"local": source.video_id, "duration": round(source.duration)}
    else:
        info["reason"] = "not a YouTube link and no local file to upload"; return [], info
    if CFG.get("gemini_agentic_video", False):
        media["processing"] = "agentic"
    stage_set("gemini_video_direction", "running", "gemini")
    try:
        res = gemini_structured("director_" + source.video_id, prompt, DIRECTOR_SCHEMA, tier="reasoning", media=media, pin=pin, cache_media=cache_media,
                                validator=lambda d: [] if len(d.get("clips", [])) >= 3 else ["fewer than 3 clips returned"])
    finally:
        if uploaded:
            GEMINI.delete_file(uploaded)
    if not res["ok"]:
        info["reason"] = f"{res['error_kind']}: {str(res['error'])[:120]}"
        stage_set("gemini_video_direction", "completed_using_fallback" if res["error_kind"] != "waiting_for_quota" else "waiting_for_quota",
                  "local", info["reason"])
        return [], info
    plans, rejected = director_plans_from_response(res["data"], transcript, source.duration, required)
    info.update(used=bool(plans), rejected=rejected, model=res["model"], project=res["project"], summary=res["data"].get("video_summary", ""))
    stage_set("gemini_video_direction", "completed" if len(plans) >= required else "completed_using_fallback", "gemini+local",
              f"{len(plans)} accepted, {len(rejected)} rejected by validation")
    return plans, info


def merge_director_with_local(director_plans, local_plans, required):
    """Director picks come first; local plans fill gaps without overlapping them."""
    out = list(director_plans)
    for lp in sorted(local_plans, key=lambda p: -p.notes.get("editorial_score", 0)):
        if len(out) >= required:
            break
        ov = sum(max(0.0, min(a.source_end, b.source_end) - max(a.source_start, b.source_start)) for a in lp.segments for p in out for b in p.segments)
        if ov / max(lp.duration, 1.0) <= 0.30:
            lp.notes.setdefault("decided_by", "local_selector")
            out.append(lp)
    return out[:required]


def build_gemini_proxy(source):
    """Small low-fps proxy for upload (keeps the timeline identical, so timestamps map 1:1 to the original)."""
    proxy = os.path.join(os.path.dirname(source.video_path), "gemini_proxy.mp4")
    if not os.path.isfile(proxy):
        run(["ffmpeg", "-y", "-v", "error", "-i", source.video_path, "-vf", "scale=-2:360,fps=4", "-c:v", "libx264", "-preset", "veryfast",
             "-crf", "34", "-c:a", "aac", "-b:a", "48k", "-ac", "1", "-ar", "16000", "-movflags", "+faststart", proxy], quiet=True)
    return proxy


# ---- Gemini ranks licensed search results (it only reorders; licence/decode verification still happens afterwards)
RERANK_CONTEXT = {"music": "", "broll": ""}
RERANK_SCHEMA = {"type": "object", "properties": {"ranking": {"type": "array", "items": {"type": "integer"}}}, "required": ["ranking"]}


def _describe_asset(item):
    if not isinstance(item, dict):
        return str(item)
    bits = []
    for k in ("title", "name", "tags", "subject", "description"):
        v = item.get(k)
        if v:
            bits.append(" ".join(map(str, v)) if isinstance(v, list) else str(v))
    return " | ".join(bits) or "untitled"


def rerank_candidates(kind, items, describe=_describe_asset, top_n=15):
    ctx = RERANK_CONTEXT.get(kind) or ""
    pool = globals().get("GEMINI")
    if not ctx or not items or not CFG.get("gemini_rerank", True) or pool is None or not getattr(pool, "enabled", False):
        return items
    head, tail = list(items[:top_n]), list(items[top_n:])
    if len(head) < 2:
        return items
    lines = "\n".join(f"{i}: {describe(x)[:140]}" for i, x in enumerate(head))
    what = "background music track" if kind == "music" else "stock video clip"
    prompt = (f"Rank these {what} candidates from best to worst fit for the brief below. Judge only from the text given. "
              f"Return JSON {{\"ranking\": [ids best-first]}} using each id at most once.\n\nBRIEF: {ctx[:700]}\n\nCANDIDATES:\n{lines}")
    res = gemini_structured(f"rerank_{kind}", prompt, RERANK_SCHEMA, tier="fast",
                            validator=lambda d: [] if all(isinstance(i, int) and 0 <= i < len(head) for i in d["ranking"]) else ["ranking ids out of range"])
    if not res["ok"]:
        return items
    order = []
    for i in res["data"]["ranking"]:
        if i not in order:
            order.append(i)
    order += [i for i in range(len(head)) if i not in order]
    return [head[i] for i in order] + tail


DIRECTOR_INFO = {"used": False, "reason": "not run"}


In [ ]:
# CELL 6c — Director tests (synthetic transcript + fake Gemini; no network, no key)
def run_director_tests():
    import types, tempfile
    global GEMINI, GEMINI_CACHE_DIR
    saved, saved_cache = globals().get("GEMINI"), GEMINI_CACHE_DIR
    GEMINI_CACHE_DIR = tempfile.mkdtemp()
    # 40 sentences x 5s = 200s source. Sentence i says "sentence number i is spoken here now".
    sents = [Sentence(text=f"Sentence number {i} is spoken here now.", start=i * 5.0, end=i * 5.0 + 4.6) for i in range(40)]
    tr = Transcript(language="en", words=[], sentences=sents)
    SRC = Source(video_id="vid", title="T", duration=200.0, video_path="x")
    def clip(t0, t1, o, c, score=80, mood="uplifting"):
        return {"title": f"clip {t0}", "start": t0, "end": t1, "hook": "h", "why_it_works": "w", "opening_words": o,
                "closing_words": c, "score": score, "topics": ["Money"],
                "music": {"mood": mood, "energy": "medium", "search_terms": "warm piano"}, "broll_query": "city skyline"}
    try:
        assert parse_timecode("01:30") == 90.0 and parse_timecode("1:02:03") == 3723.0 and parse_timecode("abc") is None
        # exact timestamps -> sentence-aligned + verified
        data = {"clips": [clip("00:20", "00:55", "Sentence number 4 is spoken", "Sentence number 10 is spoken here now")]}
        plans, rej = director_plans_from_response(data, tr, 200.0, 5)
        assert len(plans) == 1 and plans[0].notes["boundary_quality"] == "verified"
        assert abs(plans[0].segments[0].source_start - 19.95) < 0.2 and plans[0].notes["music_brief"]["mood"] == "uplifting"
        # drifted timestamp (3s early/late) is re-located using the quoted words
        data = {"clips": [clip("00:17", "00:58", "Sentence number 4 is spoken", "Sentence number 10 is spoken here now")]}
        plans, _ = director_plans_from_response(data, tr, 200.0, 5)
        assert plans and abs(plans[0].segments[0].source_start - 19.95) < 0.2
        # wrong quote -> only timestamp trusted, flagged with lower confidence
        data = {"clips": [clip("00:20", "00:55", "totally different words here", "nothing like it at all")]}
        plans, _ = director_plans_from_response(data, tr, 200.0, 5)
        assert plans[0].notes["boundary_quality"] == "timestamp_only" and plans[0].segments[0].confidence < 1
        # rejects: outside video, reversed, too short, overlap, unparseable; over-length trimmed at sentence boundary
        data = {"clips": [clip("03:00", "04:30", "a", "b", 90), clip("00:50", "00:40", "a", "b", 89), clip("00:10", "00:15", "a", "b", 88),
                          clip("x", "y", "a", "b", 87), clip("00:00", "02:20", "Sentence number 0", "Sentence number 28", 86),
                          clip("00:10", "00:50", "Sentence number 2", "Sentence number 9", 85)]}
        plans, rej = director_plans_from_response(data, tr, 200.0, 5)
        assert len(plans) == 1 and plans[0].duration <= 90.0 and len(rej) == 5, (len(plans), rej)
        for p in plans: p.validate(200.0)
        # merge: director first, local fills gaps but never overlaps
        lp1 = ClipPlan("L1", "x", [Segment(100, 140, "p", "t")], notes={"editorial_score": 50})
        lp2 = ClipPlan("L2", "x", [Segment(plans[0].segments[0].source_start + 1, plans[0].segments[0].source_start + 30, "p", "t")], notes={"editorial_score": 99})
        merged = merge_director_with_local(plans, [lp2, lp1], 3)
        assert [m.title for m in merged][1:] == ["L1"] and merged[0].notes["decided_by"] == "gemini_director"

        # end-to-end with a fake pool: YouTube URL is sent as a video part; non-YouTube / disabled -> local fallback
        seen = {}
        class FakePool:
            enabled = True; disabled_reason = ""
            def generate(self, models, inp, rf, system, cfg, task):
                seen["input"] = inp
                body = {"video_summary": "s", "clips": [clip(f"{m:02d}:{s:02d}", f"{m:02d}:{s + 40:02d}", f"Sentence number {(m*60+s)//5} is", f"Sentence number {(m*60+s+35)//5} is spoken here now", 90 - m)
                                                      for m, s in ((0, 10), (0, 70), (2, 10), (2, 20), (0, 130))]}
                return GeminiResult(json.dumps(body), models[0], "P1", "key1", 1)
        GEMINI = FakePool()
        url = "https://www.youtube.com/watch?v=abc123"
        old_sets = CFG.get("use_gemini_director", True)
        CFG["use_gemini_director"] = True
        plans, info = gemini_director(url, SRC, tr)
        assert info["used"] and isinstance(seen["input"], list) and seen["input"][0] == {"type": "video", "uri": url}, seen
        assert STAGES["gemini_video_direction"]["status"] in ("completed", "completed_using_fallback")
        assert parse_sfx_cues([{"time": "00:12", "kind": "whoosh"}, {"time": "x", "kind": "pop"}, {"time": "00:20", "kind": "bogus"}]) == [(12.0, "whoosh")]
        plans2, info2 = gemini_director("/content/local.mp4", SRC, tr)
        assert plans2 == [] and "not a YouTube link" in info2["reason"]
        # local-file path: proxy is uploaded, call is pinned to the uploading key, file deleted afterwards
        events = {}
        class UpPool(FakePool):
            def generate(self, models, inp, rf, system, cfg, task, pin=None):
                events["pin"], events["inp"] = pin, inp; return FakePool.generate(self, models, inp, rf, system, cfg, task)
            def upload_file(self, path): events["uploaded"] = path; return {"label": "key2", "name": "files/z", "uri": "https://x/files/z", "mime_type": "video/mp4"}
            def delete_file(self, info): events["deleted"] = info["name"]
        real_proxy = globals()["build_gemini_proxy"]
        globals()["build_gemini_proxy"] = lambda s: "/tmp/proxy.mp4"
        GEMINI = UpPool()
        LSRC = Source(video_id="localvid", title="L", duration=200.0, video_path=__file__ if "__file__" in globals() else "/etc/hostname")
        if not os.path.isfile(LSRC.video_path): LSRC.video_path = "/etc/hostname"
        plans4, info4 = gemini_director("/content/local.mp4", LSRC, tr)
        globals()["build_gemini_proxy"] = real_proxy
        assert info4["used"] and events["pin"] == "key2" and events["deleted"] == "files/z" and events["inp"][0]["uri"] == "https://x/files/z", events
        # rerank: reorders by Gemini ranking, ignores bad answers, no call without a brief
        calls_n = {"n": 0}
        class RankPool:
            enabled = True; disabled_reason = ""
            def __init__(self, answer): self.answer = answer
            def generate(self, models, inp, rf, system, cfg, task):
                calls_n["n"] += 1; return GeminiResult(json.dumps(self.answer), models[0], "P", "k", 1)
        items = [{"title": f"t{i}"} for i in range(4)]
        RERANK_CONTEXT["music"] = "warm piano for a calm lesson"
        GEMINI = RankPool({"ranking": [2, 0, 1]})
        assert [x["title"] for x in rerank_candidates("music", items)] == ["t2", "t0", "t1", "t3"]
        RERANK_CONTEXT["music"] = "a different brief so the cache cannot answer"
        GEMINI = RankPool({"ranking": [9, 9]})
        assert rerank_candidates("music", items, top_n=4) == items
        RERANK_CONTEXT["music"] = ""
        n0 = calls_n["n"]; assert rerank_candidates("music", items) == items and calls_n["n"] == n0
        GEMINI = types.SimpleNamespace(enabled=False, disabled_reason="off")
        plans3, info3 = gemini_director(url, SRC, tr)
        assert plans3 == [] and info3["reason"] == "gemini disabled"
        CFG["use_gemini_director"] = old_sets
        STAGES.pop("gemini_video_direction", None)
        # framing hints (Gemini) -> framing dicts
        h = parse_framing_hint({"layout": "single_speaker", "speaker_x": [{"time": "00:10", "x": 0.3}, {"time": "00:14", "x": 0.35}, {"time": "00:20", "x": 7}, {"time": "bad", "x": 0.5}]})
        assert h["layout"] == "single_speaker" and h["points"] == [(10.0, 0.3), (14.0, 0.35)]
        return True
    finally:
        GEMINI, GEMINI_CACHE_DIR = saved, saved_cache


if CFG.get("run_self_tests"):
    DIRECTOR_TESTS_PASSED = run_director_tests()
    print("DIRECTOR_TESTS_PASSED =", DIRECTOR_TESTS_PASSED)
else:
    DIRECTOR_TESTS_PASSED = None
    print("Self-tests skipped (this is normal). Nothing to do in this cell.")


In [ ]:
# CELL 7 — Editorial window scoring, cross-timestamp story reconstruction, and five-clip diversity
from sklearn.metrics.pairwise import cosine_similarity


def editorial_features(text, confidence, duration, words_per_second):
    low = (text or "").lower()
    tok = set(tokens(low))
    hook = min(10.0, 2.0 * len(tok & HOOK_CUES)) + (3.0 if "?" in text else 0.0)
    specificity = min(10.0, 3.5 * bool(re.search(r"\b\d+(?:\.\d+)?\b", text)) + 2.5 * bool(re.search(r"\b(first|second|third|percent|million|billion|years|months|days)\b", low)))
    story = min(10.0, 2.0 * len(tok & {x.lower() for x in STORY_CUES}))
    payoff = min(10.0, 2.5 * sum(phrase in low for phrase in ("i learned", "the lesson", "that's why", "in the end", "the result", "which means")))
    action = min(10.0, 2.0 * len(tok & ACTION_CUES))
    density = max(0.0, 8.0 - abs(words_per_second - 2.6) * 3.0)
    confidence_score = max(0.0, min(10.0, confidence * 10.0))
    clarity = 6.0 if len(text.split()) >= 15 and len(text.split()) <= 90 else 3.0
    return {
        "hook": hook, "specificity": specificity, "story": story, "payoff": payoff,
        "actionability": action, "speech_density": density,
        "transcription_confidence": confidence_score, "standalone_clarity": clarity,
    }


def make_editorial_windows(transcript, content_map):
    windows = []
    n = len(transcript.sentences)
    starts = [w.start for w in transcript.words]
    for i in range(0, n, 2):
        best = None
        for j in range(i + 1, min(n, i + 8) + 1):
            start = transcript.sentences[i].start
            end = transcript.sentences[j - 1].end
            duration = end - start
            if duration > 27.0:
                break
            if duration < 8.0:
                continue
            text = " ".join(s.text for s in transcript.sentences[i:j])
            lo = bisect.bisect_left(starts, start - 0.01)
            hi = bisect.bisect_right(starts, end + 0.01)
            ws = transcript.words[lo:hi]
            if len(ws) < 12:
                continue
            avg_conf = float(np.mean([w.prob for w in ws]))
            wps = len(ws) / max(duration, 0.1)
            feats = editorial_features(text, avg_conf, duration, wps)
            topics, emotions, role = classify_text(text)
            # Explainable editorial heuristic; intentionally not labelled a virality prediction.
            score = (2.2 * feats["hook"] + 1.2 * feats["specificity"] + 1.1 * feats["story"] +
                     1.25 * feats["payoff"] + 1.1 * feats["actionability"] +
                     0.7 * feats["speech_density"] + 0.8 * feats["transcription_confidence"] +
                     0.8 * feats["standalone_clarity"])
            row = {
                "window_id": len(windows), "sentence_start_index": i,
                "sentence_end_index": j - 1, "source_start": start,
                "source_end": end, "duration": duration, "text": text,
                "topics": topics, "emotions": emotions, "role": role,
                "confidence": avg_conf, "word_count": len(ws),
                "words_per_second": wps, "features": feats, "score": score,
            }
            if best is None or abs(duration - 17.0) < abs(best["duration"] - 17.0):
                best = row
        if best is not None:
            windows.append(best)
    # Keep the strongest editorial windows while retaining whole-source time coverage.
    if len(windows) > 900:
        windows = sorted(windows, key=lambda x: x["score"], reverse=True)[:900]
        windows.sort(key=lambda x: x["source_start"])
    for idx, row in enumerate(windows):
        row["window_id"] = idx
    return windows


def window_vector_matrix(windows, sentence_vectors):
    rows = []
    for win in windows:
        s0, s1 = win["sentence_start_index"], win["sentence_end_index"] + 1
        if hasattr(sentence_vectors, "tocsr"):
            v = sentence_vectors[s0:s1].mean(axis=0)
            rows.append(np.asarray(v).ravel())
        else:
            v = np.asarray(sentence_vectors[s0:s1]).mean(axis=0)
            norm = np.linalg.norm(v)
            rows.append(v / norm if norm else v)
    if hasattr(sentence_vectors, "tocsr"):
        from scipy.sparse import csr_matrix
        return csr_matrix(np.vstack(rows))
    return np.vstack(rows).astype(np.float32)


def similarity_matrix(windows, vectors):
    win_vectors = window_vector_matrix(windows, vectors)
    return cosine_similarity(win_vectors).astype(np.float32)


def headline_from_source_text(text, max_words=8):
    # Extractive title: it is assembled from the speaker's transcript, not invented dialogue.
    first_sentence = re.split(r"(?<=[.!?])\s+", text.strip())[0]
    words = first_sentence.split()
    headline = " ".join(words[:max_words]).strip(" ,:;—-")
    return headline or "A useful idea worth remembering"


def overlap_ratio(a, b):
    intersection = max(0.0, min(a["source_end"], b["source_end"]) - max(a["source_start"], b["source_start"]))
    union = max(a["source_end"], b["source_end"]) - min(a["source_start"], b["source_start"])
    return intersection / union if union > 0 else 0.0


def candidate_story(anchor_idx, windows, sim, source):
    anchor = windows[anchor_idx]
    support_rows = []
    for j, other in enumerate(windows):
        if j == anchor_idx or overlap_ratio(anchor, other) > 0.05:
            continue
        similarity = float(sim[anchor_idx, j])
        if similarity < (0.10 if SIMILARITY_BACKEND == "tfidf_fallback" else 0.20):
            continue
        # Complementary beats improve narrative progression; this remains an explainable heuristic.
        role_bonus = 0.0
        if other["role"] != anchor["role"]:
            role_bonus += 0.12
        if other["role"] in ("explanation", "story", "lesson", "payoff"):
            role_bonus += 0.10
        topic_bonus = 0.12 if set(anchor["topics"]) & set(other["topics"]) else 0.0
        quality = min(1.0, other["score"] / 95.0)
        # Prefer passages related to the anchor and not simply adjacent repeats.
        rank_score = 0.58 * similarity + role_bonus + topic_bonus + 0.12 * quality
        support_rows.append((rank_score, j, similarity))
    support_rows.sort(reverse=True)

    chosen = [anchor]
    total = anchor["duration"]
    for rank_score, j, similarity in support_rows:
        other = windows[j]
        if any(overlap_ratio(other, x) > 0.05 for x in chosen):
            continue
        # A coherent short can be under 90 seconds; no filler is added to hit a target.
        if total + other["duration"] > CFG["max_clip_duration"]:
            continue
        chosen.append(other)
        total += other["duration"]
        if total >= CFG["preferred_min_duration"] or len(chosen) >= 4:
            break

    # If semantic support is sparse, extend the anchor with its immediately adjacent context
    # without crossing the source boundary. This remains one real continuous source interval.
    if len(chosen) == 1 and total < 42.0:
        idx0, idx1 = anchor["sentence_start_index"], anchor["sentence_end_index"]
        sentences = TRANSCRIPT.sentences
        start, end = anchor["source_start"], anchor["source_end"]
        left, right = idx0, idx1
        while end - start < min(65.0, CFG["max_clip_duration"]):
            left_dur = sentences[left - 1].end - start if left > 0 else -1
            right_dur = sentences[right + 1].end - start if right + 1 < len(sentences) else 999
            if right_dur <= CFG["max_clip_duration"] and (left_dur < 0 or abs(62 - right_dur) <= abs(62 - left_dur)):
                right += 1; end = sentences[right].end
            elif left_dur >= 0 and left_dur <= CFG["max_clip_duration"]:
                left -= 1; start = sentences[left].start
            else:
                break
        if end > start and end - start <= CFG["max_clip_duration"]:
            text = " ".join(s.text for s in sentences[left:right+1])
            chosen = [{**anchor, "source_start": start, "source_end": end,
                       "duration": end-start, "text": text, "role": classify_text(text)[2]}]
            total = end - start

    # Hook is first; supporting material is then arranged in source-time order.
    support = sorted(chosen[1:], key=lambda x: x["source_start"])
    sequence = [chosen[0]] + support
    segments = []
    for idx, item in enumerate(sequence):
        role = "hook" if idx == 0 else item["role"]
        segments.append(Segment(
            source_start=round(item["source_start"], 3),
            source_end=round(item["source_end"], 3),
            purpose=role,
            transcript=item["text"],
            role=role,
            confidence=float(item["confidence"]),
        ))
    if sum(s.duration for s in segments) > CFG["max_clip_duration"] + 0.02:
        return None
    sem_sim = [x[2] for x in support_rows if any(x[1] == windows.index(c) for c in support)]
    mean_sim = float(np.mean(sem_sim)) if sem_sim else 0.0
    diversity = len({topic for item in sequence for topic in item["topics"]})
    avg_conf = float(np.mean([s.confidence for s in segments]))
    hook_score = anchor["features"]["hook"]
    payoff_score = max([x["features"]["payoff"] for x in sequence] or [0.0])
    story_score = min(100.0, 0.42 * anchor["score"] + 22.0 * mean_sim + 2.5 * diversity + 0.8 * payoff_score + 5.0 * avg_conf)
    title = headline_from_source_text(anchor["text"])
    plan = ClipPlan(
        title=title,
        narrative_type=anchor["role"],
        segments=segments,
        target_duration=CFG["max_clip_duration"],
        notes={
            "editorial_score": round(story_score, 2),
            "score_type": "explainable editorial heuristic; not a virality or retention prediction",
            "hook_score": round(hook_score, 2),
            "semantic_similarity_mean": round(mean_sim, 3),
            "similarity_backend": SIMILARITY_BACKEND,
            "topics": sorted({t for x in sequence for t in x["topics"]}),
            "emotions": sorted({e for x in sequence for e in x["emotions"]}),
            "source_intervals": [[s.source_start, s.source_end] for s in segments],
            "non_chronological_reorder": any(segments[i].source_start < segments[i-1].source_start for i in range(1, len(segments))),
            "context_review_required": True,
            "source_dialogue_only": True,
            "retention_quality_map": [
                {"output_start": round(sum(x.duration for x in segments[:k]), 2),
                 "output_end": round(sum(x.duration for x in segments[:k+1]), 2),
                 "editorial_role": x.purpose,
                 "attention_risk": "review for slow explanation" if x.purpose == "explanation" else "review for missing setup" if x.purpose == "hook" else "review for payoff strength" if x.purpose == "payoff" else "standard review",
                 "score_type": "editorial review heuristic, not measured retention"}
                for k, x in enumerate(segments)
            ],
        },
    )
    plan.validate(source.duration)
    return plan


def select_five_distinct_stories(windows, sim, source):
    """Strict pass first; if fewer than five stories qualify, relax the gates step by step (never below a floor) and
    label every plan that needed relaxing so you can review it. Distinct-source-time protection always stays on."""
    required = int(CFG["required_clip_count"])
    base = float(CFG["min_story_score"])
    ladder = [  # (min story score, min anchor score, max source overlap, enforce same-topic rule)
        (base, 18, 0.30, True),
        (base * 0.85, 16, 0.40, True),
        (base * 0.70, 14, 0.50, False),
        (base * 0.55, 10, 0.60, False),
    ]
    ranked_anchors = sorted(range(len(windows)), key=lambda i: windows[i]["score"], reverse=True)
    plan_cache, selected, all_candidates = {}, [], []

    def overlap_with(plan, prior):
        total = 0.0
        for a in plan.segments:
            for b in prior.segments:
                total += max(0.0, min(a.source_end, b.source_end) - max(a.source_start, b.source_start))
        return total / max(plan.duration, 1.0)

    for level, (min_score, anchor_floor, max_overlap, topic_rule) in enumerate(ladder):
        candidates, seen = [], set()
        for idx in ranked_anchors:
            if windows[idx]["score"] < anchor_floor:
                continue
            if idx not in plan_cache:
                plan_cache[idx] = candidate_story(idx, windows, sim, source)
            plan = plan_cache[idx]
            if plan is None or plan.notes["editorial_score"] < min_score:
                continue
            signature = tuple((round(s.source_start / 5), round(s.source_end / 5)) for s in plan.segments)
            if signature in seen:
                continue
            seen.add(signature)
            candidates.append(plan)
            if len(candidates) >= 120:
                break
        all_candidates = candidates or all_candidates
        for plan in sorted(candidates, key=lambda p: p.notes["editorial_score"], reverse=True):
            if len(selected) >= required:
                break
            if any(plan is p for p in selected):
                continue
            too_close = False
            for prior in selected:
                if overlap_with(plan, prior) > max_overlap:
                    too_close = True
                    break
                pa, pb = set(plan.notes.get("topics", [])), set(prior.notes.get("topics", []))
                if topic_rule and pa and pb and len(pa & pb) / max(1, len(pa | pb)) > 0.80 and plan.narrative_type == prior.narrative_type:
                    too_close = True
                    break
            if not too_close:
                if level > 0:
                    plan.notes["relaxed_selection_level"] = level
                    plan.notes["review_note"] = "chosen after relaxing the selection gate; check it before posting"
                selected.append(plan)
        if len(selected) >= required:
            break
        log("editorial", f"only {len(selected)}/{required} stories at selection level {level + 1}; relaxing the gate")
    # Last resort: stories built ONLY from source time that no chosen story uses (so they are genuinely different).
    while len(selected) < required:
        covered = [(sg.source_start, sg.source_end) for p in selected for sg in p.segments]

        def is_free(w):
            ov = sum(max(0.0, min(w["source_end"], e) - max(w["source_start"], s)) for s, e in covered)
            return ov / max(w["duration"], 1e-6) <= 0.05
        free_idx = [i for i, w in enumerate(windows) if is_free(w)]
        if len(free_idx) < 2:
            log("editorial", f"no unused source material left ({len(free_idx)} free windows of {len(windows)})")
            break
        sub = [windows[i] for i in free_idx]
        sim_sub = sim[np.ix_(free_idx, free_idx)]
        best = None
        for k in sorted(range(len(sub)), key=lambda k: sub[k]["score"], reverse=True)[:80]:
            plan = candidate_story(k, sub, sim_sub, source)
            if plan is None or plan.duration < 25.0:
                continue
            if any(overlap_with(plan, prior) > 0.20 for prior in selected):
                continue
            if best is None or plan.notes["editorial_score"] > best.notes["editorial_score"]:
                best = plan
        if best is None:
            log("editorial", "unused source material is too thin to build another distinct story")
            break
        best.notes["relaxed_selection_level"] = 4
        best.notes["review_note"] = "built from leftover source material; check it before posting"
        selected.append(best)
    if len(selected) < required:
        log("editorial", f"DIAGNOSTIC: source {float(source.duration):.0f}s, {len(windows)} windows, {len(plan_cache)} anchors tried, "
                         f"{sum(1 for v in plan_cache.values() if v is not None)} produced a story, {len(selected)} selected")
    return selected, all_candidates


def plans_checkpoint_path():
    return os.path.join(SOURCE_DIR, "plans_checkpoint.json")


def plans_checkpoint_key():
    """Identity of everything that determines the plans; a mismatch invalidates the checkpoint."""
    return {"schema": 1, "source_id": SOURCE.video_id, "source_duration": round(float(SOURCE.duration), 2),
            "words": len(TRANSCRIPT.words), "sentences": len(TRANSCRIPT.sentences),
            "similarity_backend": SIMILARITY_BACKEND, "min_story_score": CFG["min_story_score"],
            "max_clip_duration": CFG["max_clip_duration"], "required_clip_count": CFG["required_clip_count"],
            "director": bool(GEMINI.enabled and CFG.get("use_gemini_director", True))}


def save_plans_checkpoint(plans):
    atomic_write_json(plans_checkpoint_path(), {"key": plans_checkpoint_key(), "plans": [p.to_dict() for p in plans]})


def load_plans_checkpoint():
    """Return validated ClipPlans saved for exactly this source/transcript/config, else None."""
    path = plans_checkpoint_path()
    if not os.path.isfile(path):
        return None
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        if data.get("key") != plans_checkpoint_key():
            return None
        plans = [ClipPlan(title=p["title"], narrative_type=p["narrative_type"],
                          segments=[Segment(**s) for s in p["segments"]],
                          target_duration=p.get("target_duration", 90.0), notes=p.get("notes", {}))
                 for p in data["plans"]]
        for plan in plans:
            plan.validate(SOURCE.duration)
        return plans
    except Exception as exc:
        log("editorial", f"plans checkpoint ignored: {type(exc).__name__}: {exc}")
        return None


PLANS = load_plans_checkpoint() if CFG.get("resume_plans", True) else None
if PLANS and len(PLANS) == CFG["required_clip_count"]:
    STORY_CANDIDATES = []
    log("editorial", "resumed five story plans from checkpoint (same source, transcript and config)")
else:
    PLANS = None
    # BRAIN: Gemini watches the video link and proposes clips; BODY: this notebook validates and executes them.
    _director_plans, DIRECTOR_INFO = gemini_director(globals().get("USER_INPUT", ""), SOURCE, TRANSCRIPT)
    log("director", f"Gemini director: {len(_director_plans)} validated clip(s)" if _director_plans else f"local selector in charge ({DIRECTOR_INFO['reason']})")
    for _t, _why in DIRECTOR_INFO.get("rejected", []):
        log("director", f"rejected '{_t}': {_why}")
    EDITORIAL_WINDOWS = make_editorial_windows(TRANSCRIPT, CONTENT_MAP)
    if len(EDITORIAL_WINDOWS) < CFG["required_clip_count"] and len(_director_plans) < CFG["required_clip_count"]:
        raise RuntimeError(f"Only {len(EDITORIAL_WINDOWS)} usable editorial windows were found; cannot responsibly produce five distinct clips")
    log("editorial", f"scored {len(EDITORIAL_WINDOWS)} whole-source candidate windows")
    if len(_director_plans) >= CFG["required_clip_count"]:
        PLANS, STORY_CANDIDATES = _director_plans[:CFG["required_clip_count"]], []
    else:
        _local_plans, STORY_CANDIDATES = [], []
        if len(EDITORIAL_WINDOWS) >= 1:
            SIM_MATRIX = similarity_matrix(EDITORIAL_WINDOWS, SENTENCE_VECTORS)
            _local_plans, STORY_CANDIDATES = select_five_distinct_stories(EDITORIAL_WINDOWS, SIM_MATRIX, SOURCE)
        PLANS = merge_director_with_local(_director_plans, _local_plans, CFG["required_clip_count"])
print("\nSELECTED STORY BLUEPRINTS")
for i, plan in enumerate(PLANS, 1):
    print(f"{i}. {plan.title[:70]} | {plan.duration:.1f}s | {len(plan.segments)} source interval(s) | score={plan.notes['editorial_score']} | by {plan.notes.get('decided_by', 'local_selector')}")
if len(PLANS) != CFG["required_clip_count"]:
    if CFG.get("allow_fewer_clips", True) and len(PLANS) >= int(CFG.get("min_clip_count", 3)):
        log("editorial", f"only {len(PLANS)} distinct stories qualified; continuing with {len(PLANS)} clips instead of manufacturing weak ones")
        CFG["required_clip_count"] = len(PLANS)
    else:
        raise RuntimeError(f"Only {len(PLANS)} distinct stories cleared the editorial selection gate. No final Drive upload will occur. This is preferable to manufacturing weak clips.")
atomic_write_json(os.path.join(REPORT_DIR, "editing_blueprints.json"), [p.to_dict() for p in PLANS])
save_plans_checkpoint(PLANS)
save_state("five_story_plans_ready", count=len(PLANS), candidate_count=len(STORY_CANDIDATES))


In [ ]:
# CELL 8 — Multi-provider licensed music + B-roll acquisition (surgical V3 patch)
# Providers: Coverr API (free tier), Internet Archive CC search, Pixabay API, and verified manual pools.
# Final fallbacks: original procedural music and the source footage. No licence gate is bypassed.

import requests
from urllib.parse import quote
from scipy.io import wavfile

# Do NOT include Public Domain Mark here: it is a rights-status label, not a copyright licence.
ARCHIVE_LICENSE_ALLOWLIST = {
    "https://creativecommons.org/publicdomain/zero/1.0/": "CC0 1.0",
    "https://creativecommons.org/licenses/by/4.0/": "CC BY 4.0",
    "https://creativecommons.org/licenses/by/3.0/": "CC BY 3.0",
}
LICENSE_ALLOWLIST = dict(ARCHIVE_LICENSE_ALLOWLIST)
COVERR_LICENSE_URL = "https://coverr.co/license/"
PIXABAY_LICENSE_URL = "https://pixabay.com/service/license-summary/"
MIXKIT_MUSIC_LICENSE_URL = "https://mixkit.co/license/modal/musicFree/"
MIXKIT_VIDEO_LICENSE_URL = "https://mixkit.co/license/modal/videoFree/"

# The persistent cache prevents repeated API calls and preserves downloaded media between runs.
# Pixabay explicitly requires API responses to be cached for at least 24 hours.
ASSET_CACHE_DIR = os.path.join(DRIVE_ROOT, "_ASSET_CACHE") if DRIVE_AVAILABLE else os.path.join(CACHE_ROOT, "_ASSET_CACHE")
os.makedirs(ASSET_CACHE_DIR, exist_ok=True)
MUSIC_POOL_DIR = os.path.join(DRIVE_ROOT if DRIVE_AVAILABLE else CFG["work_dir"], "music_pool")
BROLL_POOL_DIR = os.path.join(DRIVE_ROOT if DRIVE_AVAILABLE else CFG["work_dir"], "broll_pool")
os.makedirs(MUSIC_POOL_DIR, exist_ok=True)
os.makedirs(BROLL_POOL_DIR, exist_ok=True)

ASSET_MANIFEST = {"generated_utc": datetime.now(timezone.utc).isoformat(), "music": [], "broll": []}
USED_MUSIC_IDS = set()
USED_BROLL_IDS = set()
BROLL_DOWNLOADS_THIS_RUN = 0


def _secret_value(name):
    """Read a Colab Secret first, then an environment variable; never print API keys."""
    value = os.environ.get(name, "")
    if value:
        return str(value).strip()
    try:
        from google.colab import userdata
        return str(userdata.get(name) or "").strip()
    except Exception:
        return ""


def _normalise_license_url(value):
    value = str(value or "").strip().split("#", 1)[0].split("?", 1)[0].rstrip("/")
    if value.startswith("http://creativecommons.org/"):
        value = "https://" + value[len("http://"):]
    return (value + "/") if value else ""


def _safe_pool_file(pool_dir, filename):
    """Resolve a manifest filename and reject absolute paths or traversal outside the pool."""
    try:
        root = Path(pool_dir).resolve()
        raw = Path(str(filename or ""))
        if raw.is_absolute():
            return None
        candidate = (root / raw).resolve()
        if candidate == root or root not in candidate.parents:
            return None
        return str(candidate) if candidate.is_file() else None
    except Exception:
        return None


def _stream_download(url, local_path, max_bytes=180 * 1024 * 1024, headers=None, timeout=(15, 90)):
    """Download atomically; never leave a partial file that can be mistaken for an asset."""
    if not url or not str(url).startswith("https://"):
        raise ValueError("Asset URL must be HTTPS")
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if os.path.isfile(local_path) and os.path.getsize(local_path) > 20_000:
        return local_path
    part_path = local_path + ".part"
    try:
        with requests.get(url, stream=True, timeout=timeout, headers=headers or {}) as response:
            response.raise_for_status()
            declared = int(response.headers.get("Content-Length") or 0)
            if declared and declared > max_bytes:
                raise ValueError(f"Asset exceeds download cap ({declared} bytes)")
            total = 0
            with open(part_path, "wb") as out:
                for block in response.iter_content(1024 * 256):
                    if not block:
                        continue
                    total += len(block)
                    if total > max_bytes:
                        raise ValueError("Asset exceeded download cap while streaming")
                    out.write(block)
        if total < 20_000:
            raise ValueError(f"Downloaded asset is suspiciously small ({total} bytes)")
        os.replace(part_path, local_path)
        return local_path
    except Exception:
        try:
            if os.path.exists(part_path):
                os.remove(part_path)
        except OSError:
            pass
        raise


def _probe_audio_asset(path, min_duration=8.0):
    try:
        info = ffprobe_json(path)
        streams = info.get("streams", [])
        has_audio = any(s.get("codec_type") == "audio" for s in streams)
        duration = float(info.get("format", {}).get("duration", 0) or 0)
        return bool(has_audio and duration >= min_duration), duration
    except Exception:
        return False, 0.0


def _probe_video_asset(path, min_duration=2.0):
    try:
        info = ffprobe_json(path)
        streams = info.get("streams", [])
        has_video = any(s.get("codec_type") == "video" for s in streams)
        duration = float(info.get("format", {}).get("duration", 0) or 0)
        return bool(has_video and duration >= min_duration), duration
    except Exception:
        return False, 0.0


def _read_fresh_json(path, ttl_seconds):
    try:
        with open(path, encoding="utf-8") as f:
            row = json.load(f)
        if time.time() - float(row.get("cached_at", 0)) <= ttl_seconds:
            return row
    except Exception:
        pass
    return None


def _write_json_cache(path, payload):
    atomic_write_json(path, {"cached_at": time.time(), **payload})


MUSIC_QUERY_OVERRIDE = ""      # set per clip from Gemini's music brief; empty = built-in mood terms
_BRIEF_MOODS = ("reflective", "uplifting", "playful", "curious", "documentary")


def _music_query_tag():
    return hashlib.sha1(MUSIC_QUERY_OVERRIDE.encode()).hexdigest()[:6] if MUSIC_QUERY_OVERRIDE else "std"


def plan_mood(plan):
    _brief_mood = (plan.notes.get("music_brief") or {}).get("mood")
    if _brief_mood in _BRIEF_MOODS:
        return _brief_mood
    emotions = set(plan.notes.get("emotions", []))
    text = " ".join(s.transcript for s in plan.segments).lower()
    if "tension" in emotions or any(x in text for x in ("fear", "risk", "danger", "crisis")):
        return "reflective"
    if "hope" in emotions or any(x in text for x in ("believe", "future", "opportunity", "dream")):
        return "uplifting"
    if "amusement" in emotions or any(x in text for x in ("funny", "joke", "laugh")):
        return "playful"
    if "surprise" in emotions:
        return "curious"
    return "documentary"


def _coverr_headers():
    key = str(CFG.get("coverr_api_key") or _secret_value("COVERR_API_KEY")).strip()
    return {"x-api-key": key} if key else {}


def _download_coverr_music(mood, clip_index):
    """Search Coverr's official API. Free-tier music must be credited in the publishing pack."""
    api_key = str(CFG.get("coverr_api_key") or _secret_value("COVERR_API_KEY")).strip()
    if not api_key:
        return None
    terms = {
        "reflective": "ambient cinematic piano",
        "uplifting": "inspirational uplifting cinematic",
        "playful": "light upbeat acoustic",
        "curious": "minimal suspense ambient",
        "documentary": "documentary ambient instrumental",
    }
    cache_path = os.path.join(ASSET_CACHE_DIR, f"coverr_audio_{mood}_{_music_query_tag()}.json")
    cached = _read_fresh_json(cache_path, 24 * 3600)
    if cached is None:
        try:
            resp = requests.get(
                "https://api.coverr.co/audios",
                params={"query": MUSIC_QUERY_OVERRIDE or terms.get(mood, "ambient instrumental"), "page_size": 30, "sort": "popular"},
                headers=_coverr_headers(), timeout=(15, 30),
            )
            resp.raise_for_status()
            hits = resp.json().get("hits", [])
            _write_json_cache(cache_path, {"hits": hits})
            cached = {"hits": hits}
        except Exception as exc:
            log("music", f"Coverr music API unavailable ({type(exc).__name__}); trying other cleared sources")
            return None
    hits = cached.get("hits", [])
    # Prefer stock, non-premium tracks with a direct master download URL.
    hits = sorted(hits, key=lambda h: (bool(h.get("isPremium")), -int(h.get("downloads", 0) or 0)))
    hits = rerank_candidates("music", hits)
    detail_lookups = 0
    for hit in hits:
        track_id = str(hit.get("id") or "")
        if not track_id or track_id in USED_MUSIC_IDS or hit.get("isPremium") is True or hit.get("is_premium") is True:
            continue
        sponsor_fields = any(hit.get(k) for k in ("sponsored_by", "sponsor", "sponsor_name", "partner_content", "is_sponsored", "is_partner"))
        partner_text = " ".join(str(hit.get(k) or "") for k in ("title", "name", "description", "provider", "source")).lower()
        if sponsor_fields or any(partner in partner_text for partner in ("shutterstock", "istock", "premiumbeat")):
            continue
        urls = hit.get("urls") or {}
        download_url = urls.get("masterDownload") or urls.get("master_download")
        if not download_url:
            # Some catalogue responses omit signed URLs; cap detail lookups to protect the free API quota.
            if detail_lookups >= 3:
                break
            detail_lookups += 1
            try:
                detail = requests.get(f"https://api.coverr.co/audios/{quote(track_id)}", headers=_coverr_headers(), timeout=(15, 25))
                detail.raise_for_status()
                hit = detail.json()
                urls = hit.get("urls") or {}
                download_url = urls.get("masterDownload") or urls.get("master_download")
            except Exception:
                continue
        if not download_url:
            continue
        filename = str(hit.get("filename") or f"coverr_{track_id}.wav")
        ext = Path(filename).suffix.lower() or ".wav"
        local_path = os.path.join(ASSET_CACHE_DIR, f"coverr_music_{safe_filename(track_id, 32)}{ext}")
        try:
            _stream_download(download_url, local_path, max_bytes=100 * 1024 * 1024)
            valid, duration = _probe_audio_asset(local_path)
            if not valid:
                try: os.remove(local_path)
                except OSError: pass
                continue
            creator = hit.get("creator") or hit.get("author") or hit.get("user") or hit.get("createdBy")
            if isinstance(creator, dict):
                creator_name = creator.get("name") or creator.get("username") or creator.get("full_name")
                creator_url = creator.get("url") or creator.get("profile_url")
            else:
                creator_name, creator_url = (str(creator) if creator else None), None
            if creator_name and not creator_url:
                continue  # Free API attribution must link the named creator's Coverr profile.
            asset = {
                "asset_type": "music", "provider": "Coverr API", "asset_id": track_id,
                "title": hit.get("title") or hit.get("name") or Path(filename).stem,
                "artist": creator_name or "Creator not listed in API response",
                "creator_url": creator_url, "source_url": hit.get("url") or hit.get("page_url") or "https://coverr.co/",
                "download_url": download_url, "license": "Coverr Stock Library License",
                "license_url": COVERR_LICENSE_URL, "attribution_required": True,
                "attribution_text": (f"Music: {creator_name} on Coverr" if creator_name else f"Music: {hit.get('title') or hit.get('name') or 'Track'} via Coverr"),
                "local_path": local_path, "duration": duration, "mood": mood,
                "clip_index": clip_index, "is_ai_generated": hit.get("is_ai_generated"),
                "sha256": sha256_file(local_path), "accessed_utc": datetime.now(timezone.utc).isoformat(),
            }
            USED_MUSIC_IDS.add(track_id)
            return asset
        except Exception as exc:
            log("music", f"Coverr track rejected after download/probe ({type(exc).__name__})")
            continue
    return None


def search_internet_archive_music(mood, cache_path, clip_index=0):
    """Broad metadata discovery followed by exact item-level licence and decoded-file verification."""
    cached = _read_fresh_json(cache_path, 6 * 3600)
    if cached:
        if cached.get("status") == "no_verified_asset":
            return None
        cached_id = str(cached.get("identifier") or cached.get("asset_id") or "")
        path = cached.get("local_path")
        if cached_id not in USED_MUSIC_IDS and path and os.path.isfile(path):
            valid, _ = _probe_audio_asset(path)
            if valid:
                USED_MUSIC_IDS.add(cached_id)
                return cached

    # Search the licensed corpus first; rank by mood after retrieval instead of AND-ing every mood word.
    licence_urls = []
    for u in ARCHIVE_LICENSE_ALLOWLIST:
        licence_urls.extend([u, u.replace("https://", "http://", 1)])
    licence_terms = " OR ".join(f'licenseurl:"{u}"' for u in licence_urls)
    query = f"mediatype:audio AND ({licence_terms})"
    try:
        response = requests.get("https://archive.org/advancedsearch.php", params={
            "q": query,
            "fl[]": ["identifier", "title", "creator", "licenseurl", "subject", "description", "downloads", "mediatype"],
            "rows": 100, "page": 1, "output": "json", "sort[]": "downloads desc",
        }, timeout=(15, 35))
        response.raise_for_status()
        docs = response.json().get("response", {}).get("docs", [])
    except Exception as exc:
        log("music", f"Internet Archive unavailable ({type(exc).__name__}); continuing to other music sources")
        return None

    mood_terms = set(re.findall(r"[a-z0-9]+", (mood + " " + MUSIC_QUERY_OVERRIDE).replace("_", " ").lower()))
    # Pull mood-related text from titles/subjects, but do not make it a hard search filter.
    docs.sort(key=lambda d: (
        len(mood_terms & set(re.findall(r"[a-z0-9]+", " ".join(map(str, [d.get("title", ""), d.get("subject", ""), d.get("description", "")])).lower()))),
        int(d.get("downloads", 0) or 0),
    ), reverse=True)

    docs = rerank_candidates("music", docs)
    for doc in docs[:80]:
        identifier = str(doc.get("identifier") or "")
        if not identifier or identifier in USED_MUSIC_IDS:
            continue
        try:
            meta_resp = requests.get(f"https://archive.org/metadata/{quote(identifier)}", timeout=(15, 30))
            meta_resp.raise_for_status()
            item = meta_resp.json()
            md = item.get("metadata", {})
            license_url = _normalise_license_url(md.get("licenseurl") or doc.get("licenseurl"))
            if license_url not in ARCHIVE_LICENSE_ALLOWLIST:
                continue
            creator = md.get("creator") or md.get("artist") or doc.get("creator")
            # CC BY needs a real attribution name; skip metadata that cannot be credited honestly.
            if license_url != "https://creativecommons.org/publicdomain/zero/1.0/" and not creator:
                continue
            metadata_text = " ".join(str(x or "") for x in (md.get("title"), md.get("subject"), md.get("description"), doc.get("title"), doc.get("subject"))).lower()
            music_terms = ("music", "instrumental", "ambient", "piano", "soundtrack", "composition", "cinematic", "orchestra", "synth", "guitar", "jazz", "classical", "folk", "score")
            spoken_terms = ("podcast", "audiobook", "lecture", "sermon", "interview", "speech", "talk show", "radio news", "spoken word")
            if not any(term in metadata_text for term in music_terms) or any(term in metadata_text for term in spoken_terms):
                continue
            files = item.get("files", [])
            audio_files = [f for f in files if str(f.get("name", "")).lower().endswith((".mp3", ".wav", ".ogg", ".flac", ".m4a"))
                           and not f.get("private") and 100_000 <= int(f.get("size", 0) or 0) <= 80 * 1024 * 1024
                           and not any(x in str(f.get("name", "")).lower() for x in ("sample", "preview", "spectrogram", "thumbnail"))]
            if not audio_files:
                continue
            audio_files.sort(key=lambda f: (not str(f.get("name", "")).lower().endswith(".mp3"), -int(f.get("size", 0) or 0)))
            for file_row in audio_files[:3]:
                filename = str(file_row["name"])
                download_url = f"https://archive.org/download/{quote(identifier)}/{quote(filename)}"
                ext = Path(filename).suffix.lower()
                local_path = os.path.join(ASSET_CACHE_DIR, f"ia_{safe_filename(identifier, 35)}_{hashlib.sha256(filename.encode()).hexdigest()[:10]}{ext}")
                try:
                    _stream_download(download_url, local_path, max_bytes=90 * 1024 * 1024)
                    valid, duration = _probe_audio_asset(local_path)
                    if not valid:
                        try: os.remove(local_path)
                        except OSError: pass
                        continue
                    attribution_required = license_url != "https://creativecommons.org/publicdomain/zero/1.0/"
                    asset = {
                        "asset_type": "music", "provider": "Internet Archive", "identifier": identifier,
                        "title": md.get("title") or doc.get("title") or identifier,
                        "artist": creator or "Public domain (creator not listed)",
                        "source_url": f"https://archive.org/details/{identifier}", "download_url": download_url,
                        "license": ARCHIVE_LICENSE_ALLOWLIST[license_url], "license_url": license_url,
                        "attribution_required": attribution_required,
                        "attribution_text": (f"Music: {md.get('title') or doc.get('title') or identifier} by {creator} ({ARCHIVE_LICENSE_ALLOWLIST[license_url]})" if attribution_required else "Public-domain music via Internet Archive"),
                        "license_verified_from_item_metadata": True, "local_path": local_path,
                        "duration": duration, "mood": mood, "clip_index": clip_index,
                        "sha256": sha256_file(local_path), "accessed_utc": datetime.now(timezone.utc).isoformat(),
                    }
                    USED_MUSIC_IDS.add(identifier)
                    _write_json_cache(cache_path, asset)
                    return asset
                except Exception:
                    continue
        except Exception:
            continue

    _write_json_cache(cache_path, {"status": "no_verified_asset", "mood": mood})
    log("music", f"Internet Archive: no matching item with verifiable CC0/CC BY metadata for '{mood}'")
    return None


def generate_original_music(path, duration, mood, seed):
    """Generate an original, low-level instrumental bed; no third-party recording or sample is used."""
    sr = 24000
    count = int(max(1.0, duration) * sr)
    t = np.arange(count, dtype=np.float32) / sr
    rng = np.random.default_rng(seed)
    progressions = {
        "reflective": [(110.00, 164.81, 220.00), (98.00, 146.83, 196.00), (130.81, 196.00, 261.63), (110.00, 164.81, 220.00)],
        "uplifting": [(130.81, 164.81, 196.00), (174.61, 220.00, 261.63), (146.83, 185.00, 220.00), (196.00, 246.94, 293.66)],
        "playful": [(146.83, 185.00, 220.00), (164.81, 207.65, 246.94), (196.00, 246.94, 293.66), (164.81, 207.65, 246.94)],
        "curious": [(110.00, 138.59, 164.81), (116.54, 146.83, 174.61), (98.00, 123.47, 146.83), (110.00, 138.59, 164.81)],
        "documentary": [(123.47, 155.56, 185.00), (110.00, 138.59, 164.81), (130.81, 164.81, 196.00), (123.47, 155.56, 185.00)],
    }
    progression = progressions.get(mood, progressions["documentary"])
    chord_sec = 4.0
    chord_idx = np.minimum((t / chord_sec).astype(int) % len(progression), len(progression)-1)
    pad = np.zeros(count, dtype=np.float32)
    for chord_i, notes in enumerate(progression):
        mask = chord_idx == chord_i
        local_t = (t[mask] % chord_sec)
        env = np.minimum(1.0, local_t / 1.2) * np.minimum(1.0, (chord_sec-local_t) / 1.0)
        tone = np.zeros(mask.sum(), dtype=np.float32)
        for note in notes:
            tone += np.sin(2*np.pi*note*t[mask]) + 0.22*np.sin(4*np.pi*note*t[mask])
        pad[mask] = tone / len(notes) * env
    pulse = np.zeros(count, dtype=np.float32)
    beat = 0.75 if mood in ("uplifting", "playful") else 1.0
    for beat_start in np.arange(0, duration, beat):
        start = int(beat_start * sr)
        end = min(count, start + int(0.20 * sr))
        if end > start:
            local = np.arange(end-start, dtype=np.float32) / sr
            freq = rng.choice([55.0, 65.4, 73.4, 82.4])
            pulse[start:end] += 0.10 * np.sin(2*np.pi*freq*local) * np.exp(-local*18.0)
    lfo = 0.82 + 0.10*np.sin(2*np.pi*0.08*t)
    mono = np.clip((0.11 * pad + pulse) * lfo, -0.30, 0.30)
    fade = min(2.0, duration/4)
    if fade > 0:
        fade_n = max(1, int(fade*sr))
        mono[:fade_n] *= np.linspace(0, 1, fade_n, dtype=np.float32)
        mono[-fade_n:] *= np.linspace(1, 0, fade_n, dtype=np.float32)
    stereo = np.stack([mono, np.roll(mono, int(0.012*sr))*0.96], axis=1)
    wavfile.write(path, sr, (stereo * 32767).astype(np.int16))
    return path


def choose_verified_manual_music(mood, index):
    """Manual pool items require a path-contained file and a licence-name/URL match."""
    manifest_path = os.path.join(MUSIC_POOL_DIR, "music_pool_manifest.json")
    if not os.path.isfile(manifest_path):
        return None
    try:
        with open(manifest_path, encoding="utf-8") as f:
            rows = json.load(f)
    except Exception:
        return None
    strict_names = {v: k for k, v in ARCHIVE_LICENSE_ALLOWLIST.items()}
    for row in rows:
        track_path = _safe_pool_file(MUSIC_POOL_DIR, row.get("filename"))
        if not track_path:
            continue
        manual_id = str(row.get("asset_id") or row.get("filename") or Path(track_path).name)
        if manual_id in USED_MUSIC_IDS:
            continue
        license_name = str(row.get("license") or "").strip()
        license_url = _normalise_license_url(row.get("license_url"))
        is_mixkit = license_name == "Mixkit Stock Music Free License" and license_url == _normalise_license_url(MIXKIT_MUSIC_LICENSE_URL)
        if is_mixkit and row.get("manual_verified") is not True:
            continue
        if not is_mixkit and strict_names.get(license_name) != license_url:
            continue
        if not row.get("source_url"):
            continue
        if license_name.startswith("CC BY") and not (row.get("artist") or row.get("creator") or row.get("attribution")):
            continue
        valid, duration = _probe_audio_asset(track_path)
        if not valid:
            continue
        creator = row.get("artist") or row.get("creator") or "Unknown"
        USED_MUSIC_IDS.add(manual_id)
        return {
            "asset_type": "music", "provider": row.get("provider", "User-imported verified pool"),
            "title": row.get("title", Path(track_path).stem), "artist": creator,
            "source_url": row.get("source_url"), "download_url": row.get("download_url"),
            "license": license_name, "license_url": license_url,
            "attribution_required": bool(row.get("attribution_required", license_name.startswith("CC BY"))),
            "attribution_text": row.get("attribution") or f"Music: {row.get('title', Path(track_path).stem)} by {creator}",
            "local_path": track_path, "duration": duration, "mood": mood, "clip_index": index,
            "sha256": sha256_file(track_path), "selection": "verified_manual_import",
            "accessed_utc": datetime.now(timezone.utc).isoformat(),
        }
    return None


def choose_music_for_plan(plan, index):
    global MUSIC_QUERY_OVERRIDE
    mood = plan_mood(plan)
    _brief = plan.notes.get("music_brief") or {}
    MUSIC_QUERY_OVERRIDE = re.sub(r"[^a-z0-9 ]", " ", str(_brief.get("search_terms") or "").lower()).strip()[:60]
    RERANK_CONTEXT["music"] = (f"Background music for a short vertical video titled '{plan.title}'. Mood: {mood}. Energy: {_brief.get('energy', 'medium')}. "
                               f"Wanted: {_brief.get('search_terms') or mood}. Avoid: {_brief.get('avoid') or 'vocals, loud drums'}. It must not compete with speech.") if _brief else ""
    try:
        return _choose_music_for_plan_inner(plan, index, mood)
    finally:
        MUSIC_QUERY_OVERRIDE = ""
        RERANK_CONTEXT["music"] = ""


def _choose_music_for_plan_inner(plan, index, mood):
    asset = _download_coverr_music(mood, index) if CFG.get("enable_coverr_assets", True) else None
    if not asset and CFG.get("enable_internet_archive_music", True):
        ia_cache = os.path.join(ASSET_CACHE_DIR, f"ia_music_{mood}_{_music_query_tag()}.json")
        asset = search_internet_archive_music(mood, ia_cache, index)
    if not asset:
        asset = choose_verified_manual_music(mood, index)
    if asset and asset.get("local_path") and os.path.isfile(asset["local_path"]):
        asset["clip_index"] = index
        ASSET_MANIFEST["music"].append(asset)
        log("music", f"clip {index}: licensed track selected — {asset.get('provider')} / {asset.get('title')}")
        return asset["local_path"], asset

    music_path = os.path.join(ASSET_DIR, f"original_music_{index:02d}.wav")
    seed = int(hashlib.sha256((SOURCE.video_id + str(index) + plan.title).encode()).hexdigest()[:8], 16)
    generate_original_music(music_path, plan.duration, mood, seed)
    asset = {
        "asset_type": "music", "provider": "Project Ziroq procedural generator",
        "title": f"Original {mood} instrumental bed", "artist": "Generated locally",
        "source_url": None, "download_url": None,
        "license": "Original procedural audio generated by this notebook", "license_url": None,
        "attribution_required": False, "attribution_text": "Original music generated locally by Project Ziroq",
        "local_path": music_path, "mood": mood, "clip_index": index,
        "sha256": sha256_file(music_path), "selection": "original_fallback",
        "accessed_utc": datetime.now(timezone.utc).isoformat(),
    }
    ASSET_MANIFEST["music"].append(asset)
    log("music", f"clip {index}: original instrumental bed generated (no unverified third-party track used)")
    return music_path, asset


def _coverr_search_video(query, cache_path):
    """Coverr free API search; reject premium, sponsored, and AI-generated records."""
    api_key = str(CFG.get("coverr_api_key") or _secret_value("COVERR_API_KEY")).strip()
    if not api_key:
        return None
    cached = _read_fresh_json(cache_path, 24 * 3600)
    if cached is None:
        try:
            resp = requests.get("https://api.coverr.co/videos", params={
                "query": query, "page_size": 20, "sort": "popular", "urls": "true",
                "filter": "is_ai_generated:false",
            }, headers=_coverr_headers(), timeout=(15, 30))
            resp.raise_for_status()
            hits = resp.json().get("hits", [])
            _write_json_cache(cache_path, {"hits": hits})
            cached = {"hits": hits}
        except Exception as exc:
            log("broll", f"Coverr API unavailable ({type(exc).__name__})")
            return None
    hits = cached.get("hits", [])
    query_terms = set(re.findall(r"[a-z0-9]+", query.lower()))
    hits = sorted(hits, key=lambda h: (
        len(query_terms & set(re.findall(r"[a-z0-9]+", " ".join([str(h.get("title", "")), str(h.get("description", "")), " ".join(h.get("tags") or [])]).lower()))),
        bool(h.get("is_vertical")), int(h.get("max_width", 0) or 0),
    ), reverse=True)
    global BROLL_DOWNLOADS_THIS_RUN
    hits = rerank_candidates("broll", hits)
    for hit in hits:
        asset_id = str(hit.get("id") or "")
        if not asset_id or asset_id in USED_BROLL_IDS:
            continue
        if hit.get("is_ai_generated") is not False:
            continue
        if hit.get("is_premium") is True or hit.get("isPremium") is True:
            continue
        # Sponsored partner items are not covered by the Coverr stock-library licence.
        sponsor_fields = any(hit.get(k) for k in ("sponsored_by", "sponsor", "sponsor_name", "partner_content", "is_sponsored", "is_partner"))
        partner_text = " ".join(str(hit.get(k) or "") for k in ("title", "description", "provider", "source", "sponsor_name")).lower()
        if sponsor_fields or any(partner in partner_text for partner in ("shutterstock", "istock", "premiumbeat")):
            continue
        duration = float(hit.get("duration", 0) or 0)
        if duration < 3.0:
            continue
        urls = hit.get("urls") or {}
        download_url = urls.get("mp4_download")
        if not download_url:
            continue
        local_path = os.path.join(ASSET_CACHE_DIR, f"coverr_video_{safe_filename(asset_id, 32)}.mp4")
        try:
            if not os.path.isfile(local_path):
                if BROLL_DOWNLOADS_THIS_RUN >= int(CFG.get("broll_max_downloads_per_run", 5)):
                    log("broll", "per-run stock-download cap reached; using already cached/manual assets only")
                    return None
                _stream_download(download_url, local_path, max_bytes=180 * 1024 * 1024)
                BROLL_DOWNLOADS_THIS_RUN += 1
            valid, probed_duration = _probe_video_asset(local_path)
            if not valid:
                try: os.remove(local_path)
                except OSError: pass
                continue
            creator = hit.get("creator") or hit.get("author") or hit.get("user") or hit.get("created_by")
            if isinstance(creator, dict):
                creator_name = creator.get("name") or creator.get("username")
                creator_url = creator.get("url") or creator.get("profile_url")
            else:
                creator_name, creator_url = (str(creator) if creator else None), None
            if creator_name and not creator_url:
                continue  # Free API attribution must link the named creator's Coverr profile.
            # Download tracking is a condition of Coverr API access under its 4 Nov 2026 terms.
            tracked = requests.patch(f"https://api.coverr.co/videos/{quote(asset_id)}/stats/downloads", headers=_coverr_headers(), timeout=(10, 20))
            tracked.raise_for_status()
            asset = {
                "asset_type": "broll_video", "provider": "Coverr API", "asset_id": asset_id,
                "title": hit.get("title") or "Coverr stock video", "author": creator_name or "Creator not listed in API response",
                "creator_url": creator_url, "source_url": hit.get("url") or hit.get("page_url") or "https://coverr.co/", "download_url": download_url,
                "license": "Coverr Stock Library License (free API tier)", "license_url": COVERR_LICENSE_URL,
                "attribution_required": True,
                "attribution_text": (f"Video by {creator_name} on Coverr" if creator_name else "Video footage via Coverr"),
                "local_path": local_path, "duration": probed_duration, "query": query,
                "is_ai_generated": False, "sha256": sha256_file(local_path),
                "clip_index": None, "accessed_utc": datetime.now(timezone.utc).isoformat(),
            }
            USED_BROLL_IDS.add(asset_id)
            return asset
        except Exception as exc:
            log("broll", f"Coverr result {asset_id} rejected ({type(exc).__name__})")
            try:
                if os.path.exists(local_path): os.remove(local_path)
            except OSError:
                pass
    return None


def pixabay_search_video(query, cache_path):
    """Pixabay API with durable 24-hour response caching, including empty results."""
    if not CFG.get("enable_pixabay_broll", True):
        return None
    api_key = str(CFG.get("pixabay_api_key") or _secret_value("PIXABAY_API_KEY")).strip()
    if not api_key:
        return None
    cached = _read_fresh_json(cache_path, max(24 * 3600, int(CFG.get("broll_cache_hours", 24)) * 3600))
    if cached is None:
        try:
            response = requests.get("https://pixabay.com/api/videos/", params={
                "key": api_key, "q": query[:100], "safesearch": "true", "per_page": 20,
                "order": "popular", "min_width": 640,
            }, timeout=(15, 30))
            response.raise_for_status()
            hits = response.json().get("hits", [])
            # Cache the full search response, including an empty result, for 24 hours as required by Pixabay.
            _write_json_cache(cache_path, {"hits": hits, "query": query})
            cached = {"hits": hits}
        except Exception as exc:
            log("broll", f"Pixabay search failed ({type(exc).__name__})")
            return None
    hits = cached.get("hits", [])
    global BROLL_DOWNLOADS_THIS_RUN
    hits.sort(key=lambda h: (int(h.get("duration", 0)) < 4, abs(int(h.get("duration", 0)) - 8), -int(h.get("views", 0) or 0)))
    hits = rerank_candidates("broll", hits)
    for hit in hits:
        asset_id = str(hit.get("id") or "")
        if not asset_id or asset_id in USED_BROLL_IDS:
            continue
        variants = hit.get("videos") or {}
        choices = sorted(variants.values(), key=lambda v: (abs(int(v.get("width", 0) or 0) - 1080), -int(v.get("width", 0) or 0)))
        if not choices or not choices[0].get("url"):
            continue
        chosen = choices[0]
        local_path = os.path.join(ASSET_CACHE_DIR, f"pixabay_video_{safe_filename(asset_id, 32)}.mp4")
        try:
            if not os.path.isfile(local_path):
                if BROLL_DOWNLOADS_THIS_RUN >= int(CFG.get("broll_max_downloads_per_run", 5)):
                    log("broll", "per-run stock-download cap reached; using cached/manual assets only")
                    return None
                _stream_download(chosen["url"], local_path, max_bytes=180 * 1024 * 1024)
                BROLL_DOWNLOADS_THIS_RUN += 1
            valid, duration = _probe_video_asset(local_path)
            if not valid:
                try: os.remove(local_path)
                except OSError: pass
                continue
            asset = {
                "asset_type": "broll_video", "provider": "Pixabay API", "asset_id": asset_id,
                "title": hit.get("tags") or "Pixabay stock video", "author": hit.get("user", "Unknown"),
                "source_url": hit.get("pageURL"), "download_url": chosen["url"],
                "license": "Pixabay Content License", "license_url": PIXABAY_LICENSE_URL,
                "attribution_required": False, "attribution_text": f"Video by {hit.get('user', 'creator')} via Pixabay",
                "local_path": local_path, "duration": duration, "query": query,
                "sha256": sha256_file(local_path), "clip_index": None,
                "accessed_utc": datetime.now(timezone.utc).isoformat(),
            }
            USED_BROLL_IDS.add(asset_id)
            return asset
        except Exception as exc:
            log("broll", f"Pixabay result {asset_id} rejected ({type(exc).__name__})")
            try:
                if os.path.exists(local_path): os.remove(local_path)
            except OSError:
                pass
    return None


def choose_verified_manual_broll(query, index):
    """Manual B-roll pool for no-key runs; reject unverified paths/licences and invalid media."""
    manifest_path = os.path.join(BROLL_POOL_DIR, "broll_pool_manifest.json")
    if not os.path.isfile(manifest_path):
        return None
    try:
        with open(manifest_path, encoding="utf-8") as f:
            rows = json.load(f)
    except Exception:
        return None
    allowed_manual = {
        "Mixkit Stock Video Free License": _normalise_license_url(MIXKIT_VIDEO_LICENSE_URL),
        "Pixabay Content License": _normalise_license_url(PIXABAY_LICENSE_URL),
        "Coverr Stock Library License": _normalise_license_url(COVERR_LICENSE_URL),
        **{name: _normalise_license_url(url) for url, name in ARCHIVE_LICENSE_ALLOWLIST.items()},
    }
    for row in rows:
        local_path = _safe_pool_file(BROLL_POOL_DIR, row.get("filename"))
        if not local_path or row.get("manual_verified") is not True or not row.get("source_url"):
            continue
        manual_id = str(row.get("asset_id") or row.get("filename") or Path(local_path).name)
        if manual_id in USED_BROLL_IDS:
            continue
        license_name = str(row.get("license") or "")
        license_url = _normalise_license_url(row.get("license_url"))
        if allowed_manual.get(license_name) != license_url:
            continue
        if license_name.startswith("CC BY") and not (row.get("author") or row.get("creator") or row.get("attribution")):
            continue
        valid, duration = _probe_video_asset(local_path)
        if not valid:
            continue
        USED_BROLL_IDS.add(manual_id)
        return {
            "asset_type": "broll_video", "provider": row.get("provider", "User-imported verified pool"),
            "asset_id": row.get("asset_id", Path(local_path).stem), "title": row.get("title", Path(local_path).stem),
            "author": row.get("author", "Creator not listed"), "source_url": row.get("source_url"),
            "download_url": row.get("download_url"), "license": license_name, "license_url": license_url,
            "attribution_required": bool(row.get("attribution_required", license_name.startswith("Coverr") or license_name.startswith("CC BY"))),
            "attribution_text": row.get("attribution") or f"Footage: {row.get('title', Path(local_path).stem)}",
            "local_path": local_path, "duration": duration, "query": query,
            "sha256": sha256_file(local_path), "clip_index": index,
            "accessed_utc": datetime.now(timezone.utc).isoformat(),
        }
    return None


def broll_query_for_plan(plan):
    _gq = re.sub(r"[^a-zA-Z0-9 ]", " ", str(plan.notes.get("broll_query") or "")).strip()
    if len(_gq.split()) >= 2:
        return _gq[:80]
    topic = (plan.notes.get("topics") or ["business"])[0]
    phrase_map = {
        "money": "financial planning calculator business office",
        "business": "entrepreneur working business office",
        "entrepreneurship": "small business founder working",
        "self-improvement": "person training sunrise focused work",
        "psychology": "human behavior thoughtful conversation",
        "leadership": "team collaboration meeting",
        "technology": "technology computer data close up",
        "relationships": "people conversation human connection",
    }
    return phrase_map.get(topic, f"{topic} documentary cinematic")


def choose_broll_for_plan(plan, index):
    query = broll_query_for_plan(plan)
    RERANK_CONTEXT["broll"] = (f"A 3-4 second cutaway for a short video about: {plan.title}. Wanted visuals: {query}. "
                               f"Must illustrate the topic, show no identifiable people claiming to be the speaker, no text or logos.") if plan.notes.get("decided_by") == "gemini_director" else ""
    try:
        return _choose_broll_inner(plan, index, query)
    finally:
        RERANK_CONTEXT["broll"] = ""


def _choose_broll_inner(plan, index, query):
    # Coverr first (free key, commercially usable stock); Pixabay second (free key).
    if CFG.get("enable_coverr_assets", True) and (CFG.get("coverr_api_key") or _secret_value("COVERR_API_KEY")):
        for candidate_query in (query, "cinematic " + query.split()[0]):
            cache = os.path.join(ASSET_CACHE_DIR, f"coverr_broll_{hashlib.sha256(candidate_query.encode()).hexdigest()[:12]}.json")
            asset = _coverr_search_video(candidate_query, cache)
            if asset:
                asset["clip_index"] = index
                return asset
    if CFG.get("pixabay_api_key") or _secret_value("PIXABAY_API_KEY"):
        for candidate_query in (query, query.split()[0] + " documentary"):
            cache = os.path.join(ASSET_CACHE_DIR, f"pixabay_broll_{hashlib.sha256(candidate_query.encode()).hexdigest()[:12]}.json")
            asset = pixabay_search_video(candidate_query, cache)
            if asset:
                asset["clip_index"] = index
                return asset
    return choose_verified_manual_broll(query, index)


MUSIC_ASSETS, BROLL_ASSETS = {}, {}
for clip_index, plan in enumerate(PLANS, 1):
    music_path, music_asset = choose_music_for_plan(plan, clip_index)
    MUSIC_ASSETS[clip_index] = {"path": music_path, "asset": music_asset}
    broll_asset = choose_broll_for_plan(plan, clip_index)
    BROLL_ASSETS[clip_index] = broll_asset
    if broll_asset:
        ASSET_MANIFEST["broll"].append(broll_asset)
        log("broll", f"clip {clip_index}: contextual {broll_asset.get('provider')} visual selected ({broll_asset.get('asset_id')})")
    else:
        keyless = not (CFG.get("coverr_api_key") or _secret_value("COVERR_API_KEY") or CFG.get("pixabay_api_key") or _secret_value("PIXABAY_API_KEY"))
        why = "no free stock API key configured and no verified manual asset pool found" if keyless else "no relevant, downloadable, verified asset passed provider checks"
        log("broll", f"clip {clip_index}: {why}; source footage remains primary")

atomic_write_json(os.path.join(REPORT_DIR, "asset_license_manifest.json"), ASSET_MANIFEST)


In [ ]:
# CELL 9 — Dynamic framing, premium word-level ASS captions, mixed audio, rendering, and QA
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageEnhance


def find_font_path():
    candidates = glob.glob("/usr/share/fonts/**/DejaVuSans-Bold.ttf", recursive=True)
    if not candidates:
        raise RuntimeError("DejaVu Sans Bold font is missing; run Cell 0")
    return candidates[0]


def escape_ass(text):
    return (text or "").replace("{", "(").replace("}", ")").replace("\\", "")


def ass_time(seconds):
    cs = int(round(max(0.0, seconds) * 100))
    return f"{cs//360000}:{(cs//6000)%60:02d}:{(cs//100)%60:02d}.{cs%100:02d}"


def build_ass(plan, words, ass_path):
    mapped = timeline_words(plan, words)
    if not mapped:
        raise RuntimeError("No source-timestamped words map to this story plan")
    groups, current = [], []
    def flush():
        nonlocal current
        if current:
            groups.append(current)
            current = []
    for item in mapped:
        if current:
            gap = item["start"] - current[-1]["end"]
            char_count = sum(len(x["word"]) + 1 for x in current) + len(item["word"])
            if item["segment_index"] != current[-1]["segment_index"] or gap > 0.55 or len(current) >= 4 or char_count > 26 or re.search(r"[.!?,;:]$", current[-1]["word"]):
                flush()
        current.append(item)
    flush()
    header = (
        "[Script Info]\nScriptType: v4.00+\nPlayResX: 1080\nPlayResY: 1920\nWrapStyle: 0\nScaledBorderAndShadow: yes\n\n"
        "[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
        "Style: Cap,DejaVu Sans,68,&H0000D7FF,&H00FFFFFF,&H00101010,&HAA000000,-1,0,0,0,100,100,0,0,1,6,2,2,80,80,500,1\n"
        "Style: Hook,DejaVu Sans,56,&H00FFFFFF,&H00FFFFFF,&H00101010,&HAA000000,-1,0,0,0,100,100,0,0,1,5,2,8,90,90,160,1\n\n"
        "[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
    )
    lines, events = [], []
    # Title is explicitly a generated editorial headline, not spoken dialogue.
    title = escape_ass(plan.title.upper()[:72])
    lines.append(f"Dialogue: 1,{ass_time(0)},{ass_time(min(2.6, plan.duration))},Hook,,0,0,0,,{title}")
    for group in groups:
        chunks = []
        for idx, item in enumerate(group):
            next_start = group[idx+1]["start"] if idx + 1 < len(group) else item["end"]
            karaoke_cs = max(1, int(round((next_start - item["start"]) * 100)))
            chunks.append("{\\k" + str(karaoke_cs) + "}" + escape_ass(item["word"]) + (" " if idx + 1 < len(group) else ""))
        start = group[0]["start"]
        end = max(group[-1]["end"], start + 0.12)
        phrase = " ".join(x["word"] for x in group)
        lines.append(f"Dialogue: 0,{ass_time(start)},{ass_time(end)},Cap,,0,0,0,,{''.join(chunks)}")
        events.append({"start": start, "end": end, "text": phrase})
    with open(ass_path, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(lines) + "\n")
    return events


def crop_geometry(sw, sh):
    target = CFG["out_w"] / CFG["out_h"]
    if sw / sh > target:
        ch = sh
        cw = int(round((sh * target) / 2) * 2)
    else:
        cw = sw
        ch = int(round((sw / target) / 2) * 2)
    cw, ch = min(cw, sw - sw % 2), min(ch, sh - sh % 2)
    return max(2, cw), max(2, ch)


FACE_CASCADE_NAME = "haarcascade_frontalface_default.xml"
_FACE_DETECTOR_STATE = {"resolved": False, "kind": None, "detector": None, "reason": ""}


def _load_cascade(path):
    """Return a loaded cv2.CascadeClassifier, or None if the file is missing/empty/unreadable."""
    try:
        if not path or not os.path.isfile(path):
            return None
        cascade = cv2.CascadeClassifier(path)
        return None if cascade.empty() else cascade
    except Exception:
        return None


def extract_member_from_wheel(wheel_path, member_suffix, dest_path):
    """Copy one file out of a wheel (a zip) without installing it; returns dest_path or None."""
    import zipfile
    with zipfile.ZipFile(wheel_path) as zf:
        matches = [n for n in zf.namelist() if n.endswith(member_suffix)]
        if not matches:
            return None
        data = zf.read(matches[0])
    if len(data) < 1000:
        return None
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    tmp = dest_path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(data)
    os.replace(tmp, dest_path)
    return dest_path


def recover_cascade_from_opencv4_wheel(dest_path):
    """OpenCV 5.x pip wheels ship an empty cv2/data folder. Download (not install) an OpenCV 4.x wheel
    and copy only the Haar cascade XML, so the NumPy/PyAV/OpenCV stack stays untouched."""
    tmp_dir = os.path.join(CACHE_ROOT, "_wheel_tmp")
    shutil.rmtree(tmp_dir, ignore_errors=True)
    os.makedirs(tmp_dir, exist_ok=True)
    try:
        result = run([sys.executable, "-m", "pip", "download", "opencv-python-headless<5", "--no-deps",
                      "--only-binary=:all:", "-d", tmp_dir, "-q"], check=False, timeout=300, quiet=True)
        wheels = glob.glob(os.path.join(tmp_dir, "opencv_python_headless-*.whl"))
        if result.returncode != 0 or not wheels:
            return None
        return extract_member_from_wheel(wheels[0], "/" + FACE_CASCADE_NAME, dest_path)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def get_face_detector(force_reload=False):
    """Resolve a face detector once: bundled Haar -> cached/recovered Haar -> none (stated loudly)."""
    st = _FACE_DETECTOR_STATE
    if st["resolved"] and not force_reload:
        return st
    st.update({"resolved": True, "kind": None, "detector": None, "reason": ""})
    bundled = _load_cascade(os.path.join(cv2.data.haarcascades, FACE_CASCADE_NAME)) if hasattr(cv2, "data") else None
    if bundled is not None:
        st.update(kind="haar_bundled", detector=bundled)
    else:
        cached_path = os.path.join(CACHE_ROOT, "models", FACE_CASCADE_NAME)
        cached = _load_cascade(cached_path)
        if cached is None:
            log("framing", f"OpenCV {cv2.__version__} has no bundled face cascade; trying to recover one from an OpenCV 4.x wheel")
            try:
                cached = _load_cascade(recover_cascade_from_opencv4_wheel(cached_path))
            except Exception as exc:
                st["reason"] = f"recovery failed: {type(exc).__name__}"
        if cached is not None:
            st.update(kind="haar_recovered", detector=cached)
    if st["detector"] is None:
        st["reason"] = st["reason"] or "no face cascade bundled with OpenCV and recovery was not possible"
        log("framing", f"WARNING: no face detector available ({st['reason']}); shots keep the full frame "
                       f"(framing_uncertain_mode={CFG.get('framing_uncertain_mode')})")
    else:
        log("framing", f"face detector ready: {st['kind']}")
    return st


def _uncertain_framing(reason, samples=0, **extra):
    """Visual completeness beats a blind close-up: when unsure, keep more of the original frame."""
    mode = CFG.get("framing_uncertain_mode", "fit_blur")
    if mode not in ("fit_blur", "center_crop"):
        mode = "fit_blur"
    return {"mode": mode, "reason": f"{reason}; uncertain -> {mode}", "tracking_samples": samples,
            "uncertain": True, **extra}


def _detect_local_framing(source, segment, sample_step=0.8, detector=None):
    """Face-centre tracking heuristic; never claims speaker identity or voice diarization.
    `detector` is any object with detectMultiScale(gray, ...); default resolves via get_face_detector()."""
    cw, ch = crop_geometry(source.width, source.height)
    if source.width / source.height <= CFG["out_w"] / CFG["out_h"]:
        return {"mode": "fit_blur", "reason": "source is not wider than a 9:16 crop", "tracking_samples": 0}
    det = detector if detector is not None else get_face_detector()["detector"]
    if det is None:
        return _uncertain_framing("no face detector available")
    cap = None
    try:
        cap = cv2.VideoCapture(source.video_path)
        points, two_person_samples = [], 0
        t = 0.0
        while t < segment.duration:
            cap.set(cv2.CAP_PROP_POS_MSEC, (segment.source_start + t) * 1000)
            ok, frame = cap.read()
            if not ok:
                t += sample_step
                continue
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = det.detectMultiScale(gray, scaleFactor=1.12, minNeighbors=5, minSize=(36, 36))
            if len(faces) >= 2:
                centers = sorted([x + w/2 for x, y, w, h in faces], key=lambda x: x)
                if (centers[-1] - centers[0]) > source.width * 0.28:
                    two_person_samples += 1
            if len(faces):
                # Largest face is used as a framing target; it is not a verified active-speaker detector.
                x, y, w, h = max(faces, key=lambda z: z[2] * z[3])
                points.append((t, float((x + w/2) / source.width)))
            t += sample_step
        if two_person_samples >= 2:
            return {"mode": "fit_blur", "reason": "multiple separated faces detected; preserve the full composition",
                    "tracking_samples": len(points), "multi_person_samples": two_person_samples}
        if len(points) < 2:
            return _uncertain_framing("insufficient stable face detections (slides, screen share or no visible face?)", len(points))
        # Smooth tracked centre positions to avoid crop jitter.
        smoothed, alpha = [], 0.35
        state = points[0][1]
        for tt, xx in points:
            state = alpha * xx + (1-alpha) * state
            smoothed.append((tt, state))
        max_x = max(0, source.width - cw)
        keyframes = [(tt, max(0.0, min(float(max_x), cx * source.width - cw/2))) for tt, cx in smoothed]
        return {"mode": "face_track", "keyframes": keyframes, "tracking_samples": len(points),
                "reason": "largest detected face centre, smoothed over time"}
    except Exception as exc:
        return _uncertain_framing(f"tracking error {type(exc).__name__}")
    finally:
        if cap is not None:
            cap.release()


def framing_from_hint(source, segment, hint):
    """Turn Gemini's layout/speaker_x hint into a framing dict, or None if the hint cannot be trusted for this segment."""
    if not hint:
        return None
    layout = hint.get("layout")
    xs = hint.get("speakers_x") or []
    if layout == "multi_speaker" and len(xs) == 2 and CFG.get("enable_split_screen", True):
        w2 = int(round(source.height * 1080 / 960)); w2 -= w2 % 2
        if source.width >= w2 + 8 and (xs[1] - xs[0]) >= 0.18:
            xl = int(max(0, min(source.width - w2, xs[0] * source.width - w2 / 2)))
            xr = int(max(0, min(source.width - w2, xs[1] * source.width - w2 / 2)))
            if xr - xl >= 0.18 * source.width:
                return {"mode": "split_two", "crop_w": w2, "x_left": xl, "x_right": xr, "tracking_samples": 0, "decided_by": "gemini",
                        "reason": "Gemini: two speakers side by side -> stacked layout (not locally verified)"}
    if layout in ("multi_speaker", "slides_or_screen"):
        return {"mode": "fit_blur", "reason": f"Gemini layout '{layout}'; keep the full composition", "tracking_samples": 0, "decided_by": "gemini"}
    if layout != "single_speaker":
        return None
    cw, ch = crop_geometry(source.width, source.height)
    max_x = max(0, source.width - cw)
    rel = [(t - segment.source_start, x) for t, x in hint.get("points", []) if segment.source_start - 3.0 <= t <= segment.source_end + 3.0]
    if len(rel) < 2:
        return None
    if max(x for _, x in rel) - min(x for _, x in rel) > 0.45:     # implausible jumps -> do not trust
        return None
    state, kf = rel[0][1], []
    for tt, xx in rel:
        state = 0.5 * xx + 0.5 * state
        kf.append((max(0.0, tt), max(0.0, min(float(max_x), state * source.width - cw / 2))))
    return {"mode": "face_track", "keyframes": kf, "tracking_samples": len(kf),
            "reason": "Gemini speaker position samples (not locally verified)", "decided_by": "gemini"}


def detect_segment_framing(source, segment, sample_step=0.8, detector=None, hint=None):
    """Local face tracking first; Gemini's hint is used when local detection is unavailable/uncertain, or when Gemini says the shot is a
    multi-person / slides layout (where a close crop would cut content)."""
    if source.width / source.height <= CFG["out_w"] / CFG["out_h"]:
        return _detect_local_framing(source, segment, sample_step, detector)
    hinted = framing_from_hint(source, segment, hint)
    if hinted and hinted["mode"] in ("fit_blur", "split_two") and CFG.get("trust_gemini_layout", True):
        return hinted
    local = _detect_local_framing(source, segment, sample_step, detector)
    if local.get("uncertain") and hinted and CFG.get("use_gemini_framing_hint", True):
        return hinted
    return local


def ffmpeg_crop_x_expr(keyframes, max_x):
    if not keyframes:
        return str(int(max_x/2))
    # Piecewise-linear interpolation in segment-local seconds; commas are escaped for filtergraph parsing.
    expr = f"{keyframes[-1][1]:.2f}"
    for i in range(len(keyframes)-2, -1, -1):
        t0, x0 = keyframes[i]
        t1, x1 = keyframes[i+1]
        dt = max(0.001, t1-t0)
        linear = f"({x0:.2f}+({x1-x0:.2f})*(t-{t0:.3f})/{dt:.3f})"
        expr = f"if(lt(t\\,{t1:.3f})\\,{linear}\\,{expr})"
    t0, x0 = keyframes[0]
    expr = f"if(lt(t\\,{t0:.3f})\\,{x0:.2f}\\,{expr})"
    return f"max(0\\,min({max_x}\\,{expr}))"


def make_audio_mix_graph(music_input_index, total_duration, volume=0.18, ceiling=0.80, sfx_input_index=None):
    """Pure helper used by render_plan: dialogue-triggered sidechain ducking, optional SFX bed, safe final mix."""
    parts = [
        "[abase]asplit=2[speech][sidechain]",
        f"[{music_input_index}:a]aresample=48000,atrim=duration={total_duration:.3f},asetpts=PTS-STARTPTS,volume={volume:.3f}[music]",
        "[music][sidechain]sidechaincompress=threshold=0.035:ratio=8:attack=25:release=450[ducked]",
    ]
    mix_in, n = "[speech][ducked]", 2
    if sfx_input_index is not None:
        parts.append(f"[{sfx_input_index}:a]aresample=48000,atrim=duration={total_duration:.3f},asetpts=PTS-STARTPTS,volume=0.55[sfx]")
        mix_in, n = "[speech][ducked][sfx]", 3
    parts.append(f"{mix_in}amix=inputs={n}:duration=first:dropout_transition=2,highpass=f=65,loudnorm=I=-16:TP=-2.0:LRA=10,"
                 f"alimiter=limit={ceiling:.3f}:attack=3:release=40:level=disabled,aresample=48000[aout]")
    return parts


SFX_KINDS = ("whoosh", "impact", "pop", "riser")
SFX_SR = 48000


def synth_sfx(kind, seed=0):
    """Procedural sound effects (no samples, no licences). Normalised to a 0.6 peak; the mix level is set in the graph."""
    rng = np.random.default_rng(seed)
    sr = SFX_SR
    if kind == "whoosh":
        n = int(0.55 * sr); t = np.linspace(0, 1, n); noise = rng.standard_normal(n)
        low = np.convolve(noise, np.ones(15) / 15, mode="same")
        high = noise - np.convolve(noise, np.ones(61) / 61, mode="same")
        y = ((1 - t) * low + t * high) * np.sin(np.pi * t) ** 2
    elif kind == "impact":
        n = int(0.7 * sr); t = np.arange(n) / sr
        y = np.sin(2 * np.pi * (55 + 40 * np.exp(-t * 18)) * t) * np.exp(-t * 7) + 0.3 * rng.standard_normal(n) * np.exp(-t * 40)
    elif kind == "pop":
        n = int(0.12 * sr); t = np.arange(n) / sr
        y = np.sin(2 * np.pi * (700 + 500 * np.exp(-t * 60)) * t) * np.exp(-t * 45)
    else:  # riser
        n = int(1.2 * sr); t = np.linspace(0, 1, n)
        noise = np.convolve(rng.standard_normal(n), np.ones(9) / 9, mode="same")
        y = (noise * 0.6 + 0.4 * np.sin(2 * np.pi * (200 + 900 * t ** 2) * t)) * t ** 2
    return (y / max(1e-9, float(np.max(np.abs(y)))) * 0.6).astype(np.float32)


def plan_sfx_cues(plan):
    """Gemini's cue sheet (source time -> clip time), else a rule: a whoosh at each join between source intervals. Always validated."""
    total = plan.duration
    def to_clip(ts):
        off = 0.0
        for s in plan.segments:
            if s.source_start - 0.01 <= ts <= s.source_end:
                return off + (ts - s.source_start)
            off += s.duration
        return None
    cues = []
    for ts, kind in plan.notes.get("sfx_cues") or []:
        ct = to_clip(ts)
        if ct is not None and kind in SFX_KINDS:
            cues.append((ct, kind, "gemini"))
    if not cues and CFG.get("sfx_local_fallback", True):
        off = 0.0
        for s in plan.segments[:-1]:
            off += s.duration
            cues.append((off - 0.15, "whoosh", "local_rule"))
    out = []
    for c in sorted(x for x in cues if 0.5 <= x[0] <= total - 0.8):
        if not out or c[0] - out[-1][0] >= 2.0:
            out.append(c)
    return out[:int(CFG.get("sfx_max_cues", 5))]


def build_sfx_track(plan, out_path):
    """Write a stereo WAV with the clip's SFX at their cue times; returns the path, or None when there are no cues."""
    cues = plan_sfx_cues(plan)
    if not cues:
        return None
    sr = SFX_SR
    buf = np.zeros(int((plan.duration + 1.0) * sr), dtype=np.float32)
    for i, (t, kind, _who) in enumerate(cues):
        fx = synth_sfx(kind, seed=i + 1)
        start = t - (0.27 if kind == "whoosh" else (len(fx) / sr if kind == "riser" else 0.0))
        s0 = max(0, int(start * sr)); seg = fx[:max(0, len(buf) - s0)]
        buf[s0:s0 + len(seg)] += seg
    buf = np.clip(buf, -0.8, 0.8)
    stereo = np.repeat(buf[:, None], 2, axis=1)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with wave.open(out_path, "wb") as w:
        w.setnchannels(2); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((stereo * 32767).astype("<i2").tobytes())
    return out_path


def render_plan(plan, source, transcript_words, music_path, out_path, work_dir, broll_asset=None, crf=None, audio_ceiling=0.80, sfx_path=None):
    plan.validate(source.duration)
    total = plan.duration
    if total > CFG["max_clip_duration"] + 0.02:
        raise ValueError("Refusing to render a clip longer than 90 seconds")
    os.makedirs(work_dir, exist_ok=True)
    ass_path = os.path.join(work_dir, "captions.ass")
    caption_events = build_ass(plan, transcript_words, ass_path)
    inputs, chains, concat_inputs, framing_reports = [], [], "", []
    for i, seg in enumerate(plan.segments):
        inputs += ["-ss", f"{seg.source_start:.3f}", "-t", f"{seg.duration:.3f}", "-i", source.video_path]
        framing = detect_segment_framing(source, seg, hint=plan.notes.get("gemini_framing"))
        framing_reports.append(framing)
        if framing["mode"] == "fit_blur":
            chains.append(f"[{i}:v]setpts=PTS-STARTPTS,fps={CFG['out_fps']},split=2[bg{i}][fg{i}]")
            chains.append(f"[bg{i}]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,boxblur=22:4[blur{i}]")
            chains.append(f"[fg{i}]scale=1080:1920:force_original_aspect_ratio=decrease[fit{i}]")
            chains.append(f"[blur{i}][fit{i}]overlay=(W-w)/2:(H-h)/2,setsar=1[v{i}]")
        elif framing["mode"] == "split_two":
            w2, hh = framing["crop_w"], source.height
            chains.append(f"[{i}:v]setpts=PTS-STARTPTS,fps={CFG['out_fps']},split=2[sa{i}][sb{i}]")
            chains.append(f"[sa{i}]crop={w2}:{hh}:x={framing['x_left']}:y=0,scale=1080:960:flags=lanczos[ta{i}]")
            chains.append(f"[sb{i}]crop={w2}:{hh}:x={framing['x_right']}:y=0,scale=1080:960:flags=lanczos[tb{i}]")
            chains.append(f"[ta{i}][tb{i}]vstack=inputs=2,setsar=1[v{i}]")
        else:
            cw, ch = crop_geometry(source.width, source.height)
            max_x = max(0, source.width-cw)
            if framing["mode"] == "face_track":
                x_expr = ffmpeg_crop_x_expr(framing["keyframes"], max_x)
            else:
                x_expr = str(int(max_x/2))
            y = max(0, (source.height-ch)//2)
            zoom = ",scale=1166:2074:flags=lanczos,crop=1080:1920" if seg.role == "tight_zoom" else ""
            chains.append(f"[{i}:v]setpts=PTS-STARTPTS,fps={CFG['out_fps']},crop={cw}:{ch}:x='{x_expr}':y={y},scale=1080:1920:flags=lanczos{zoom},setsar=1[v{i}]")
        _fd = min(0.012, seg.duration / 4)     # 12 ms fades remove clicks at every cut
        chains.append(f"[{i}:a]aresample=48000,aformat=channel_layouts=stereo,asetpts=PTS-STARTPTS,afade=t=in:st=0:d={_fd:.3f},afade=t=out:st={max(0.0, seg.duration - _fd):.3f}:d={_fd:.3f}[a{i}]")
        concat_inputs += f"[v{i}][a{i}]"
    chains.insert(0, f"{concat_inputs}concat=n={len(plan.segments)}:v=1:a=1[vbase][abase]")
    music_index = len(plan.segments)
    inputs += ["-i", music_path]
    sfx_index = None
    if sfx_path and os.path.isfile(sfx_path):
        sfx_index = len(plan.segments) + 1
        inputs += ["-i", sfx_path]
    chains.extend(make_audio_mix_graph(music_index, total, volume=0.18, ceiling=audio_ceiling, sfx_input_index=sfx_index))
    video_label = "vbase"
    broll_window = None
    if broll_asset and broll_asset.get("local_path") and os.path.isfile(broll_asset["local_path"]):
        broll_index = len(plan.segments) + 1 + (1 if sfx_index is not None else 0)
        # A single short contextual cutaway, not a random stock montage.
        broll_duration = min(4.0, max(2.5, total * 0.09))
        broll_start = min(max(5.0, total * 0.43), max(0.0, total - broll_duration - 1.0))
        broll_window = (broll_start, broll_start+broll_duration)
        inputs += ["-stream_loop", "-1", "-i", broll_asset["local_path"]]
        chains.append(f"[{broll_index}:v]trim=duration={broll_duration:.3f},setpts=PTS-STARTPTS+{broll_start:.3f}/TB,fps={CFG['out_fps']},scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,setsar=1[br]")
        chains.append(f"[vbase][br]overlay=0:0:enable='between(t\\,{broll_start:.3f}\\,{broll_start+broll_duration:.3f})':eof_action=pass[vbroll]")
        video_label = "vbroll"
    chains.append(f"[{video_label}]ass=captions.ass:fontsdir=/usr/share/fonts/truetype/dejavu,format=yuv420p,fade=t=out:st={max(0.0,total-0.25):.3f}:d=0.25[vout]")
    chains.append("[aout]afade=t=in:st=0:d=0.08,afade=t=out:st=" + f"{max(0.0,total-0.35):.3f}:d=0.35[aout_final]")
    tmp_path = out_path.replace(".mp4", ".rendering.mp4")
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "warning"] + inputs + [
        "-filter_complex", ";".join(chains), "-map", "[vout]", "-map", "[aout_final]",
        "-t", f"{total:.3f}", "-r", str(CFG["out_fps"]),
        "-c:v", "libx264", "-preset", CFG["render_preset"], "-crf", str(crf if crf is not None else CFG["render_crf"]),
        "-profile:v", "high", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", "-ar", "48000",
        "-movflags", "+faststart", tmp_path,
    ]
    run(cmd, cwd=work_dir, timeout=1800)
    if not os.path.isfile(tmp_path) or os.path.getsize(tmp_path) == 0:
        raise RuntimeError("FFmpeg did not produce a nonempty render")
    return {"temporary_path": tmp_path, "caption_events": caption_events,
            "framing": framing_reports, "broll_window": broll_window,
            "actual_duration": total, "music_path": music_path}


def audio_metrics(path):
    # Peak is measured per channel at the native rate (never on a mono downmix, which adds +3 dB for identical channels).
    proc = subprocess.run(["ffmpeg", "-v", "error", "-i", path, "-vn", "-ac", "2", "-ar", "48000", "-f", "f32le", "pipe:1"],
                          capture_output=True, text=False, timeout=180)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.decode("utf-8", "replace")[-1000:])
    stereo = np.frombuffer(proc.stdout, dtype=np.float32)
    stereo = stereo[: (stereo.size // 2) * 2].reshape(-1, 2)
    if stereo.size == 0:
        return {"rms": 0.0, "peak": 0.0}
    mono = stereo.mean(axis=1)
    return {"rms": float(np.sqrt(np.mean(mono * mono))), "peak": float(np.max(np.abs(stereo)))}


def measure_loudness(path):
    """Measure integrated loudness and true peak (dBTP) on the finished AAC master."""
    proc = subprocess.run(
        ["ffmpeg", "-hide_banner", "-nostats", "-i", path, "-vn",
         "-af", "loudnorm=I=-16:TP=-1.5:LRA=10:print_format=json", "-f", "null", "-"],
        capture_output=True, text=True, timeout=180,
    )
    if proc.returncode != 0:
        raise RuntimeError((proc.stderr or "FFmpeg loudness measurement failed")[-1200:])
    reports = re.findall(r"\{\s*\"input_i\".*?\}", proc.stderr or "", flags=re.S)
    if not reports:
        raise RuntimeError("FFmpeg did not return loudnorm measurement JSON")
    raw = json.loads(reports[-1])
    return {
        "integrated_lufs": float(raw["input_i"]),
        "true_peak_dbtp": float(raw["input_tp"]),
        "loudness_range_lu": float(raw.get("input_lra", "nan")),
        "normalization_type": raw.get("normalization_type", "unknown"),
    }


def run_technical_qa(path, plan, caption_events):
    checks = []
    def add(name, passed, detail):
        checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})
    if not os.path.isfile(path) or os.path.getsize(path) <= 0:
        add("file_exists_nonzero", False, "missing or empty")
        return checks
    add("file_exists_nonzero", True, f"{os.path.getsize(path)/1e6:.1f} MB")
    info = ffprobe_json(path)
    video = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), None)
    audio = next((s for s in info.get("streams", []) if s.get("codec_type") == "audio"), None)
    duration = float(info["format"]["duration"])
    add("duration_under_90s", 0 < duration <= CFG["max_clip_duration"] + 0.05, f"{duration:.3f}s")
    add("duration_matches_timeline", abs(duration-plan.duration) <= 0.15, f"render={duration:.3f}s timeline={plan.duration:.3f}s")
    if video:
        add("resolution_1080x1920", (int(video["width"]), int(video["height"])) == (1080, 1920), f"{video['width']}x{video['height']}")
        rate = video.get("r_frame_rate", "0/1").split("/")
        fps = float(rate[0]) / max(float(rate[1]), 1.0)
        add("fps_30", abs(fps-30) < 0.02, f"{fps:.3f}")
        add("codec_h264", video.get("codec_name") == "h264", video.get("codec_name"))
    else:
        add("video_stream", False, "missing")
    if audio:
        add("codec_aac", audio.get("codec_name") == "aac", audio.get("codec_name"))
        if video and video.get("duration") and audio.get("duration"):
            av_delta = abs(float(video["duration"]) - float(audio["duration"]))
            add("audio_video_sync", av_delta <= 0.15, f"duration delta={av_delta:.3f}s")
    else:
        add("audio_stream", False, "missing")
    cap = cv2.VideoCapture(path)
    opened = cap.isOpened()
    add("opencv_decode", opened, "opened" if opened else "cannot decode")
    black_samples = 0
    if opened:
        for t in np.linspace(min(0.5, duration/4), max(0.5, duration-0.5), 8):
            cap.set(cv2.CAP_PROP_POS_MSEC, float(t)*1000)
            ok, frame = cap.read()
            if not ok:
                black_samples += 1
            else:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                if gray.mean() < 5.0 and gray.std() < 2.0:
                    black_samples += 1
        cap.release()
    add("no_black_or_unreadable_samples", black_samples == 0, f"{black_samples}/8 failed samples")
    add("caption_events_present", len(caption_events) > 0, f"{len(caption_events)} timed caption events")
    caption_hits = 0
    if opened and caption_events:
        for event in caption_events[:min(5, len(caption_events))]:
            cap = cv2.VideoCapture(path)
            cap.set(cv2.CAP_PROP_POS_MSEC, (event["start"] + min(0.12, (event["end"]-event["start"])/2)) * 1000)
            ok, frame = cap.read()
            cap.release()
            if ok:
                band = frame[int(frame.shape[0]*0.62):int(frame.shape[0]*0.88), :, :]
                b, g, r = band[:,:,0], band[:,:,1], band[:,:,2]
                yellow = ((r > 205) & (g > 125) & (g < 245) & (b < 100)).sum()
                if yellow > 35:
                    caption_hits += 1
    add("caption_pixels_visible", caption_hits >= (1 if caption_events else 0), f"{caption_hits}/{min(5, len(caption_events))} sampled caption frames")
    try:
        metrics = audio_metrics(path)
        add("audio_not_silent", metrics["rms"] > 0.003, f"RMS={metrics['rms']:.4f}")
        add("audio_peak_below_clipping", metrics["peak"] < 0.999, f"sample peak={metrics['peak']:.4f}")
        loudness = measure_loudness(path)
        add("integrated_loudness_measured", math.isfinite(loudness["integrated_lufs"]),
            f"{loudness['integrated_lufs']:.2f} LUFS")
        add("true_peak_below_minus_1dbtp", loudness["true_peak_dbtp"] <= -1.0,
            f"{loudness['true_peak_dbtp']:.2f} dBTP")
    except Exception as exc:
        add("audio_loudness_metrics", False, f"{type(exc).__name__}: {exc}")
    return checks


DANGLING_OPENERS = {"this", "that", "it", "they", "them", "those", "these", "he", "she", "so", "and", "but",
                    "because", "also", "which", "then", "its", "their"}


def _qa_tokens(text):
    return re.findall(r"[a-z0-9']+", (text or "").lower())


def run_editorial_qa(plan, source, transcript=None):
    """Editorial gate. With `transcript`, quotation integrity and sentence-boundary checks are verified against
    the real transcript; warnings (severity="warning") never block but mark what a human must review."""
    plan.validate(source.duration)
    checks = []
    checks.append({"check": "single_central_idea", "passed": len(set(plan.notes.get("topics", []))) <= 4,
                   "detail": ", ".join(plan.notes.get("topics", [])) or "topic not classified"})
    if transcript is not None and transcript.words:
        starts = [w.start for w in transcript.words]
        worst, worst_seg = 1.0, None
        for i, seg in enumerate(plan.segments):
            lo = bisect.bisect_left(starts, seg.source_start - 0.5)
            hi = bisect.bisect_right(starts, seg.source_end + 0.5)
            rebuilt = [w.word for w in transcript.words[lo:hi]
                       if seg.source_start <= (w.start + w.end) / 2 < seg.source_end]
            a_tok, b_tok = set(_qa_tokens(" ".join(rebuilt))), set(_qa_tokens(seg.transcript))
            ratio = len(a_tok & b_tok) / max(1, len(a_tok | b_tok))
            if ratio < worst:
                worst, worst_seg = ratio, i
        checks.append({"check": "source_quotation_integrity", "passed": worst >= 0.90,
                       "detail": f"segment text matches the transcript words in its interval (worst overlap {worst:.2f}"
                                 + (f" at segment {worst_seg})" if worst_seg is not None else ")")})
        s_starts = sorted(s.start for s in transcript.sentences)
        s_ends = sorted(s.end for s in transcript.sentences)
        bad = []
        for i, seg in enumerate(plan.segments):
            d0 = min((abs(seg.source_start - x) for x in s_starts), default=99.0)
            d1 = min((abs(seg.source_end - x) for x in s_ends), default=99.0)
            # cuts made by tightening sit inside a sentence by design (in silence/filler); only the clip's real start/end must be on sentence edges
            if seg.role in ("tight_cont", "tight_zoom"):
                d0 = 0.0
            if i + 1 < len(plan.segments) and plan.segments[i + 1].role in ("tight_cont", "tight_zoom"):
                d1 = 0.0
            if d0 > 0.25 or d1 > 0.25:
                bad.append(f"seg{i}(start off {d0:.2f}s, end off {d1:.2f}s)")
        checks.append({"check": "segments_on_sentence_boundaries", "passed": not bad,
                       "detail": "every cut starts and ends on a sentence boundary" if not bad else "mid-sentence cut: " + ", ".join(bad)})
    else:
        checks.append({"check": "source_quotation_integrity", "passed": all(bool(s.transcript.strip()) for s in plan.segments),
                       "detail": "NOT verified against the transcript (no transcript supplied); only checks non-empty text"})
    checks.append({"check": "natural_duration", "passed": 8 <= plan.duration <= 90,
                   "detail": f"{plan.duration:.1f}s actual assembled content; no cloned-frame padding"})
    checks.append({"check": "five_video_diversity_metadata", "passed": bool(plan.notes.get("editorial_score")),
                   "detail": f"editorial heuristic score={plan.notes.get('editorial_score')}"})
    reasons = []
    order = [round(s.source_start) for s in plan.segments]
    if any(order[i] < order[i-1] for i in range(1, len(order))):
        reasons.append("clip opens with a later passage than some of the material that follows (source order: "
                       + " -> ".join(f"{x}s" for x in order) + ")")
    reasons += list(plan.notes.get("critic_flags", []))
    first = _qa_tokens(plan.segments[0].transcript)[:1]
    if first and first[0] in DANGLING_OPENERS:
        reasons.append(f"opening word '{first[0]}' may refer to something said earlier in the source")
    checks.append({"check": "context_review_flagged", "passed": True, "severity": "warning", "flagged": bool(reasons),
                   "detail": ("; ".join(reasons) + ". Review the preview before publishing.") if reasons else
                             "no automatic context warning; automatic checks cannot prove a joined story is fair, so still preview it"})
    return checks


def make_cover(video_path, title, out_path):
    frame_path = out_path + ".frame.jpg"
    run(["ffmpeg", "-y", "-ss", "1.0", "-i", video_path, "-frames:v", "1", "-q:v", "2", frame_path], quiet=True)
    image = Image.open(frame_path).convert("RGB").resize((1080, 1920))
    image = ImageEnhance.Brightness(image).enhance(0.72)
    overlay = Image.new("RGBA", image.size, (0,0,0,0))
    draw = ImageDraw.Draw(overlay)
    draw.rectangle((0, 0, 1080, 1920), fill=(0,0,0,36))
    image = Image.alpha_composite(image.convert("RGBA"), overlay)
    draw = ImageDraw.Draw(image)
    font_path = find_font_path()
    font = ImageFont.truetype(font_path, 84)
    words = title.split()
    lines, line = [], ""
    for word in words:
        attempt = (line + " " + word).strip()
        if draw.textbbox((0,0), attempt, font=font)[2] > 920 and line:
            lines.append(line); line = word
        else:
            line = attempt
    if line: lines.append(line)
    lines = lines[:4]
    y = 220
    draw.rounded_rectangle((56, 140, 1024, 740), radius=44, fill=(0,0,0,150), outline=(255,255,255,100), width=4)
    for line in lines:
        draw.text((104, y), line, font=font, fill=(255,255,255), stroke_width=4, stroke_fill=(0,0,0,220))
        y += 124
    image.convert("RGB").save(out_path, quality=92)
    try: os.remove(frame_path)
    except OSError: pass
    return out_path

log("render", "dynamic framing and complete render/QA functions defined")

In [ ]:
# CELL 9b — Finishing studio: completeness gate, Gemini critic, filler/pause tightening
# Why: the common failures of AI clippers are clips that end mid-thought, clips that need earlier context, and dead air / filler words.
# Vizard describes a dedicated "find the natural ending, then check completeness" step; Descript-style tools remove filler and silence.
# This cell does both at zero cost: deterministic local rules always run; Gemini (one extra request for all five clips) acts as an editor
# whose instructions are executed ONLY as bounded sentence-count fixes that the code validates.
FILLER_WORDS = {"um", "uh", "umm", "uhm", "uhh", "er", "erm", "hmm", "mm", "ah"}
INCOMPLETE_ENDINGS = {"and", "but", "because", "which", "that", "to", "of", "the", "a", "an", "or", "if", "when", "with", "for", "as",
                      "than", "then", "is", "are", "was", "were", "in", "on", "at", "my", "your", "our", "their", "this", "these"}
START_CONNECTORS = {"and", "but", "so", "because", "which", "or", "plus", "also", "then", "anyway", "cause"}


def sentence_complete(text):
    """Whisper punctuates finished sentences; a sentence with no terminal punctuation is treated as possibly cut off."""
    t = str(text or "").strip()
    return bool(t) and bool(re.search(r"[.!?][\"')\]]?$", t))


def _first_token(text):
    m = re.findall(r"[A-Za-z']+", str(text or "").lower())
    return m[0] if m else ""


def _sentence_range(transcript, start, end):
    idx = [i for i, s in enumerate(transcript.sentences) if s.end > start + 0.05 and s.start < end - 0.05]
    return (idx[0], idx[-1]) if idx else None


def _set_bounds(seg, transcript, si, ei, source_duration):
    s = transcript.sentences
    seg.source_start = round(max(0.0, s[si].start - 0.05), 2)
    seg.source_end = round(min(source_duration, s[ei].end + 0.10), 2)
    seg.transcript = " ".join(x.text for x in s[si:ei + 1]).strip()


def complete_plan_boundaries(plan, transcript, source, max_len=None):
    """Local completeness gate: never end on an unfinished sentence, never open on a dangling connector. Returns a list of log strings."""
    max_len = float(max_len or CFG["max_clip_duration"])
    log_, sents = [], transcript.sentences
    if not sents:
        return log_
    last = plan.segments[-1]
    r = _sentence_range(transcript, last.source_start, last.source_end)
    if r:
        si, ei = r
        changed = False
        for _ in range(2):
            if sentence_complete(sents[ei].text) or ei + 1 >= len(sents):
                break
            nxt = sents[ei + 1]
            if nxt.start - sents[ei].end > 2.5 or (plan.duration - last.duration + (nxt.end + 0.10 - last.source_start)) > max_len:
                break
            ei += 1; changed = True
        if changed:
            _set_bounds(last, transcript, si, ei, source.duration); log_.append("ended on an unfinished sentence; extended to complete it")
        elif not sentence_complete(sents[ei].text):
            log_.append("ending may be unfinished (could not extend within limits)")
    first = plan.segments[0]
    r = _sentence_range(transcript, first.source_start, first.source_end)
    if r:
        si, ei = r
        trimmed = False
        while si < ei and _first_token(sents[si].text) in START_CONNECTORS and (plan.duration - (sents[si + 1].start - first.source_start)) >= 20.0:
            si += 1; trimmed = True
        if trimmed:
            _set_bounds(first, transcript, si, ei, source.duration); log_.append("opened on a dangling connector; started one sentence later")
        elif _first_token(sents[si].text) in START_CONNECTORS:
            log_.append("opens on a connector word (may need earlier context)")
    return log_


CRITIC_SCHEMA = {"type": "object", "properties": {"verdicts": {"type": "array", "items": {"type": "object", "properties": {
    "index": {"type": "integer", "minimum": 0},
    "complete_thought": {"type": "boolean"}, "needs_prior_context": {"type": "boolean"}, "ends_on_payoff": {"type": "boolean"},
    "issue": {"type": "string"},
    "extend_end_sentences": {"type": "integer", "minimum": 0, "maximum": 2},
    "trim_start_sentences": {"type": "integer", "minimum": 0, "maximum": 2}},
    "required": ["index", "complete_thought", "needs_prior_context", "ends_on_payoff", "extend_end_sentences", "trim_start_sentences"]}}},
    "required": ["verdicts"]}


def _plan_text(plan, limit=2300):
    return " [CUT] ".join(s.transcript.strip() for s in plan.segments)[:limit]


def _following_text(plan, transcript, n=2):
    r = _sentence_range(transcript, plan.segments[-1].source_start, plan.segments[-1].source_end)
    if not r:
        return ""
    return " ".join(x.text for x in transcript.sentences[r[1] + 1:r[1] + 1 + n])


def gemini_critic(plans, transcript):
    """One request reviews all clips like a strict editor. Returns {index: verdict} or {} when Gemini is unavailable."""
    if not (GEMINI.enabled and CFG.get("enable_gemini_critic", True)):
        return {}
    blocks = []
    for i, p in enumerate(plans):
        blocks.append(f"CLIP {i} — title: {p.title}\nTEXT: {_plan_text(p)}\nNEXT SENTENCES AFTER THE CLIP (not included): {_following_text(p, transcript) or '(none)'}")
    prompt = ("You are a strict short-form editor. For each clip decide: (1) complete_thought: does it end after a finished thought or payoff, "
              "not mid-idea? (2) needs_prior_context: would a cold viewer be lost because it refers to something said earlier? "
              "(3) ends_on_payoff: does the last line land? If the clip ends too early, set extend_end_sentences (0-2) using the NEXT SENTENCES. "
              "If the first sentence is a dependent fragment, set trim_start_sentences (0-2). Judge only from the text. Be conservative: "
              "use 0 unless the fix is clearly needed. Return one verdict per clip index.\n\n" + "\n\n".join(blocks))
    stage_set("gemini_critic", "running", "gemini")
    res = gemini_structured("critic_" + hashlib.sha1(prompt.encode()).hexdigest()[:10], prompt, CRITIC_SCHEMA, tier="reasoning",
                            validator=lambda d: [] if all(0 <= v["index"] < len(plans) for v in d["verdicts"]) else ["verdict index out of range"])
    if not res["ok"]:
        stage_set("gemini_critic", "completed_using_fallback", "local", f"{res['error_kind']}")
        return {}
    stage_set("gemini_critic", "completed", "gemini", f"{len(res['data']['verdicts'])} verdicts")
    return {v["index"]: v for v in res["data"]["verdicts"]}


def apply_critic(plan, verdict, transcript, source, max_len=None):
    """Bounded, validated execution of the critic's advice. Returns log strings."""
    max_len = float(max_len or CFG["max_clip_duration"])
    notes, sents, out = [], transcript.sentences, []
    flags = plan.notes.setdefault("critic_flags", [])
    last = plan.segments[-1]
    if not verdict.get("complete_thought", True) and int(verdict.get("extend_end_sentences", 0)) > 0:
        r = _sentence_range(transcript, last.source_start, last.source_end)
        if r:
            si, ei = r
            for _ in range(min(2, int(verdict["extend_end_sentences"]))):
                if ei + 1 >= len(sents) or sents[ei + 1].start - sents[ei].end > 2.5:
                    break
                if plan.duration - last.duration + (sents[ei + 1].end + 0.10 - last.source_start) > max_len:
                    break
                ei += 1
            if ei != r[1]:
                _set_bounds(last, transcript, si, ei, source.duration); out.append("critic: extended ending to finish the thought")
    first = plan.segments[0]
    if verdict.get("needs_prior_context") and int(verdict.get("trim_start_sentences", 0)) > 0:
        r = _sentence_range(transcript, first.source_start, first.source_end)
        if r:
            si, ei = r
            k = min(2, int(verdict["trim_start_sentences"]))
            while k > 0 and si < ei and (plan.duration - (sents[si + 1].start - first.source_start)) >= 20.0:
                si += 1; k -= 1
            if si != r[0]:
                _set_bounds(first, transcript, si, ei, source.duration); out.append("critic: removed a context-dependent opening sentence")
    if verdict.get("needs_prior_context") and not out:
        flags.append("Gemini critic: a cold viewer may need earlier context" + (f" ({verdict['issue'][:100]})" if verdict.get("issue") else ""))
    if not verdict.get("ends_on_payoff", True):
        flags.append("Gemini critic: the ending may not land on a payoff")
    plan.notes["critic"] = {k: verdict.get(k) for k in ("complete_thought", "needs_prior_context", "ends_on_payoff", "issue")}
    return out


def tighten_plan(plan, transcript, source):
    """Remove filler words and long pauses by splitting segments into kept pieces. Cuts are made only in silence/filler
    (never inside a spoken word), keep at least ~0.24 s of natural pause, leave the first/last 0.4 s of each segment alone, and
    remove at most `tighten_max_fraction` of a segment. Adds 12 ms audio fades at every cut (render_plan) and alternates a mild
    punch-in on continuation pieces to hide the jump cut."""
    stats = {"removed_sec": 0.0, "fillers": 0, "pauses": 0, "pieces_added": 0}
    if not CFG.get("enable_tighten", True) or not transcript.words:
        return stats
    thr, max_frac = float(CFG.get("tighten_pause_sec", 0.60)), float(CFG.get("tighten_max_fraction", 0.20))
    starts = [w.start for w in transcript.words]
    new_segments = []
    for seg in plan.segments:
        lo, hi = bisect.bisect_left(starts, seg.source_start - 0.5), bisect.bisect_right(starts, seg.source_end + 0.5)
        ws = [w for w in transcript.words[lo:hi] if seg.source_start <= (w.start + w.end) / 2 < seg.source_end]
        cuts = []
        for k, w in enumerate(ws):
            nxt = ws[k + 1] if k + 1 < len(ws) else None
            if CFG.get("tighten_fillers", True) and re.sub(r"[^a-z]", "", w.word.lower()) in FILLER_WORDS:
                prev_end = ws[k - 1].end if k > 0 else seg.source_start
                a, b = prev_end + 0.06, (nxt.start if nxt else seg.source_end) - 0.06
                if b - a >= 0.18:
                    cuts.append([a, b, "filler"])
                continue
            if nxt is not None and nxt.start - w.end > thr:
                cuts.append([w.end + 0.12, nxt.start - 0.12, "pause"])
        guard_a, guard_b = seg.source_start + 0.4, seg.source_end - 0.4
        cuts = [[max(a, guard_a), min(b, guard_b), kind] for a, b, kind in cuts if min(b, guard_b) - max(a, guard_a) >= 0.18]
        merged = []
        for c in sorted(cuts):
            if merged and c[0] <= merged[-1][1] + 0.01:
                merged[-1][1] = max(merged[-1][1], c[1]); merged[-1][2] = "filler" if "filler" in (merged[-1][2], c[2]) else "pause"
            else:
                merged.append(c)
        # budget: drop the smallest cuts first until within the fraction
        while merged and sum(b - a for a, b, _ in merged) > max_frac * seg.duration:
            merged.remove(min(merged, key=lambda c: c[1] - c[0]))
        # every kept piece must stay >= 0.8 s
        kept, cursor = [], seg.source_start
        for a, b, kind in merged:
            if a - cursor >= 0.8 and (seg.source_end - b) >= 0.8:
                kept.append((a, b, kind)); cursor = b
        if not kept:
            new_segments.append(seg); continue
        pieces, cursor = [], seg.source_start
        for a, b, kind in kept:
            pieces.append((cursor, a)); cursor = b
            stats["fillers" if kind == "filler" else "pauses"] += 1
        pieces.append((cursor, seg.source_end))
        for n, (a, b) in enumerate(pieces):
            text = " ".join(w.word for w in ws if a <= (w.start + w.end) / 2 < b).strip()
            if n == 0:
                role = seg.role
            else:
                role = "tight_zoom" if (n % 2 == 1 and CFG.get("punch_in_zoom", True)) else "tight_cont"
            new_segments.append(Segment(source_start=round(a, 3), source_end=round(b, 3), purpose=seg.purpose, transcript=text,
                                        crop_fx=seg.crop_fx, role=role, confidence=seg.confidence))
        stats["removed_sec"] += sum(b - a for a, b, _ in kept)
        stats["pieces_added"] += len(pieces) - 1
    if stats["pieces_added"]:
        plan.segments = new_segments
    stats["removed_sec"] = round(stats["removed_sec"], 2)
    return stats


def finish_plans(plans, transcript, source):
    """Run once before rendering. Idempotent (plans carry notes['finished'])."""
    todo = [p for p in plans if not p.notes.get("finished")]
    if not todo:
        return []
    for p in todo:
        p.notes["finish_log"] = []
        if CFG.get("enable_completeness_gate", True):
            p.notes["finish_log"] += complete_plan_boundaries(p, transcript, source)
    verdicts = gemini_critic(todo, transcript)
    for i, p in enumerate(todo):
        if i in verdicts:
            p.notes["finish_log"] += apply_critic(p, verdicts[i], transcript, source)
        p.notes["tighten"] = tighten_plan(p, transcript, source)
        t = p.notes["tighten"]
        if t["pieces_added"]:
            p.notes["finish_log"].append(f"tightened: removed {t['removed_sec']}s ({t['fillers']} filler, {t['pauses']} pause cuts)")
        p.validate(source.duration)
        p.notes["finished"] = True
    for i, p in enumerate(todo, 1):
        for line in p.notes["finish_log"]:
            log("finish", f"clip {i}: {line}")
    return [p.notes["finish_log"] for p in todo]


In [ ]:
# CELL 10 — Publishing metadata, editorial research playbook, source register, contact sheet, and staged delivery


def hashtags_for_plan(plan):
    tags = {"#Shorts", "#Reels", "#LearnSomething"}
    topic_tags = {
        "money": "#Money", "business": "#Business", "entrepreneurship": "#Entrepreneurship",
        "self-improvement": "#SelfImprovement", "psychology": "#Psychology", "leadership": "#Leadership",
        "technology": "#Technology", "relationships": "#Relationships",
    }
    for topic in plan.notes.get("topics", []):
        if topic in topic_tags:
            tags.add(topic_tags[topic])
    return list(sorted(tags))[:7]


def asset_credit_records(clip_index):
    """Return attribution lines required by the exact assets used in one finished clip."""
    credits = []
    for asset_type in ("music", "broll"):
        for asset in ASSET_MANIFEST.get(asset_type, []):
            if int(asset.get("clip_index") or 0) != int(clip_index):
                continue
            if not asset.get("attribution_required"):
                continue
            label = asset.get("attribution_text") or asset.get("title") or asset_type
            link = asset.get("creator_url") or asset.get("source_url") or asset.get("license_url")
            credits.append({"asset_type": asset_type, "text": label, "url": link,
                            "creator_url": asset.get("creator_url"), "source_url": asset.get("source_url"),
                            "provider": asset.get("provider"), "license": asset.get("license")})
    return credits


def format_asset_credits(credits):
    lines = []
    for row in credits:
        links = []
        if row.get("creator_url"):
            links.append("Creator: " + row["creator_url"])
        source_url = row.get("source_url") or (row.get("url") if not row.get("creator_url") else None)
        if source_url:
            links.append("Coverr/source: " + source_url)
        elif row.get("url"):
            links.append(row["url"])
        lines.append(row["text"] + ((" — " + " | ".join(links)) if links else ""))
    return "\n".join(lines)


def create_publishing_metadata(plans):
    if "build_publishing_packs" in globals():       # CELL 10b: platform-native packs (Gemini + validators, local fallback)
        return build_publishing_packs(plans)
    return legacy_publishing_metadata(plans)


def legacy_publishing_metadata(plans):
    youtube, tiktok, instagram = {}, {}, {}
    for idx, plan in enumerate(plans, 1):
        title = plan.title.strip().rstrip(".!? ")[:92]
        hashtags = hashtags_for_plan(plan)
        transcript_excerpt = " ".join(s.transcript for s in plan.segments)
        prompt = "What part of this idea do you agree or disagree with, and why?"
        credits = asset_credit_records(idx)
        credit_block = format_asset_credits(credits)
        credit_suffix = ("\n\nAsset credits:\n" + credit_block) if credit_block else ""
        youtube[str(idx)] = {
            "title": title,
            "description": (f"{title}.\n\nA short extract from an authorized long-form source." + credit_suffix + "\n\n" + " ".join(hashtags)),
            "hashtags": hashtags, "pinned_comment": prompt,
            "asset_credits": credits,
            "music_note": "Embedded music is the cleared track listed in asset_license_manifest.json; do not replace it with unlicensed audio.",
        }
        tiktok[str(idx)] = {
            "caption": f"{title}. {prompt}" + (f"\n\n{credit_block}" if credit_block else "") + "\n" + " ".join(hashtags[:5]),
            "hashtags": hashtags[:5], "discussion_prompt": prompt, "asset_credits": credits,
        }
        instagram[str(idx)] = {
            "caption": f"{title}.\n\n{prompt}" + (f"\n\n{credit_block}" if credit_block else "") + "\n\n" + " ".join(hashtags),
            "hashtags": hashtags, "cover_text": title[:58], "asset_credits": credits,
        }
    return {"youtube": youtube, "tiktok": tiktok, "instagram": instagram}


EDITORIAL_PLAYBOOK = r"""# PROJECT ZIROQ — Editorial Playbook

## Operating principle
The selection score is a transparent editorial heuristic, not a model of actual audience retention or a promise of virality. The editor optimizes for a clear idea, source-faithful context, specificity, narrative progression, useful payoff, and distinctness across the five cuts.

## Curiosity and information gaps
Use a real question, contrast, unexpected but supportable statement, concrete detail, or unresolved problem as an opening. The following material must answer the question or deliver the promised explanation. Never invent a mystery, statistic, quotation, or payoff. This translates the information-gap account of curiosity into an editorial rule: open a gap only when the source contains a real answer.

## Narrative transportation
Look for a person, situation, stakes, conflict, turning point, and consequence. A personal example can be stronger than an abstract claim when the source contains enough context. Reordered excerpts must not imply that separate events occurred in a false chronology. Green and Brock describe transportation in terms of imagery, affect, and attentional focus; the practical edit should preserve those elements rather than merely increasing cut frequency.

## Memorability and shareability
Prefer concrete details, specific examples, clear contrasts, and a single understandable point. Avoid generic motivational filler. A useful clip should leave one sentence, question, or practical action that a viewer could accurately repeat to someone else. Shareability is a creative hypothesis, not an analytics result.

## Story shapes
Choose the structure supported by the source: hook → context → development → payoff; problem → mistake → consequence → solution; story → conflict → turning point → lesson; question → answer → explanation; or claim → evidence → qualification → conclusion. Do not force every source into one template.

## Editing and pacing
Remove silence and repetition only where meaning survives. Let a meaningful pause remain when it contributes to emotion or emphasis. Use a crop change, B-roll cutaway, diagram, statistic, or sound accent only when it clarifies or renews attention. Captions should group natural phrases and emphasize meaningful words; do not animate every word with arbitrary colour changes.

## Audio policy
Speech remains dominant. Music is ducked under speech and faded at the edges. The default fallback is locally synthesized original instrumental audio. An external track is accepted only when item-level metadata exposes a license on the explicit allowlist (CC0 or CC BY 3.0/4.0). Public Domain Mark is a rights-status mark, not a copyright licence, so it is not treated as a licence grant. CC BY-NC, CC BY-ND, CC BY-SA, unknown licenses, and unclear synchronization rights are excluded. Keep each license manifest with the master export.

## Visual truth
B-roll must illustrate the topic, not pretend that a stock actor participated in the speaker's event. If no suitable licensed asset is available, retain the speaker footage and use only extractive title/caption graphics. Do not fabricate charts, statistics, people, quotes, or archival evidence.

## Final human review
Automatic checks can measure codec, duration, frame samples, audio level, and timestamp mapping. They cannot conclusively judge whether a reordered story is fair, whether a joke lands, whether a facial crop follows the active speaker, or whether a particular example needs more context. Preview all five clips before publication.
"""

SOURCE_REGISTER = [
    {"title": "Loewenstein (1994), The Psychology of Curiosity", "url": "https://www.cmu.edu/dietrich/sds/docs/loewenstein/PsychofCuriosity.pdf", "use": "Information-gap framing; original source to consult"},
    {"title": "Green & Brock (2000), The Role of Transportation in the Persuasiveness of Public Narratives", "url": "https://doi.org/10.1037/0022-3514.79.5.701", "use": "Narrative transportation: imagery, affect, attention"},
    {"title": "YouTube audience retention help", "url": "https://support.google.com/youtube/answer/9314415", "use": "Use actual retention analytics after publication; do not treat editorial scores as audience data"},
    {"title": "YouTube Shorts guidance", "url": "https://support.google.com/youtube/answer/15424877", "use": "Platform-native publishing guidance"},
    {"title": "TikTok creator tips", "url": "https://newsroom.tiktok.com/5-tips-for-tiktok-creators", "use": "Platform creative guidance; review current guidance at publishing time"},
    {"title": "Mixkit Stock Music Free License", "url": "https://mixkit.co/license/modal/musicFree/", "use": "Manual import only; no scraping or automated catalogue downloading"},
    {"title": "Mixkit User Terms", "url": "https://mixkit.co/terms/", "use": "Restrictions include mass downloading/scraping and building a competing service"},
    {"title": "Pixabay API documentation", "url": "https://pixabay.com/api/docs/", "use": "Official video search endpoint, API request limits, 24-hour caching, and no systematic mass downloads"},
    {"title": "Pixabay Content License", "url": "https://pixabay.com/service/license-summary/", "use": "Use/adaptation permission and prohibited-use checks"},
    {"title": "Coverr API documentation", "url": "https://api.coverr.co/docs/start/", "use": "Free API tier, authentication, audio/video search, and download tracking"},
    {"title": "Coverr Content License", "url": "https://coverr.co/license/", "use": "Commercial use, attribution, premium/sponsored media, and AI-training restrictions"},
    {"title": "Internet Archive Metadata API", "url": "https://internetarchive.readthedocs.io/en/stable/metadata.html", "use": "Verify item-level licenseurl and available files"},
    {"title": "Internet Archive search guide", "url": "https://archivesupport.zendesk.com/hc/en-us/articles/360018359991-Search-A-Basic-Guide", "use": "Fielded queries and Boolean search operators"},
    {"title": "Sentence Transformers all-MiniLM-L6-v2 model card", "url": "https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2", "use": "Optional local embeddings; model card identifies Apache-2.0 license"},
    {"title": "Google Colab FAQ and resource limits", "url": "https://research.google.com/colaboratory/faq.html", "use": "Free GPU, VM lifetime, and usage are dynamic/not guaranteed"},
]


def create_contact_sheet(video_paths, out_path):
    thumbs = []
    for idx, path in enumerate(video_paths, 1):
        frame_path = os.path.join(RENDER_DIR, f"contact_{idx}.jpg")
        info = ffprobe_json(path)
        dur = float(info["format"]["duration"])
        run(["ffmpeg", "-y", "-ss", f"{min(dur*0.45, 20):.2f}", "-i", path, "-frames:v", "1", "-q:v", "3", frame_path], quiet=True)
        im = Image.open(frame_path).convert("RGB")
        im.thumbnail((240, 426))
        canvas = Image.new("RGB", (260, 470), "black")
        canvas.paste(im, ((260-im.width)//2, 12))
        draw = ImageDraw.Draw(canvas)
        draw.text((10, 444), f"CLIP {idx:02d}  |  {dur:.1f}s", fill="white")
        thumbs.append(canvas)
    sheet = Image.new("RGB", (260*len(thumbs), 470), "#151515")
    for i, thumb in enumerate(thumbs):
        sheet.paste(thumb, (260*i, 0))
    sheet.save(out_path, quality=92)
    return out_path


def safe_copy_verified(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    temp = dst + ".uploading"
    shutil.copy2(src, temp)
    if os.path.getsize(temp) != os.path.getsize(src) or sha256_file(temp) != sha256_file(src):
        try: os.remove(temp)
        except OSError: pass
        raise RuntimeError(f"Upload copy integrity mismatch: {os.path.basename(src)}")
    os.replace(temp, dst)
    return dst


def write_research_assets():
    with open(os.path.join(REPORT_DIR, "editorial_playbook.md"), "w", encoding="utf-8") as f:
        f.write(EDITORIAL_PLAYBOOK)
    atomic_write_json(os.path.join(REPORT_DIR, "source_register.json"), SOURCE_REGISTER)


def write_performance_template(path):
    header = ["run_id", "clip_index", "platform", "post_url", "views", "average_view_duration_sec",
              "completion_rate", "shares", "saves", "comments", "likes", "collected_utc"]
    if not os.path.isfile(path):
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(header)


def enforce_five_output_count(plans, clip_results):
    if len(plans) != CFG["required_clip_count"] or len(clip_results) != CFG["required_clip_count"]:
        raise RuntimeError(f"Delivery blocked: expected exactly {CFG['required_clip_count']} plans and completed results; got {len(plans)} and {len(clip_results)}")
    return True


def build_and_deliver_package(plans, clip_results):
    enforce_five_output_count(plans, clip_results)
    if not all(x["technical_pass"] and x["editorial_pass"] for x in clip_results):
        raise RuntimeError("Delivery gate blocked: one or more clips failed QA")

    metadata = create_publishing_metadata(plans)
    atomic_write_json(os.path.join(REPORT_DIR, "youtube_metadata.json"), metadata["youtube"])
    atomic_write_json(os.path.join(REPORT_DIR, "tiktok_metadata.json"), metadata["tiktok"])
    atomic_write_json(os.path.join(REPORT_DIR, "instagram_metadata.json"), metadata["instagram"])
    atomic_write_json(os.path.join(REPORT_DIR, "clip_report.json"), {
        "run_id": RUN_ID, "source": asdict(SOURCE), "language": TRANSCRIPT.language,
        "similarity_backend": SIMILARITY_BACKEND, "clips": clip_results,
        "disclaimer": "Editorial heuristic scores are not measured audience retention or virality predictions.",
    })
    atomic_write_json(os.path.join(REPORT_DIR, "quality_report.json"), {
        "run_id": RUN_ID,
        "summary": {
            "expected_clips": CFG["required_clip_count"],
            "verified_clips": sum(1 for item in clip_results if item["technical_pass"] and item["editorial_pass"]),
            "all_technical_checks_passed": all(item["technical_pass"] for item in clip_results),
            "all_editorial_checks_passed": all(item["editorial_pass"] for item in clip_results),
        },
        "clips": [{
            "index": item["index"], "title": item["title"], "duration": item["duration"],
            "technical_checks": item["technical_checks"], "editorial_checks": item["editorial_checks"],
            "human_review_required": item.get("human_review_required", True),
        } for item in clip_results],
        "limitations": [
            "Automated checks cannot prove narrative fairness, factual truth, or whether a reordered excerpt changes implied chronology.",
            "Human review of all five complete exports remains required before publication.",
        ],
    })
    write_research_assets()
    write_performance_template(os.path.join(REPORT_DIR, "performance_history.csv"))
    contact_sheet = create_contact_sheet([x["final_path"] for x in clip_results], os.path.join(REPORT_DIR, "preview_contact_sheet.jpg"))
    guide = ["# Publishing guide", "", f"Run ID: `{RUN_ID}`", "", "Review each MP4 and cover before upload.",
             "Use the platform-specific JSON file for the title/caption and hashtags.",
             "Verify the music and stock-footage license manifest before publishing or transferring the masters.",
             "Use the platform metadata asset_credits; free-tier Coverr API assets require a Coverr link and a creator-profile link when a creator is listed.",
             "Do not use sponsored partner items or watermarked premium previews unless separately licensed.",
             "Coverr assets and metadata must never be used for AI training, fine-tuning, model evaluation, or datasets.",
             "The source speaker's words are original excerpts; cover titles are editorial overlays.",
             "Do not describe the heuristic score as measured retention or guaranteed virality.", ""]
    with open(os.path.join(REPORT_DIR, "publishing_guide.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(guide))
    atomic_write_json(os.path.join(REPORT_DIR, "asset_license_manifest.json"), ASSET_MANIFEST)

    # All five local outputs are QA-passed before any final Drive folder is created.
    if DRIVE_AVAILABLE:
        source_folder = safe_filename(SOURCE.title, 64) + "_" + RUN_ID
        drive_folder = os.path.join(DRIVE_ROOT, source_folder)
        folders = {
            "videos": os.path.join(drive_folder, "FINAL_VIDEOS"),
            "covers": os.path.join(drive_folder, "COVERS"),
            "publishing": os.path.join(drive_folder, "PUBLISHING_PACK"),
            "research": os.path.join(drive_folder, "RESEARCH"),
            "reports": os.path.join(drive_folder, "REPORTS"),
        }
        for folder in folders.values(): os.makedirs(folder, exist_ok=True)
        delivered = []
        for item in clip_results:
            delivered_path = safe_copy_verified(item["final_path"], os.path.join(folders["videos"], os.path.basename(item["final_path"])))
            drive_info = ffprobe_json(delivered_path)
            drive_duration = float(drive_info["format"]["duration"])
            if abs(drive_duration-item["duration"]) > 0.15:
                raise RuntimeError("Drive copy duration verification failed")
            cover_src = item["cover_path"]
            safe_copy_verified(cover_src, os.path.join(folders["covers"], os.path.basename(cover_src)))
            delivered.append({"path": delivered_path, "duration": drive_duration, "sha256": sha256_file(delivered_path)})
        for name in ("youtube_metadata.json", "tiktok_metadata.json", "instagram_metadata.json", "publishing_guide.md"):
            safe_copy_verified(os.path.join(REPORT_DIR, name), os.path.join(folders["publishing"], name))
        pack_src = os.path.join(REPORT_DIR, "publishing_pack")
        if os.path.isdir(pack_src):
            for name in sorted(os.listdir(pack_src)):
                safe_copy_verified(os.path.join(pack_src, name), os.path.join(folders["publishing"], name))
        for name in ("editorial_playbook.md", "source_register.json"):
            safe_copy_verified(os.path.join(REPORT_DIR, name), os.path.join(folders["research"], name))
        for name in os.listdir(REPORT_DIR):
            src = os.path.join(REPORT_DIR, name)
            if os.path.isfile(src):
                safe_copy_verified(src, os.path.join(folders["reports"], name))
        atomic_write_json(os.path.join(drive_folder, "DELIVERY_VERIFICATION.json"), {
            "run_id": RUN_ID, "five_videos": len(delivered), "verified": len(delivered) == int(CFG["required_clip_count"]),
            "files": delivered, "contact_sheet": os.path.join(folders["reports"], os.path.basename(contact_sheet)),
        })
        print("\nDRIVE DELIVERY VERIFIED:", drive_folder)
        for item in delivered: print(" ", item["path"], f"({item['duration']:.2f}s)")
        return drive_folder
    else:
        print("\nAll five local outputs passed the delivery gate; Google Drive was not available in this runtime.")
        return RUN_DIR


write_research_assets()
log("delivery", "publishing, playbook, license, reporting, and integrity-gated delivery functions ready")

In [ ]:
# CELL 10b — PUBLISHING STUDIO: a platform-native posting pack for every clip (Gemini writes it, code validates it)
# Output per clip: YouTube Shorts, TikTok, Instagram Reels, Facebook Reels, X, Threads — each with title/caption, hashtags, tags,
# category, settings and PINNED COMMENTS, ready to copy and paste. If Gemini is unavailable a local template is used instead.

YT_CATEGORIES = ["Film & Animation", "Autos & Vehicles", "Music", "Pets & Animals", "Sports", "Travel & Events", "Gaming",
                 "People & Blogs", "Comedy", "Entertainment", "News & Politics", "Howto & Style", "Education",
                 "Science & Technology", "Nonprofits & Activism"]

# Limits checked 16 Nov 2026 against third-party platform guides (YouTube/Instagram/TikTok change these often; re-verify in each app).
PL = {"yt_title": 100, "yt_title_ideal": 70, "yt_title_front": 40, "yt_desc": 5000, "yt_desc_fold": 157, "yt_tags_total": 500, "yt_tag_each": 30,
      "yt_hashtags": (3, 5), "tt_caption": 2200, "tt_hashtags": (3, 5), "ig_caption": 2200, "ig_hashtags": (3, 5), "fb_desc": 2000,
      "fb_hashtags": (0, 3), "x_post": 270, "threads_post": 480, "comment": 480}

PACKAGING_PLAYBOOK = """PACKAGING RULES (apply all):
1. TRUTH FIRST. Every claim, number and name in titles/captions must come from the CLIP TRANSCRIPT or SOURCE INFO below. Never invent statistics, results, quotes or promises. Misleading metadata gets videos limited.
2. CURIOSITY WITH A REAL ANSWER. Open an information gap (a question, contrast, or surprising-but-true statement) that THIS clip actually closes. Specific beats vague: use the concrete noun/number/situation from the clip.
3. ONE IDEA. The title promises one outcome or insight a viewer could repeat in one sentence. Prefer plain, direct benefit or tension ("Stop X: do Y") over hype words. At most one emoji, usually none. No ALL CAPS shouting.
4. FRONT-LOAD. The first ~40 characters of a YouTube Shorts title and the first ~100 characters of every caption carry the main keyword/hook, because feeds truncate. YouTube titles <= 70 characters; never > 100.
5. SEARCH LANGUAGE. Use the exact words a viewer would type (topic, problem, speaker/book/concept names that appear in the source). Put them naturally in the first line of the description/caption.
6. HASHTAGS: few and relevant. YouTube 3-5 (the first three show above the title; if more than 15 YouTube ignores all). TikTok 3-5. Instagram at most 5. Facebook at most 3. Threads uses ONE topic tag instead. Mix 1-2 broad and 2-3 niche. No banned/irrelevant or spammy tags.
7. TAGS (YouTube tags field): 8-15 short tags, each <= 30 characters, total <= 500 characters including commas; include the core phrase, close variants, common misspellings of names, and the topic category. YouTube says tags matter little, so never let them replace a good title/description.
8. PINNED COMMENTS drive replies, which platforms read as engagement. Write a short comment (<= 300 characters) that does ONE of: asks a specific, easy-to-answer question tied to the clip's idea (not 'thoughts?'); asks the viewer to pick a side; or adds one useful extra line (a takeaway or action) and then asks. No begging for likes, no fake giveaways, no links to other platforms. Give TWO options per platform.
9. HOOK LINE for every caption: the first sentence stands alone and earns the tap on 'more'.
10. TONE: confident, warm, human. Match the clip's language. No claims of guaranteed results (money, health, success)."""

PACK_STR = {"type": "string"}
PACK_LIST = {"type": "array", "items": {"type": "string"}}
PACK_SCHEMA = {
    "type": "object",
    "properties": {
        "youtube": {"type": "object", "properties": {
            "title": PACK_STR, "title_alternatives": PACK_LIST, "description_hook": PACK_STR, "description_body": PACK_STR,
            "hashtags": PACK_LIST, "tags": PACK_LIST, "category": PACK_STR, "pinned_comments": PACK_LIST},
            "required": ["title", "description_hook", "description_body", "hashtags", "tags", "category", "pinned_comments"]},
        "tiktok": {"type": "object", "properties": {
            "caption": PACK_STR, "hashtags": PACK_LIST, "search_keywords": PACK_LIST, "on_screen_text": PACK_STR, "pinned_comments": PACK_LIST},
            "required": ["caption", "hashtags", "on_screen_text", "pinned_comments"]},
        "instagram": {"type": "object", "properties": {
            "caption": PACK_STR, "hashtags": PACK_LIST, "cover_text": PACK_STR, "alt_text": PACK_STR, "pinned_comments": PACK_LIST},
            "required": ["caption", "hashtags", "cover_text", "pinned_comments"]},
        "facebook": {"type": "object", "properties": {"description": PACK_STR, "hashtags": PACK_LIST, "pinned_comments": PACK_LIST},
                     "required": ["description", "hashtags", "pinned_comments"]},
        "x": {"type": "object", "properties": {"post": PACK_STR, "reply_comments": PACK_LIST}, "required": ["post", "reply_comments"]},
        "threads": {"type": "object", "properties": {"post": PACK_STR, "topic_tag": PACK_STR, "reply_comments": PACK_LIST},
                    "required": ["post", "topic_tag", "reply_comments"]},
    },
    "required": ["youtube", "tiktok", "instagram", "facebook", "x", "threads"],
}


def _clean_tag(t):
    t = re.sub(r"[^\w]", "", str(t).replace("#", ""), flags=re.UNICODE)
    return t


def _dedupe(seq):
    seen, out = set(), []
    for x in seq:
        k = x.lower()
        if x and k not in seen:
            seen.add(k)
            out.append(x)
    return out


def _clip_text(text, limit):
    """Trim at a word boundary without leaving a dangling fragment."""
    text = re.sub(r"[ \t]+", " ", str(text or "")).strip()
    if len(text) <= limit:
        return text
    cut = text[:limit].rsplit(" ", 1)[0].rstrip(" ,;:-—")
    return cut


def _tags_line(tags, total=PL["yt_tags_total"], each=PL["yt_tag_each"]):
    out, used = [], 0
    for t in _dedupe([re.sub(r"\s+", " ", str(x).replace("#", "")).strip() for x in tags]):
        if not t or len(t) > each:
            continue
        add = len(t) + (2 if out else 0)      # ", " separator is counted conservatively
        if used + add > total:
            break
        out.append(t)
        used += add
    return ", ".join(out)


def _hash_list(tags, lo, hi):
    tags = _dedupe([_clean_tag(x) for x in tags])
    return ["#" + t for t in tags[:hi] if t]


def _numbers_in(text):
    return set(re.findall(r"\d[\d,\.]*", text or ""))


def finalize_pack(raw, ctx):
    """Deterministic clean-up: clamp lengths, fix hashtags/tags, assemble the final fields. Never invents content."""
    yt, tt, ig, fb, xx, th = (raw.get(k, {}) for k in ("youtube", "tiktok", "instagram", "facebook", "x", "threads"))
    credit = ctx.get("credit_block", "")
    source_line = ctx.get("source_line", "")

    title = _clip_text(str(yt.get("title", "")).replace("\n", " "), PL["yt_title"]).rstrip(".")
    yt_hash = _hash_list(yt.get("hashtags", []), *PL["yt_hashtags"])
    hook = _clip_text(yt.get("description_hook", ""), PL["yt_desc_fold"])
    body = str(yt.get("description_body", "")).strip()
    tail = [x for x in ([source_line] if source_line else []) + (["Asset credits:\n" + credit] if credit else [])]
    hash_line = " ".join(yt_hash)
    fixed_len = len(hook) + sum(len(x) for x in tail) + len(hash_line) + 12
    body = _clip_text(body, max(0, PL["yt_desc"] - fixed_len))
    description = "\n\n".join(p for p in [hook, body] + tail + [hash_line] if p)
    category = yt.get("category") if yt.get("category") in YT_CATEGORIES else "Education"
    langname = ctx.get("language_name", "English")

    youtube = {
        "title": title, "title_alternatives": [_clip_text(t, PL["yt_title"]) for t in (yt.get("title_alternatives") or [])][:3],
        "description": description, "hashtags": yt_hash, "tags_line": _tags_line(yt.get("tags", [])),
        "category": category, "pinned_comments": [_clip_text(c, PL["comment"]) for c in (yt.get("pinned_comments") or [])][:2],
        "settings": ["Visibility: choose when you publish", "Audience: select 'No, it's not made for kids' (keeps comments and notifications on)",
                     "Altered content: No (speech and footage are original recordings; answer Yes only if you add realistic AI-generated visuals or voices)",
                     "Paid promotion: No (unless you are sponsored)", f"Language: {langname}", "Location: leave blank (natural global discovery)",
                     "Comments: On, then pin the comment below right after publishing",
                     "Shorts remixing: your choice (allowing it can widen reach)"],
    }
    tt_hash = _hash_list(tt.get("hashtags", []), *PL["tt_hashtags"])
    tt_cap = _clip_text(str(tt.get("caption", "")), PL["tt_caption"] - len(" ".join(tt_hash)) - 2)
    tiktok = {"caption": (tt_cap + "\n\n" + " ".join(tt_hash)).strip(), "hashtags": tt_hash,
              "search_keywords": _dedupe([str(k).strip() for k in (tt.get("search_keywords") or [])])[:8],
              "on_screen_text": _clip_text(tt.get("on_screen_text", ""), 60),
              "pinned_comments": [_clip_text(c, PL["comment"]) for c in (tt.get("pinned_comments") or [])][:2]}
    ig_hash = _hash_list(ig.get("hashtags", []), *PL["ig_hashtags"])
    ig_cap = _clip_text(str(ig.get("caption", "")), PL["ig_caption"] - len(" ".join(ig_hash)) - 2)
    instagram = {"caption": (ig_cap + "\n\n" + " ".join(ig_hash)).strip(), "hashtags": ig_hash,
                 "cover_text": _clip_text(ig.get("cover_text", ""), 40), "alt_text": _clip_text(ig.get("alt_text", ""), 400),
                 "pinned_comments": [_clip_text(c, PL["comment"]) for c in (ig.get("pinned_comments") or [])][:2]}
    fb_hash = _hash_list(fb.get("hashtags", []), *PL["fb_hashtags"])
    fb_desc = _clip_text(str(fb.get("description", "")), PL["fb_desc"] - len(" ".join(fb_hash)) - 2)
    facebook = {"description": (fb_desc + "\n\n" + " ".join(fb_hash)).strip(), "hashtags": fb_hash,
                "pinned_comments": [_clip_text(c, PL["comment"]) for c in (fb.get("pinned_comments") or [])][:2]}
    x_post = _clip_text(str(xx.get("post", "")), PL["x_post"])
    x_out = {"post": x_post, "reply_comments": [_clip_text(c, 270) for c in (xx.get("reply_comments") or [])][:2]}
    topic = _clean_tag(th.get("topic_tag", ""))
    th_post = _clip_text(str(th.get("post", "")), PL["threads_post"])
    threads = {"post": th_post, "topic_tag": topic, "reply_comments": [_clip_text(c, 400) for c in (th.get("reply_comments") or [])][:2]}
    return {"youtube": youtube, "tiktok": tiktok, "instagram": instagram, "facebook": facebook, "x": x_out, "threads": threads}


def validate_pack(raw, ctx):
    """Problems that clamping cannot fix honestly. Returned to Gemini once for repair."""
    errs = []
    yt = raw.get("youtube", {})
    title = str(yt.get("title", "")).strip()
    if len(title) < 12:
        errs.append("youtube.title is too short; make it a specific, front-loaded title of about 40-70 characters")
    if len(title) > PL["yt_title"]:
        errs.append(f"youtube.title is {len(title)} characters; it must be at most {PL['yt_title_ideal']} (hard limit {PL['yt_title']})")
    spoken = ctx.get("transcript", "") + " " + ctx.get("source_title", "")
    for n in _numbers_in(title) - _numbers_in(spoken):
        errs.append(f"youtube.title contains the number '{n}' that is not in the clip; remove or replace it (no invented numbers)")
    if len(re.findall(r"[A-Z]", title)) > 0.5 * max(1, len(re.findall(r"[A-Za-z]", title))) and len(title) > 12:
        errs.append("youtube.title is mostly capital letters; use normal title case")
    for plat in ("youtube", "tiktok", "instagram", "facebook"):
        pcs = [c for c in (raw.get(plat, {}).get("pinned_comments") or []) if str(c).strip()]
        if not pcs:
            errs.append(f"{plat}.pinned_comments is empty; write two options")
    if not [t for t in (raw.get("youtube", {}).get("tags") or []) if str(t).strip()]:
        errs.append("youtube.tags is empty")
    return errs


# ------------------------------------------------------------------ local (no-Gemini) fallback
_STOP = set("a an the and or but if then than that this those these of to in on at for from by with as is are was were be been being it its it's i you he she we they them his her our your my me do does did have has had not no yes so just very can could would should will what when where who why how about into over under again more most some any all each other such only own same too up down out off there here".split())


def _top_keywords(text, n=8):
    words = [w for w in re.findall(r"[A-Za-z][A-Za-z'-]{3,}", text.lower()) if w not in _STOP]
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    return [w for w, _ in sorted(freq.items(), key=lambda kv: (-kv[1], kv[0]))[:n]]


def local_pack(plan, idx, ctx):
    text = ctx["transcript"]
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]
    hook = (sents[0] if sents else plan.title).strip()
    title = _clip_text(plan.title if len(plan.title) >= 12 else hook, PL["yt_title_ideal"]).rstrip(".")
    kws = _top_keywords(text)
    topics = [t for t in plan.notes.get("topics", [])]
    base_tags = _dedupe(topics + kws + [t for t in (ctx.get("source_tags") or [])][:6])
    tagwords = [w.replace("-", " ") for w in base_tags]
    hash_pool = ["Shorts"] + [w.title().replace(" ", "") for w in topics] + [w.title() for w in kws[:3]]
    q = "Which part of this hit home for you, and what will you do with it this week?"
    q2 = "Do you agree with this? Tell me why in one sentence."
    body = _clip_text(" ".join(sents[1:3]) if len(sents) > 1 else text, 600)
    return {
        "youtube": {"title": title, "title_alternatives": [], "description_hook": _clip_text(hook, PL["yt_desc_fold"]), "description_body": body,
                    "hashtags": hash_pool[:4], "tags": tagwords[:14], "category": "Education", "pinned_comments": [q, q2]},
        "tiktok": {"caption": _clip_text(hook, 150) + " " + _clip_text(body, 220), "hashtags": hash_pool[:4], "search_keywords": kws[:5],
                   "on_screen_text": _clip_text(title, 60), "pinned_comments": [q, q2]},
        "instagram": {"caption": _clip_text(hook, 125) + "\n\n" + _clip_text(body, 400), "hashtags": hash_pool[:4], "cover_text": _clip_text(title, 40),
                      "alt_text": _clip_text("Speaker talking to camera: " + hook, 300), "pinned_comments": [q, q2]},
        "facebook": {"description": _clip_text(hook, 125) + "\n\n" + _clip_text(body, 400), "hashtags": hash_pool[:2], "pinned_comments": [q, q2]},
        "x": {"post": _clip_text(hook, 250), "reply_comments": [q2, q]},
        "threads": {"post": _clip_text(hook + " " + q2, 420), "topic_tag": (topics[0] if topics else (kws[0] if kws else "Mindset")).title().replace(" ", ""),
                    "reply_comments": [q2, q]},
    }


def _clip_context(plan, idx):
    transcript = " ".join(s.transcript for s in plan.segments).strip()
    meta = globals().get("SOURCE_META") or {}
    credits = asset_credit_records(idx) if "asset_credit_records" in globals() else []
    credit_block = format_asset_credits(credits) if credits and "format_asset_credits" in globals() else ""
    channel = meta.get("channel") or meta.get("uploader") or ""
    source_line = ""
    if CFG.get("credit_source_in_description", True) and SOURCE.title and not re.fullmatch(r"[A-Za-z0-9_-]{11}", SOURCE.title):
        source_line = f"Source: {SOURCE.title}" + (f" — {channel}" if channel else "")
    lang = getattr(TRANSCRIPT, "language", "en") or "en"
    names = {"en": "English", "es": "Spanish", "fr": "French", "pt": "Portuguese", "de": "German", "yo": "Yoruba", "ha": "Hausa", "ig": "Igbo"}
    return {"transcript": transcript, "source_title": SOURCE.title, "channel": channel, "source_tags": meta.get("tags") or [],
            "language_name": names.get(lang, lang), "credit_block": credit_block, "source_line": source_line,
            "duration": float(plan.duration), "narrative": plan.narrative_type, "topics": plan.notes.get("topics", [])}


def _pack_prompt(plan, idx, ctx):
    return (PACKAGING_PLAYBOOK + "\n\nTASK: Write the complete posting pack for ONE short vertical video clip, for YouTube Shorts, TikTok, Instagram Reels, "
            "Facebook Reels, X and Threads. Write in the language of the transcript. Return JSON only.\n"
            "FIELD NOTES: youtube.title_alternatives = 3 different angles for A/B testing. youtube.description_hook = the first 1-2 sentences "
            "(<=150 characters, keyword-rich, no hashtags). youtube.description_body = 2-4 short sentences adding context plus a soft call to action; "
            "no hashtags (they are added by code). hashtags = words WITHOUT the # symbol. youtube.category must be one of: "
            + ", ".join(YT_CATEGORIES) + ". tiktok.on_screen_text = a <=8 word text overlay idea. threads.topic_tag = one word/phrase, no #. "
            "x.post must be <= 270 characters and contain no link. pinned_comments: exactly two options each.\n\n"
            f"SOURCE INFO: title={ctx['source_title']!r}; channel={ctx['channel']!r}; the video's own tags={ctx['source_tags'][:12]}\n"
            f"CLIP: {ctx['duration']:.0f} seconds; story shape={ctx['narrative']}; topics={ctx['topics']}\n"
            f"CLIP TRANSCRIPT (the only source of facts):\n{ctx['transcript'][:6000]}")


def build_publishing_packs(plans):
    """Returns {'youtube': {'1': {...}}, 'tiktok': ..., ...} and writes copy-paste Markdown files into REPORT_DIR/publishing_pack."""
    out = {k: {} for k in ("youtube", "tiktok", "instagram", "facebook", "x", "threads")}
    pack_dir = os.path.join(REPORT_DIR, "publishing_pack")
    os.makedirs(pack_dir, exist_ok=True)
    sources = {}
    all_md = ["# PROJECT ZIROQ — Publishing packs", "", f"Source: {SOURCE.title}", ""]
    for idx, plan in enumerate(plans, 1):
        ctx = _clip_context(plan, idx)
        raw, source = None, "local template"
        if "gemini_structured" in globals() and globals().get("GEMINI") is not None and getattr(globals().get("GEMINI"), "enabled", False):
            res = gemini_structured(f"publishing_pack_{RUN_ID}_{idx}", _pack_prompt(plan, idx, ctx), PACK_SCHEMA, tier="reasoning",
                                    validator=lambda d, c=ctx: validate_pack(d, c), max_repairs=1, temperature=0.5)
            if res.get("ok"):
                raw, source = res["data"], "gemini"
            else:
                log("publishing", f"clip {idx}: Gemini pack unavailable ({res.get('error_kind')}); using the local template")
        if raw is None:
            raw = local_pack(plan, idx, ctx)
        final = finalize_pack(raw, ctx)
        # last safety net: even the Gemini pack must pass the hard limits after clamping
        assert len(final["youtube"]["title"]) <= PL["yt_title"] and len(final["x"]["post"]) <= 280
        for k in out:
            out[k][str(idx)] = dict(final[k], generated_by=source)
        sources[idx] = source
        md = render_pack_markdown(idx, plan, final, source, ctx)
        with open(os.path.join(pack_dir, f"clip_{idx:02d}_PUBLISHING_PACK.md"), "w", encoding="utf-8") as f:
            f.write(md)
        all_md.append(md)
        all_md.append("\n\n---\n")
    with open(os.path.join(pack_dir, "ALL_CLIPS_PUBLISHING_PACK.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(all_md))
    log("publishing", "posting packs written: " + ", ".join(f"clip {i}={s}" for i, s in sources.items()))
    return out


def _bul(items):
    return "\n".join(f"* {x}" for x in items)


def render_pack_markdown(idx, plan, f, source, ctx):
    yt, tt, ig, fb, xx, th = (f[k] for k in ("youtube", "tiktok", "instagram", "facebook", "x", "threads"))
    note = "" if source == "gemini" else "> Local template (Gemini was unavailable). Good enough to post after a quick read; re-run later for a sharper version.\n\n"
    L = [f"# CLIP {idx:02d} — {plan.title}", "", f"{note}Review everything once before posting. Facts in titles come only from the clip.", "",
         "## YouTube Shorts", "### Title", yt["title"], "",
         "### Alternative titles to A/B test", _bul(yt["title_alternatives"]) or "* (none)", "",
         "### Description", "(The first ~150 characters show before 'Show more'; the first three hashtags appear above the title.)", "", yt["description"], "",
         '### Tags (copy and paste this exact line into the Tags field under "Show more")', yt["tags_line"], "",
         "### Category", f"* {yt['category']}", "", "### Additional settings", _bul(yt["settings"]), "",
         "### Pinned comment (pick one, post it after publishing, then pin it)"]
    L += [f"{n}. {c}" for n, c in enumerate(yt["pinned_comments"], 1)]
    L += ["", "## TikTok", "### Caption", tt["caption"], "", "### On-screen text idea", tt["on_screen_text"] or "(none)", "",
          "### Search keywords to say or show in the first seconds", ", ".join(tt["search_keywords"]) or "(none)", "",
          "### Pinned comment (pick one)"] + [f"{n}. {c}" for n, c in enumerate(tt["pinned_comments"], 1)]
    L += ["", "## Instagram Reels", "### Caption (max 5 hashtags)", ig["caption"], "", "### Cover text", ig["cover_text"], "",
          "### Alt text", ig["alt_text"] or "(none)", "", "### First comment (pin it)"] + [f"{n}. {c}" for n, c in enumerate(ig["pinned_comments"], 1)]
    L += ["", "## Facebook Reels", "### Description", fb["description"], "", "### Pinned comment (pick one)"] + [f"{n}. {c}" for n, c in enumerate(fb["pinned_comments"], 1)]
    L += ["", "## X", "### Post", xx["post"], "", "### Reply to your own post (pick one)"] + [f"{n}. {c}" for n, c in enumerate(xx["reply_comments"], 1)]
    L += ["", "## Threads", "### Post", th["post"], "", f"### Topic tag (add one tag in the app): {th['topic_tag']}", "", "### Reply to your own post (pick one)"]
    L += [f"{n}. {c}" for n, c in enumerate(th["reply_comments"], 1)]
    return "\n".join(L)


print("Publishing Studio ready: per-clip packs for YouTube, TikTok, Instagram, Facebook, X and Threads (with pinned comments).")


In [ ]:
# CELL 11 — Run five independent productions, repair once if needed, then deliver as one verified package

def produce_five_videos():
    N = int(CFG["required_clip_count"])
    if len(PLANS) != N:
        raise RuntimeError(f"{N} qualified story plans are required before rendering (found {len(PLANS)})")
    results = []
    FINISH_LOG = finish_plans(PLANS, TRANSCRIPT, SOURCE)      # completeness gate + Gemini critic + filler/pause tightening
    for idx, plan in enumerate(PLANS, 1):
        log("production", f"START CLIP {idx}/{N} — {plan.title[:80]}")
        clip_dir = os.path.join(RENDER_DIR, f"clip_{idx:02d}")
        os.makedirs(clip_dir, exist_ok=True)
        final_name = f"{idx:02d}_{safe_filename(plan.title, 56)}.mp4"
        final_path = os.path.join(clip_dir, final_name)
        music_info = MUSIC_ASSETS[idx]
        broll = BROLL_ASSETS.get(idx)
        editorial_checks = run_editorial_qa(plan, SOURCE, TRANSCRIPT)
        if not all(c["passed"] for c in editorial_checks):
            raise RuntimeError(f"Editorial QA failed before render for clip {idx}: {editorial_checks}")

        render_info = None
        technical_checks = []
        crf = int(CFG["render_crf"])
        audio_ceiling = 0.80
        sfx_path = build_sfx_track(plan, os.path.join(clip_dir, "sfx.wav")) if CFG.get("enable_sfx", True) else None
        if sfx_path:
            print(f"  sound effects: {[(round(t, 1), k, who) for t, k, who in plan_sfx_cues(plan)]}")
        for flag in editorial_checks:
            if flag.get("severity") == "warning" and flag.get("flagged"):
                print(f"  REVIEW clip {idx}: {flag['detail']}")
        for attempt in range(CFG["qa_retry_count"] + 1):
            log("render", f"clip {idx}: render attempt {attempt+1}")
            work_dir = os.path.join(clip_dir, f"work_{attempt+1}")
            render_info = render_plan(plan, SOURCE, TRANSCRIPT.words, music_info["path"], final_path, work_dir, broll, crf=crf, audio_ceiling=audio_ceiling, sfx_path=sfx_path)
            print("  framing:", [f.get("mode") + (" (uncertain)" if f.get("uncertain") else "") for f in render_info["framing"]])
            technical_checks = run_technical_qa(render_info["temporary_path"], plan, render_info["caption_events"])
            passed = all(c["passed"] for c in technical_checks)
            for check in technical_checks:
                if CFG.get("verbose_qa") or not check["passed"]:
                    print(f"  {'PASS' if check['passed'] else 'FAIL'} {check['check']}: {check['detail']}")
            if passed and not CFG.get("verbose_qa"):
                print(f"  technical QA: all {len(technical_checks)} checks passed")
            if passed:
                os.replace(render_info["temporary_path"], final_path)
                break
            try: os.remove(render_info["temporary_path"])
            except OSError: pass
            if attempt >= CFG["qa_retry_count"]:
                raise RuntimeError(f"Clip {idx} failed technical QA after retries; package upload is blocked")
            # Retry with slightly more conservative compression quality, never with duration padding.
            crf = min(23, crf + 2)
            audio_ceiling = max(0.50, audio_ceiling - 0.10)   # a failed peak gate now gets a lower limiter ceiling on retry

        if not os.path.isfile(final_path):
            raise RuntimeError(f"Clip {idx} final file is missing")
        cover_path = os.path.join(clip_dir, f"{idx:02d}_COVER.jpg")
        make_cover(final_path, plan.title, cover_path)
        final_info = ffprobe_json(final_path)
        duration = float(final_info["format"]["duration"])
        if duration > 90.05:
            raise RuntimeError(f"Clip {idx} exceeds 90 seconds")
        item = {
            "index": idx, "title": plan.title, "narrative_type": plan.narrative_type,
            "segments": [asdict(s) for s in plan.segments], "duration": duration,
            "editorial_score": plan.notes.get("editorial_score"),
            "score_disclaimer": plan.notes.get("score_type"),
            "music": music_info["asset"], "broll": broll,
            "framing": render_info["framing"], "broll_window": render_info["broll_window"],
            "technical_checks": technical_checks, "editorial_checks": editorial_checks,
            "technical_pass": all(c["passed"] for c in technical_checks),
            "editorial_pass": all(c["passed"] for c in editorial_checks),
            "review_flags": [c["detail"] for c in editorial_checks if c.get("severity") == "warning" and c.get("flagged")],
            "finishing": {"log": plan.notes.get("finish_log", []), "tighten": plan.notes.get("tighten"), "critic": plan.notes.get("critic")},
            "final_path": final_path, "cover_path": cover_path,
            "sha256": sha256_file(final_path),
            "human_review_required": True,
        }
        results.append(item)
        log("production", f"CLIP {idx}/{N} PASS — {duration:.2f}s | {os.path.getsize(final_path)/1e6:.1f} MB")
        save_state("rendering", completed_clips=len(results), total_clips=N)

    if len(results) != N:
        raise RuntimeError(f"{N} completed outputs were not produced; no Drive upload will be attempted")
    if not all(x["technical_pass"] and x["editorial_pass"] for x in results):
        raise RuntimeError("One or more outputs failed the final QA gate")
    delivery_location = build_and_deliver_package(PLANS, results)
    save_state("delivered" if DRIVE_AVAILABLE else "local_outputs_verified", count=len(results), location=delivery_location)
    return results, delivery_location


CLIP_RESULTS, DELIVERY_LOCATION = produce_five_videos()
print("\nFINAL STATUS:", len(CLIP_RESULTS), "of", CFG["required_clip_count"], "video masters verified")
print("Output package:", DELIVERY_LOCATION)

In [ ]:
# CELL 12 — Run summary, review checklist, and a one-tap download when Google Drive is not connected
import zipfile
print("\nPROJECT ZIROQ RUN SUMMARY")
print("Source:", SOURCE.title)
print("Source duration:", f"{SOURCE.duration/60:.1f} min")
print("Transcript:", f"{len(TRANSCRIPT.words):,} words / {len(TRANSCRIPT.sentences):,} sentences")
print("Verified videos:", len(CLIP_RESULTS))
try:
    _yt = json.load(open(os.path.join(REPORT_DIR, "youtube_metadata.json"), encoding="utf-8"))
except Exception:
    _yt = {}
for item in CLIP_RESULTS:
    t = (_yt.get(str(item["index"])) or {}).get("title", item["title"])
    print(f"  {item['index']}. {item['duration']:.0f}s | {t}")
    for fl in item.get("review_flags", [])[:2]:
        print("       check:", fl[:110])
print("\nDelivery location:", DELIVERY_LOCATION)
print("Posting packs (titles, descriptions, tags, hashtags, PINNED COMMENTS for every platform):")
print("   PUBLISHING_PACK/ALL_CLIPS_PUBLISHING_PACK.md  and  clip_01_PUBLISHING_PACK.md, clip_02_... ")

if not DRIVE_AVAILABLE and CFG.get("download_zip_when_no_drive", True):
    zip_path = os.path.join(RUN_DIR, "ZIROQ_package.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
        for item in CLIP_RESULTS:
            z.write(item["final_path"], "FINAL_VIDEOS/" + os.path.basename(item["final_path"]))
            if item.get("cover_path") and os.path.isfile(item["cover_path"]):
                z.write(item["cover_path"], "COVERS/" + os.path.basename(item["cover_path"]))
        pack_dir = os.path.join(REPORT_DIR, "publishing_pack")
        for root_dir, _, names in os.walk(pack_dir):
            for n in names:
                z.write(os.path.join(root_dir, n), "PUBLISHING_PACK/" + n)
        for n in ("youtube_metadata.json", "tiktok_metadata.json", "instagram_metadata.json", "preview_contact_sheet.jpg", "asset_license_manifest.json"):
            p = os.path.join(REPORT_DIR, n)
            if os.path.isfile(p):
                z.write(p, "REPORTS/" + n)
    print(f"\nDrive is not connected, so everything is zipped here: {zip_path} ({os.path.getsize(zip_path)/1e6:.0f} MB)")
    try:
        from google.colab import files
        files.download(zip_path)
        print("Your browser should start downloading the zip now (check your Downloads).")
    except Exception as exc:
        print("Automatic download unavailable here; open the Colab Files panel (folder icon), find the zip above, and download it.", type(exc).__name__)

print("\nHuman review before publishing:")
print("  [ ] Watch every short from beginning to end")
print("  [ ] Verify reordered excerpts do not alter context or imply false chronology")
print("  [ ] Verify captions, names, numbers, and speaker meaning")
print("  [ ] Read the title and pinned comment in the posting pack; edit anything that feels off")
print("  [ ] Confirm source usage rights and any third-party asset license manifest")


In [ ]:
# CELL 13 (OPTIONAL, developers only) — Helper-level tests (no YouTube download, model download, or Colab GPU required)

def run_helper_self_tests():
    # Multi-interval source → output timeline mapping.
    plan = ClipPlan("test", "test", [
        Segment(10.0, 12.0, "hook", "alpha beta"),
        Segment(40.0, 43.0, "payoff", "gamma delta"),
    ], target_duration=90.0)
    words = [Word("alpha", 10.0, 10.4), Word("beta", 11.0, 11.4),
             Word("gamma", 40.0, 40.5), Word("delta", 42.0, 42.4)]
    mapped = timeline_words(plan, words)
    assert len(mapped) == 4
    assert abs(mapped[2]["start"] - 2.0) < 1e-6, mapped
    assert abs(mapped[-1]["end"] - 4.4) < 1e-6, mapped

    # Duration and source-boundary validation.
    assert abs(validate_segment_list(plan.segments, 100.0) - 5.0) < 1e-6
    try:
        validate_segment_list([Segment(98, 105, "bad", "bad")], 100.0)
        raise AssertionError("Invalid source segment was not rejected")
    except ValueError:
        pass
    try:
        ClipPlan("too long", "test", [Segment(0, 91, "x", "x")]).validate(120)
        raise AssertionError("Over-90-second plan was not rejected")
    except ValueError:
        pass

    # Audio graph has the ducking + speech-first final limiter path used by renderer.
    graph = ";".join(make_audio_mix_graph(2, 5.0, volume=0.18))
    assert "sidechaincompress" in graph and "amix" in graph and "alimiter" in graph
    assert "level=disabled" in graph and "alimiter=limit=0.800" in graph and "limit=0.500" in ";".join(make_audio_mix_graph(2, 5.0, ceiling=0.5))   # auto-level off: it used to push peaks back to full scale

    # Output-count gate must reject partial sets.
    try:
        enforce_five_output_count([1,2,3,4,5], [1,2])
        raise AssertionError("Partial delivery should have been blocked")
    except RuntimeError:
        pass
    assert enforce_five_output_count([1,2,3,4,5], [1,2,3,4,5]) is True

    # FFmpeg dynamic crop expression is syntactically composed and clamped.
    expression = ffmpeg_crop_x_expr([(0.0, 100.0), (1.0, 140.0), (2.0, 120.0)], 500)
    assert "if(lt(t" in expression and "min(500" in expression

    # Asset licence policy and path containment regression tests.
    assert _normalise_license_url("https://creativecommons.org/licenses/by/4.0") == "https://creativecommons.org/licenses/by/4.0/"
    assert "https://creativecommons.org/publicdomain/mark/1.0/" not in ARCHIVE_LICENSE_ALLOWLIST
    test_pool = os.path.join(CFG["work_dir"], "_asset_path_test")
    os.makedirs(test_pool, exist_ok=True)
    safe_test_file = os.path.join(test_pool, "safe.mp4")
    with open(safe_test_file, "wb") as f:
        f.write(b"test")
    assert _safe_pool_file(test_pool, "safe.mp4") == safe_test_file
    assert _safe_pool_file(test_pool, "../outside.mp4") is None
    assert _safe_pool_file(test_pool, os.path.abspath(safe_test_file)) is None
    os.remove(safe_test_file)
    os.rmdir(test_pool)

    print("PASS: timeline offset mapping")
    print("PASS: source interval / duration validation")
    print("PASS: speech-ducked audio graph construction")
    print("PASS: five-output delivery gate")
    print("PASS: dynamic crop expression construction")
    print("PASS: asset licence allowlist + safe pool paths")
    return True

def run_batch1_regression_tests():
    """Regression tests for the v4 batch-1 fixes (synthetic fixtures; no model/network access required)."""
    import tempfile, inspect, zipfile
    tmp = tempfile.mkdtemp(prefix="ziroq_batch1_")
    g = globals()
    try:
        # 1) classifier cue vocabulary matches lowercase tokens
        assert all(c == c.lower() for c in STORY_CUES), "STORY_CUES must be lowercase"

        # 2) story-plan checkpoint: round trip, and invalidation when the source changes
        saved = {k: g.get(k) for k in ("SOURCE", "TRANSCRIPT", "SIMILARITY_BACKEND", "SOURCE_DIR")}
        try:
            g["SOURCE"] = Source("tvid", "t", 300.0, "x.mp4")
            ws = [Word(f"w{i}", i * 0.5, i * 0.5 + 0.4) for i in range(40)]
            g["TRANSCRIPT"] = Transcript("en", ws, build_sentences(ws))
            g["SIMILARITY_BACKEND"], g["SOURCE_DIR"] = "tfidf_fallback", tmp
            plan = ClipPlan("Title", "story", [Segment(10.0, 20.0, "hook", "a b c"), Segment(100.0, 115.5, "payoff", "d e f")],
                            90.0, {"editorial_score": 55.5, "topics": ["money"]})
            save_plans_checkpoint([plan])
            loaded = load_plans_checkpoint()
            assert loaded and loaded[0].title == "Title" and abs(loaded[0].duration - 25.5) < 1e-9
            assert loaded[0].notes["editorial_score"] == 55.5
            g["SOURCE"] = Source("tvid", "t", 301.0, "x.mp4")
            assert load_plans_checkpoint() is None, "checkpoint must be rejected for a different source duration"
        finally:
            g.update(saved)

        # 3) editorial QA verifies against the transcript
        src = Source("v", "t", 600.0, "x.mp4")
        tw = [Word("Why", 10.0, 10.3), Word("does", 10.4, 10.7), Word("money", 10.8, 11.1), Word("matter?", 11.2, 11.6),
              Word("Because", 40.0, 40.4), Word("compounding", 40.5, 41.1), Word("works.", 41.2, 41.6)]
        tr = Transcript("en", tw, build_sentences(tw))
        def qa(segs, topics=("money",)):
            p = ClipPlan("t", "story", segs, 90.0, {"topics": list(topics), "editorial_score": 50})
            return {c["check"]: c for c in run_editorial_qa(p, src, tr)}
        good = qa([Segment(10.0, 11.6, "hook", "Why does money matter?"), Segment(40.0, 41.6, "payoff", "Because compounding works.")])
        assert good["source_quotation_integrity"]["passed"] and good["segments_on_sentence_boundaries"]["passed"]
        assert not good["context_review_flagged"]["flagged"]
        fake = qa([Segment(10.0, 11.6, "hook", "Completely different invented sentence here")])
        assert not fake["source_quotation_integrity"]["passed"], "invented text must fail quotation integrity"
        mid = qa([Segment(10.4, 11.6, "hook", "does money matter?")])
        assert not mid["segments_on_sentence_boundaries"]["passed"], "mid-sentence cut must fail"
        dangling = qa([Segment(40.0, 41.6, "hook", "Because compounding works.")])
        assert dangling["context_review_flagged"]["flagged"] and dangling["context_review_flagged"]["passed"]
        reorder = qa([Segment(40.0, 41.6, "hook", "Because compounding works."), Segment(10.0, 11.6, "support", "Why does money matter?")])
        assert "source order" in reorder["context_review_flagged"]["detail"]

        # 4) framing logic with a stub detector on a synthetic 16:9 video (a moving, then two static, bright squares)
        class BoxDetector:
            def detectMultiScale(self, gray, **kw):
                n, _, stats, _ = cv2.connectedComponentsWithStats((gray > 200).astype(np.uint8))
                return np.array([stats[i][:4] for i in range(1, n) if stats[i][4] > 1000], dtype=np.int32).reshape(-1, 4)
        class NoFaces:
            def detectMultiScale(self, gray, **kw):
                return np.empty((0, 4), dtype=np.int32)
        W, H, FPS = 1280, 720, 10
        def make_video(path, painter):
            vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
            for n in range(60):
                frame = np.zeros((H, W, 3), np.uint8); painter(frame, n / FPS); vw.write(frame)
            vw.release()
        def moving(frame, t):
            x = 100 + int(t * 150); frame[300:420, x:x + 120] = 255
        def two_static(frame, t):
            frame[300:420, 100:220] = 255; frame[300:420, 1000:1120] = 255
        v1, v2 = os.path.join(tmp, "moving.mp4"), os.path.join(tmp, "two.mp4")
        make_video(v1, moving); make_video(v2, two_static)
        seg = Segment(0.0, 5.0, "hook", "x")
        r = detect_segment_framing(Source("a", "a", 6.0, v1, width=W, height=H), seg, detector=BoxDetector())
        assert r["mode"] == "face_track" and r["keyframes"][-1][1] > r["keyframes"][0][1] + 200, r
        r = detect_segment_framing(Source("b", "b", 6.0, v2, width=W, height=H), seg, detector=BoxDetector())
        assert r["mode"] == "fit_blur" and r.get("multi_person_samples", 0) >= 2, r
        r = detect_segment_framing(Source("a", "a", 6.0, v1, width=W, height=H), seg, detector=NoFaces())
        assert r["mode"] == "fit_blur" and r.get("uncertain"), r
        old_mode = CFG.get("framing_uncertain_mode")
        try:
            CFG["framing_uncertain_mode"] = "center_crop"
            r = detect_segment_framing(Source("a", "a", 6.0, v1, width=W, height=H), seg, detector=NoFaces())
            assert r["mode"] == "center_crop", r
        finally:
            CFG["framing_uncertain_mode"] = old_mode
        r = detect_segment_framing(Source("p", "p", 6.0, v1, width=720, height=1280), seg, detector=BoxDetector())
        assert r["mode"] == "fit_blur" and "not wider" in r["reason"], r

        # 5) cascade recovery helpers (wheel extraction without installing)
        wheel = os.path.join(tmp, "fake.whl")
        with zipfile.ZipFile(wheel, "w") as zf:
            zf.writestr("cv2/data/" + FACE_CASCADE_NAME, "x" * 2000)
            zf.writestr("cv2/data/tiny.xml", "x")
        out = extract_member_from_wheel(wheel, "/" + FACE_CASCADE_NAME, os.path.join(tmp, "m", FACE_CASCADE_NAME))
        assert out and os.path.getsize(out) == 2000
        assert extract_member_from_wheel(wheel, "/tiny.xml", os.path.join(tmp, "m", "tiny.xml")) is None
        assert extract_member_from_wheel(wheel, "/missing.xml", os.path.join(tmp, "m", "missing.xml")) is None

        # 6) detector resolution order and the loud no-detector outcome (mocked)
        saved_fns = {k: g[k] for k in ("_load_cascade", "recover_cascade_from_opencv4_wheel", "get_face_detector")}
        try:
            g["_load_cascade"] = lambda p: None
            g["recover_cascade_from_opencv4_wheel"] = lambda d: None
            st = saved_fns["get_face_detector"](force_reload=True)
            assert st["detector"] is None and st["kind"] is None and st["reason"]
            g["get_face_detector"] = lambda force_reload=False: {"detector": None}
            r = detect_segment_framing(Source("a", "a", 6.0, v1, width=W, height=H), seg)
            assert r["mode"] == "fit_blur" and r.get("uncertain"), "no detector must keep the full frame, not centre-crop"
            sentinel = object()
            g["get_face_detector"] = saved_fns["get_face_detector"]
            g["_load_cascade"] = lambda p: sentinel
            st = saved_fns["get_face_detector"](force_reload=True)
            assert st["kind"] == "haar_bundled" and st["detector"] is sentinel
        finally:
            g.update(saved_fns)
            _FACE_DETECTOR_STATE.update({"resolved": False, "kind": None, "detector": None, "reason": ""})

        _keep_seg = seg
        S = Source("v", "t", 200.0, "x", width=1920, height=1080)
        seg = Segment(8.0, 20.0, "p", "t")
        h = parse_framing_hint({"layout": "single_speaker", "speaker_x": [{"time": "00:10", "x": 0.3}, {"time": "00:14", "x": 0.35}]})
        f = framing_from_hint(S, seg, h)
        assert f["mode"] == "face_track" and len(f["keyframes"]) == 2 and all(0 <= k[1] <= 1920 - 608 for k in f["keyframes"])
        assert framing_from_hint(S, seg, {"layout": "slides_or_screen", "points": []})["mode"] == "fit_blur"
        assert framing_from_hint(S, seg, {"layout": "single_speaker", "points": [(10.0, 0.1), (12.0, 0.9)]}) is None   # implausible jump
        assert framing_from_hint(S, seg, {"layout": "single_speaker", "points": [(10.0, 0.5)]}) is None                 # too few samples
        assert framing_from_hint(S, seg, None) is None
        seg = _keep_seg
        # 6b) Gemini framing hint: used only when local detection is uncertain; slides/multi-speaker layouts force fit_blur
        class _Seg: source_start, source_end, duration = 0.0, 3.0, 3.0
        _src = Source("h", "h", 6.0, v1, width=W, height=H)
        _hint = {"layout": "single_speaker", "points": [(0.5, 0.4), (2.5, 0.45)]}
        _r = detect_segment_framing(_src, seg, detector=NoFaces(), hint=_hint)
        assert _r["mode"] == "face_track" and _r.get("decided_by") == "gemini", _r
        _r = detect_segment_framing(_src, seg, detector=BoxDetector(), hint={"layout": "slides_or_screen", "points": []})
        assert _r["mode"] == "fit_blur" and _r.get("decided_by") == "gemini"
        _r = detect_segment_framing(_src, seg, detector=NoFaces(), hint=None)
        assert _r["mode"] == "fit_blur" and _r.get("uncertain")

        # 7) retry no longer needs to mutate CFG
        assert "crf" in inspect.signature(render_plan).parameters
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    print("PASS: batch-1 regression tests (classifier cues, plan checkpoint, editorial QA, framing, cascade recovery, retry crf)")
    return True


def run_batch3_music_tests():
    g = globals()
    plan = ClipPlan("t", "x", [Segment(0, 30, "p", "believe in the future")],
                    notes={"music_brief": {"mood": "playful", "energy": "high", "search_terms": "Bright Ukulele!"}, "broll_query": "city skyline night"})
    assert plan_mood(plan) == "playful"                                   # Gemini brief wins over keyword guess
    plan.notes["music_brief"]["mood"] = "bogus"
    assert plan_mood(plan) == "uplifting"                                  # invalid brief falls back to the local classifier
    assert broll_query_for_plan(plan) == "city skyline night"
    plan.notes["broll_query"] = "x"
    assert broll_query_for_plan(plan) != "x"                               # too-short brief -> local query
    seen = {}
    import tempfile
    real_get, real_cfg_key, real_cache = g["requests"].get, CFG.get("coverr_api_key"), g["ASSET_CACHE_DIR"]
    g["ASSET_CACHE_DIR"] = tempfile.mkdtemp()
    class R:
        def raise_for_status(self): pass
        def json(self): return {"hits": []}
    def fake_get(url, params=None, **kw):
        seen["query"] = (params or {}).get("query"); return R()
    try:
        g["requests"].get = fake_get
        CFG["coverr_api_key"] = "test"
        plan.notes["music_brief"] = {"mood": "playful", "search_terms": "Bright Ukulele!"}
        g["MUSIC_QUERY_OVERRIDE"] = "bright ukulele"
        g["_download_coverr_music"]("playful", 0)
        assert seen["query"] == "bright ukulele", seen
        g["MUSIC_QUERY_OVERRIDE"] = ""
    finally:
        g["requests"].get = real_get; CFG["coverr_api_key"] = real_cfg_key; g["MUSIC_QUERY_OVERRIDE"] = ""; g["ASSET_CACHE_DIR"] = real_cache
    print("PASS: Gemini music/B-roll brief reaches the provider search; invalid briefs fall back locally")
    return True


def run_batch5_sfx_tests():
    import tempfile
    seg = Segment(100.0, 160.0, "p", "t")
    plan = ClipPlan("t", "x", [seg], notes={"sfx_cues": [(110.0, "impact"), (111.0, "pop"), (150.0, "whoosh"), (99.0, "pop"), (159.9, "pop"), (120.0, "bogus")]})
    cues = plan_sfx_cues(plan)
    assert [(round(t), k) for t, k, _ in cues] == [(10, "impact"), (50, "whoosh")], cues      # too-close, outside and invalid cues are dropped
    two = ClipPlan("t", "x", [Segment(10, 40, "p", "t"), Segment(200, 230, "p", "t")])
    c2 = plan_sfx_cues(two)
    assert len(c2) == 1 and c2[0][2] == "local_rule" and abs(c2[0][0] - 29.85) < 0.01          # rule-based whoosh at the join
    assert plan_sfx_cues(ClipPlan("t", "x", [seg])) == []                                       # single interval, no cues -> nothing
    tmp = tempfile.mkdtemp()
    path = build_sfx_track(plan, os.path.join(tmp, "sfx.wav"))
    with wave.open(path) as w:
        assert w.getframerate() == 48000 and w.getnchannels() == 2 and abs(w.getnframes() / 48000 - 61.0) < 0.05
        data = np.frombuffer(w.readframes(w.getnframes()), dtype="<i2")
    assert 0.05 < np.abs(data).max() / 32767 <= 0.81                                            # audible but bounded
    for k in SFX_KINDS:
        assert np.isfinite(synth_sfx(k)).all() and abs(synth_sfx(k)).max() <= 0.61
    assert build_sfx_track(ClipPlan("t", "x", [seg]), os.path.join(tmp, "none.wav")) is None
    g = ";".join(make_audio_mix_graph(2, 5.0, sfx_input_index=3))
    assert "[3:a]" in g and "amix=inputs=3" in g and "[speech][ducked][sfx]" in g
    assert "amix=inputs=2" in ";".join(make_audio_mix_graph(2, 5.0))
    # real FFmpeg run of the 3-input graph: result decodes, and peak stays under the ceiling
    ff = os.path.join(tmp, "m.wav")
    run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "anoisesrc=d=20:c=pink:a=0.4", "-f", "lavfi", "-i", "sine=f=220:d=20", "-i", path,
         "-filter_complex", ";".join(["[0:a]aformat=channel_layouts=stereo,asplit=0[x]"][:0] + ["[0:a]aformat=channel_layouts=stereo[abase]"] + make_audio_mix_graph(1, 20.0, sfx_input_index=2)),
         "-map", "[aout]", "-t", "20", ff], quiet=True)
    audio = load_pcm16_wav(ff)[0] if False else None
    assert os.path.getsize(ff) > 1000
    print("PASS: SFX cue validation, procedural synthesis, 3-input mix graph (decoded by FFmpeg)")
    return True


def run_batch6_tests():
    import types
    S = lambda t, a, b: Sentence(t, a, b)
    sents = [S("We started with nothing.", 0, 4), S("Then the first sale came in", 4.5, 9),            # no terminal punctuation, ends on a plain word
             S("and everything changed.", 9.4, 12), S("It took years.", 12.5, 15), S("But that is the lesson.", 15.3, 18.5),
             S("And so we kept going.", 18.9, 22)]
    tr = Transcript("en", [], sents)
    SRC = Source("v", "t", 100.0, "x")
    assert sentence_complete("It ended.") and not sentence_complete("and then the") and not sentence_complete("we went to") and not sentence_complete("I think so")
    # end extension: plan ends mid-thought ("...came in" is complete-looking, so use a clearly unfinished one)
    sents2 = [S("Money works like this.", 0, 4), S("You save, then you", 4.4, 8), S("invest the difference.", 8.3, 12), S("That is it.", 12.5, 15)]
    tr2 = Transcript("en", [], sents2)
    p = ClipPlan("t", "x", [Segment(0.0, 8.1, "p", "Money works like this. You save, then you")], notes={})
    log_ = complete_plan_boundaries(p, tr2, SRC)
    assert log_ and abs(p.segments[0].source_end - 12.1) < 0.01 and p.segments[0].transcript.endswith("invest the difference."), (log_, p.segments[0])
    # cap respected: cannot extend beyond 90 s
    p = ClipPlan("t", "x", [Segment(0.0, 8.1, "p", "x")], target_duration=8.2, notes={})
    p.target_duration = 90.0
    CFG_max = CFG["max_clip_duration"]; CFG["max_clip_duration"] = 8.5
    try:
        complete_plan_boundaries(p, tr2, SRC); assert p.segments[0].source_end <= 8.51
    finally:
        CFG["max_clip_duration"] = CFG_max
    # dangling connector start trimmed when enough time remains
    long_s = [S("And so we kept going.", 0, 4)] + [S(f"Sentence {i} stays here.", 4 + i * 4, 7.5 + i * 4) for i in range(8)]
    trl = Transcript("en", [], long_s)
    p = ClipPlan("t", "x", [Segment(0.0, 36.0, "p", "x")], notes={})
    complete_plan_boundaries(p, trl, SRC); assert _first_token(p.segments[0].transcript) == "sentence" and p.segments[0].source_start > 3.9

    # tighten: filler + long pause removed, cuts in silence only, QA boundary logic accepts it
    W = lambda w, a, b: Word(w, a, b, 0.95)
    words = [W("Money", 0.5, 0.9), W("works", 1.0, 1.4), W("um", 1.6, 1.9), W("like", 2.0, 2.3), W("this", 2.4, 2.8),   # filler between 1.4 and 2.0
             W("Save", 4.2, 4.6), W("early", 4.7, 5.1), W("and", 5.2, 5.5), W("invest", 5.6, 6.1), W("today.", 6.2, 6.7),      # 1.4 s pause before "Save"
             W("That", 6.8, 7.1), W("is", 7.2, 7.4), W("it", 7.5, 7.8), W("really.", 7.9, 8.4)]
    sn = [S("Money works um like this.", 0.45, 2.85), S("Save early and invest today.", 4.15, 6.75), S("That is it really.", 6.8, 8.45)]
    tr3 = Transcript("en", words, sn)
    SR = Source("v", "t", 30.0, "x", width=1920, height=1080)
    plan = ClipPlan("t", "x", [Segment(0.4, 8.55, "p", "Money works um like this. Save early and invest today. That is it really.")], notes={"editorial_score": 50})
    before = plan.duration
    CFG["tighten_max_fraction"] = 0.30        # this toy clip is deliberately full of dead air
    try:
        st = tighten_plan(plan, tr3, SR)
    finally:
        CFG["tighten_max_fraction"] = 0.20
    assert st["fillers"] >= 1 and st["pauses"] >= 1 and len(plan.segments) >= 3 and plan.duration < before - 1.0, (st, plan.segments)
    assert plan.segments[0].role != "tight_zoom" and plan.segments[1].role == "tight_zoom" and plan.segments[2].role == "tight_cont"
    for a, b in zip(plan.segments, plan.segments[1:]):
        assert b.source_start > a.source_end                                                    # cuts only remove time, never repeat it
        assert not any(w.start < a.source_end + 0.001 and w.end > a.source_end + 0.001 or (w.start < b.source_start and w.end > b.source_start + 0.001) for w in words), "cut inside a word"
    assert all(s.duration >= 0.8 for s in plan.segments)
    mapped = [m["word"] for m in timeline_words(plan, words)]
    assert "um" not in mapped and "Money" in mapped and "today." in mapped                       # filler gone from captions, real words kept
    qa = run_editorial_qa(plan, SR, tr3)
    assert [c for c in qa if c["check"] == "segments_on_sentence_boundaries"][0]["passed"], qa
    assert [c for c in qa if c["check"] == "source_quotation_integrity"][0]["passed"], qa
    # budget guard: a clip that is mostly silence is not shredded
    sparse = ClipPlan("t", "x", [Segment(0.0, 10.0, "p", "x")], notes={})
    tr4 = Transcript("en", [W("a", 0.0, 0.3), W("b", 4.0, 4.3), W("c", 9.6, 9.9)], [S("a b c.", 0, 10)])
    t4 = tighten_plan(sparse, tr4, SR); assert t4["removed_sec"] <= 0.2 * 10.0 + 0.01
    CFG["enable_tighten"] = False
    try:
        off = ClipPlan("t", "x", [Segment(0.4, 8.55, "p", "x")], notes={}); assert tighten_plan(off, tr3, SR)["pieces_added"] == 0
    finally:
        CFG["enable_tighten"] = True

    # critic: executed only as bounded sentence fixes; bad output changes nothing
    global GEMINI
    saved = GEMINI
    class CriticPool:
        enabled = True; disabled_reason = ""
        def __init__(self, v): self.v = v
        def generate(self, models, inp, rf, system, cfg, task):
            return GeminiResult(json.dumps({"verdicts": self.v}), models[0], "P", "k", 1)
    import tempfile
    global GEMINI_CACHE_DIR
    saved_cache = GEMINI_CACHE_DIR; GEMINI_CACHE_DIR = tempfile.mkdtemp()
    try:
        tr5 = Transcript("en", [], [S("Intro line here.", 0, 4), S("The key point is this.", 4.4, 14), S("Do it daily.", 14.4, 24), S("That compounds.", 24.4, 30)])
        p1 = ClipPlan("t", "x", [Segment(0.0, 14.1, "p", "Intro line here. The key point is this.")], notes={})
        GEMINI = CriticPool([{"index": 0, "complete_thought": False, "needs_prior_context": True, "ends_on_payoff": False, "issue": "cut before the answer",
                              "extend_end_sentences": 2, "trim_start_sentences": 1}])
        v = gemini_critic([p1], tr5); assert 0 in v
        apply_critic(p1, v[0], tr5, SRC)
        assert abs(p1.segments[0].source_end - 30.1) < 0.01 and p1.segments[0].source_start > 3.9
        p2 = ClipPlan("t", "x", [Segment(0.0, 14.1, "p", "x")], notes={})
        apply_critic(p2, {"complete_thought": True, "needs_prior_context": False, "ends_on_payoff": True, "extend_end_sentences": 2, "trim_start_sentences": 2}, tr5, SRC)
        assert p2.segments[0].source_end == 14.1 and p2.segments[0].source_start == 0.0           # advice ignored when the verdict says it is fine
        GEMINI = CriticPool([{"index": 7, "complete_thought": True, "needs_prior_context": False, "ends_on_payoff": True, "extend_end_sentences": 0, "trim_start_sentences": 0}])
        assert gemini_critic([ClipPlan("t2", "x", [Segment(0.0, 5.0, "p", "z")], notes={})], tr5) == {}   # invalid index -> fallback, no changes
        GEMINI = types.SimpleNamespace(enabled=False, disabled_reason="off")
        assert gemini_critic([p1], tr5) == {}
        # whole pipeline entry point, idempotent
        GEMINI = types.SimpleNamespace(enabled=False, disabled_reason="off")
        pp = ClipPlan("t", "x", [Segment(0.4, 8.55, "p", "x")], notes={"editorial_score": 50})
        finish_plans([pp], tr3, SR); n1 = len(pp.segments); finish_plans([pp], tr3, SR)
        assert pp.notes["finished"] and len(pp.segments) == n1
    finally:
        GEMINI, GEMINI_CACHE_DIR = saved, saved_cache
    # split-screen geometry + chain
    h = parse_framing_hint({"layout": "multi_speaker", "speakers_x": [0.7, 0.25], "speaker_x": []})
    assert h["speakers_x"] == [0.25, 0.7]
    f = framing_from_hint(SR, Segment(0, 5, "p", "t"), h)
    assert f["mode"] == "split_two" and f["crop_w"] == 1214 and 0 <= f["x_left"] < f["x_right"] <= 1920 - 1214, f
    assert framing_from_hint(SR, Segment(0, 5, "p", "t"), {"layout": "multi_speaker", "speakers_x": [0.4, 0.5]})["mode"] == "fit_blur"     # too close together
    assert framing_from_hint(Source("v", "t", 9.0, "x", width=1000, height=1080), Segment(0, 5, "p", "t"), h)["mode"] == "fit_blur"        # source too narrow
    assert parse_framing_hint({"layout": "multi_speaker", "speakers_x": [0.2]})["speakers_x"] == []
    # audio meter regression: identical-channel audio with a true 0.75 peak must read ~0.75, not 1.06 (+3 dB mono-downmix artefact)
    tmpd = tempfile.mkdtemp(); mp = os.path.join(tmpd, "t.m4a")
    run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "aevalsrc=0.75*sin(2*PI*440*t)|0.75*sin(2*PI*440*t):d=2:s=48000", "-c:a", "aac", "-b:a", "192k", mp], quiet=True)
    pk = audio_metrics(mp)["peak"]
    assert 0.70 <= pk <= 0.82, pk
    print("PASS: completeness gate, Gemini critic (bounded), filler/pause tightening, split-screen geometry, audio meter")
    return True


def run_batch7_tests():
    """Publishing Studio: limits, pinned comments, local fallback, Gemini output clamping, files written."""
    import types, tempfile
    g = globals()
    saved = {k: g.get(k) for k in ("SOURCE", "TRANSCRIPT", "REPORT_DIR", "RUN_ID", "GEMINI", "gemini_structured", "SOURCE_META")}
    try:
        g["SOURCE"] = types.SimpleNamespace(title="Jim Rohn - Take Responsibility", duration=900.0)
        g["TRANSCRIPT"] = types.SimpleNamespace(language="en")
        g["SOURCE_META"] = {"channel": "Rohn Channel", "tags": ["jim rohn", "personal development"]}
        g["REPORT_DIR"] = tempfile.mkdtemp(); g["RUN_ID"] = "T7"
        seg = lambda t: types.SimpleNamespace(transcript=t)
        plan = types.SimpleNamespace(
            title="Stop blaming others for your failures", duration=62.0, narrative_type="problem-solution", notes={"topics": ["self-improvement", "leadership"]},
            segments=[seg("Failure is an event, not a person. You have to take responsibility for your results."),
                      seg("Nobody else is going to change your life, so stop making excuses and start acting today.")])
        ctx = _clip_context(plan, 1)
        # 1. local fallback honours every hard limit and has pinned comments
        f = finalize_pack(local_pack(plan, 1, ctx), ctx)
        assert 12 <= len(f["youtube"]["title"]) <= PL["yt_title"] and len(f["youtube"]["tags_line"]) <= PL["yt_tags_total"]
        assert 1 <= len(f["youtube"]["hashtags"]) <= 5 and len(f["instagram"]["hashtags"]) <= 5 and len(f["x"]["post"]) <= 280
        for plat in ("youtube", "tiktok", "instagram", "facebook"):
            assert len(f[plat]["pinned_comments"]) == 2 and all(f[plat]["pinned_comments"]), plat
        assert f["threads"]["topic_tag"] and "Source: Jim Rohn" in f["youtube"]["description"] and f["youtube"]["category"] == "Education"
        # 2. validator rejects invented numbers and missing pinned comments
        bad = {"youtube": {"title": "How 7 habits made me 10x richer in 30 days", "pinned_comments": [], "tags": ["a"]},
               "tiktok": {"pinned_comments": ["q"]}, "instagram": {"pinned_comments": ["q"]}, "facebook": {"pinned_comments": ["q"]}}
        errs = validate_pack(bad, ctx)
        assert any("not in the clip" in e for e in errs) and any("youtube.pinned_comments" in e for e in errs), errs
        # 3. Gemini output that is too long/spammy is clamped, files are written
        long_raw = {"youtube": {"title": "Stop Blaming Others " * 8, "title_alternatives": ["a b c d e f g h i j k l", "x" * 150], "description_hook": "h" * 400,
                                "description_body": "b " * 3000, "hashtags": [f"tag{i}" for i in range(14)], "tags": [f"long tag number {i}" for i in range(80)] + ["z" * 45],
                                "category": "Made Up Category", "pinned_comments": ["What will you own this week?", "Agree or disagree?"]},
                    "tiktok": {"caption": "c " * 2000, "hashtags": [f"t{i}" for i in range(12)], "search_keywords": ["a"] * 20, "on_screen_text": "o" * 200, "pinned_comments": ["q1", "q2"]},
                    "instagram": {"caption": "i " * 2000, "hashtags": [f"#i{i}" for i in range(30)], "cover_text": "c" * 90, "alt_text": "a" * 900, "pinned_comments": ["q1", "q2"]},
                    "facebook": {"description": "f " * 2000, "hashtags": [f"f{i}" for i in range(9)], "pinned_comments": ["q1", "q2"]},
                    "x": {"post": "x " * 300, "reply_comments": ["r1", "r2"]},
                    "threads": {"post": "t " * 400, "topic_tag": "#Mind set", "reply_comments": ["r1", "r2"]}}
        fl = finalize_pack(long_raw, ctx)
        assert len(fl["youtube"]["title"]) <= 100 and len(fl["youtube"]["hashtags"]) == 5 and len(fl["youtube"]["tags_line"]) <= 500
        assert all(len(t.strip()) <= 30 for t in fl["youtube"]["tags_line"].split(",")) and len(fl["youtube"]["description"]) <= 5000
        assert fl["youtube"]["category"] == "Education" and len(fl["tiktok"]["caption"]) <= 2200 and len(fl["instagram"]["caption"]) <= 2200
        assert len(fl["instagram"]["hashtags"]) == 5 and len(fl["facebook"]["hashtags"]) == 3 and len(fl["x"]["post"]) <= 270
        assert len(fl["threads"]["post"]) <= 480 and fl["threads"]["topic_tag"] == "Mindset" and len(fl["instagram"]["cover_text"]) <= 40
        g["GEMINI"] = types.SimpleNamespace(enabled=True)
        g["gemini_structured"] = lambda *a, **k: {"ok": True, "data": long_raw, "source": "gemini"}
        packs = build_publishing_packs([plan])
        md = open(os.path.join(REPORT_DIR, "publishing_pack", "clip_01_PUBLISHING_PACK.md"), encoding="utf-8").read()
        assert packs["youtube"]["1"]["generated_by"] == "gemini" and "Pinned comment" in md and "copy and paste this exact line" in md
        assert os.path.isfile(os.path.join(REPORT_DIR, "publishing_pack", "ALL_CLIPS_PUBLISHING_PACK.md"))
        # 4. Gemini unavailable -> local template still produces a complete, valid pack
        g["gemini_structured"] = lambda *a, **k: {"ok": False, "fallback_required": True, "error_kind": "unavailable"}
        packs2 = build_publishing_packs([plan])
        assert packs2["youtube"]["1"]["generated_by"] == "local template" and packs2["tiktok"]["1"]["pinned_comments"]
        print("PASS: publishing studio (limits, pinned comments, grounding checks, local fallback, files)")
        return True
    finally:
        for k, v in saved.items():
            if v is None:
                g.pop(k, None)
            else:
                g[k] = v



if CFG.get("run_self_tests"):
    HELPER_TESTS_PASSED = run_helper_self_tests()
    BATCH7_TESTS_PASSED = run_batch7_tests()
    BATCH6_TESTS_PASSED = run_batch6_tests()
    BATCH5_SFX_TESTS_PASSED = run_batch5_sfx_tests()
    BATCH3_MUSIC_TESTS_PASSED = run_batch3_music_tests()
    BATCH1_TESTS_PASSED = run_batch1_regression_tests()
else:
    print("Developer self-tests are OFF (normal). You do not need this cell; skip it.")
